# 03 · Transfer Learning and Fine-Tuning of Persian Transformers

> **Project 3 · Intelligent Analysis of Digikala Reviews**
> Phase goal: fine-tune pretrained encoders on `body` only, to predict **recommendation status** and **rating band** with one shared model.

---

## 🎯 Objective

Fine-tune a pretrained Transformer with **two classification heads** on top of one **shared encoder** (multi-task learning):

| Head | Task | Classes |
| --- | --- | --- |
| `rec` | Recommendation status | `recommended`, `not_recommended`, `no_idea` |
| `band` | Ordered rating band | `low`, `medium`, `high` |

The total loss is the sum of the two head losses. Rows with a missing target are masked out of that head's loss only.

---

## 🧭 Phase Overview

1. **Fine-tune TookaBERT** in two variants:
   - **B**: Step-1 corrected labels, no sample weights.
   - **C**: Step-1 corrected labels plus Step-2 sample weights.
2. **Choose B or C** with a rule fixed before seeing results.
3. **Fine-tune two more encoders** with the winning variant.
4. **Select the primary model** for the semi-supervised phase.
5. **Compare** against the dummy and TF-IDF baselines under the same split and protocol.

---

## 🤖 Encoders

| Model | Tokenizer | `max_length` |
| --- | --- | --- |
| `PartAI/TookaBERT-Base` | BPE (Persian) | 128 |
| `HooshvareLab/roberta-fa-zwnj-base` | Byte-level BPE (Persian) | 128 |
| `microsoft/mdeberta-v3-base` | SentencePiece (multilingual) | 160 |

---

## 📐 Ground Rules

- **Input:** `body` only. No `title`, `rate`, `product_id`, or any other metadata.
- **Labels for training and checkpoint selection:** corrected Step-1 labels only.
- **Data:** full `train_supervised` for training, full validation set for evaluation every epoch.
- **Training:** at most 4 epochs, early stopping with patience 1 on the mean Macro F1 of both heads.
- **Fair comparison:** B and C use identical seeds, batch order, and settings. Only the sample weights differ.
- **No decision uses the test set.** Test metrics are reported on both raw labels (comparable with classmates) and corrected labels.

---

## 🖥️ Where Things Run

| Where | What |
| --- | --- |
| **Training machine** (friend's PC, RTX 4060 Ti, 16 GB) | Fine-tuning and prediction scripts only |
| **This notebook** (local machine) | Analysis of the saved outputs: curves, tables, decisions, charts |

Training is run by `scripts/train_transformer.py` from YAML configs. It resumes automatically from the last saved epoch if interrupted.

---

## 📦 Main Outputs

- Training history, logs, and run summaries for every run
- Validation and test probabilities for both heads
- Saved tokenizer and encoder config (for offline inference)
- Settings table with the reason for each choice
- B vs C decision, encoder comparison, and primary-model selection
- Test metrics, confusion matrices, and four report charts

---

## 🗂️ Notebook Map

| Step | Content |
| --- | --- |
| 1 | Imports and parameters |
| 2 | Load run outputs |
| 3 | Training settings and run comparison tables |
| 4 | Learning curves and overfitting check |
| 5 | B vs C decision |
| 6 | Test metrics and baseline comparison |
| 7 | Confusion matrices |
| 8 | Encoder comparison and primary model |
| 9 | Closeout facts |

---

# Step 0: preparing

In [1]:
# ── Read-only inspection, part 1: project setup, git, ignore rules ──
import json
import subprocess
import tomllib
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def run_git(*args):
    result = subprocess.run(["git", "-C", str(root), *args], check=False, capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()


# ── 1) pyproject.toml (full text) ──
show("pyproject.toml")
print((root / "pyproject.toml").read_text(encoding="utf-8"))

# ── 1b) Versions locked in uv.lock for the packages we care about ──
show("uv.lock: selected package versions")
lock = tomllib.loads((root / "uv.lock").read_text(encoding="utf-8"))
wanted = [
    "torch", "transformers", "tokenizers", "huggingface-hub", "safetensors",
    "sentencepiece", "protobuf", "accelerate", "pyyaml", "tqdm", "pandas",
    "pyarrow", "scikit-learn", "numpy", "duckdb", "matplotlib",
]
locked = {pkg["name"]: pkg.get("version", "-") for pkg in lock["package"]}
for name in wanted:
    print(f"{name:18s} {locked.get(name, 'NOT IN LOCK')}")

# ── 2) Package __init__.py ──
show("src/digikala_review_transformer/__init__.py")
init_text = (root / "src/digikala_review_transformer/__init__.py").read_text(encoding="utf-8")
print(init_text if init_text.strip() else "(empty file)")

# ── 6) .gitignore and ignore status of the paths this phase will create ──
show(".gitignore")
print((root / ".gitignore").read_text(encoding="utf-8"))

show("git check-ignore on future paths")
sample_paths = [
    "data/training_pack/train.parquet",
    "data/interim/token_cache/tooka_128/tokens.npy",
    "models/pretrained/TookaBERT-Base/model.safetensors",
    "artifacts/transformer_runs/tooka_B/best_model.pt",
    "artifacts/transformer_runs/tooka_B/last.pt",
    "outputs_to_send/tooka_B.zip",
]
for path in sample_paths:
    verdict = run_git("check-ignore", "-v", path)
    print(f"{path}\n    -> {verdict if verdict else 'NOT IGNORED'}")

# ── 7) scripts/ and configs/ ──
show("scripts/ and configs/")
for folder_name in ["scripts", "configs", "src/digikala_review_transformer/training"]:
    folder = root / folder_name
    if folder.exists():
        print(f"{folder_name}/ exists:", sorted(p.name for p in folder.iterdir()))
    else:
        print(f"{folder_name}/ does not exist")
print("tracked in git (scripts, configs):", run_git("ls-files", "scripts", "configs") or "(none)")

# ── 8) Branch and remote (hide any token if the URL contains one) ──
show("git branch and remote")
print("branch:", run_git("branch", "--show-current"))
print(run_git("remote", "-v"))

# ── 9) Notebook numbering ──
show("notebooks/")
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    print(notebook.name)

# ── 10) Where is a text-normalization function? (source cells only, no outputs) ──
show("'normaliz' in src/*.py")
hits = 0
for py_file in sorted((root / "src").rglob("*.py")):
    for line_number, line in enumerate(py_file.read_text(encoding="utf-8").splitlines(), 1):
        if "normaliz" in line.lower() and hits < 25:
            print(f"{py_file.relative_to(root)}:{line_number}: {line.strip()[:140]}")
            hits += 1
print("matches shown:", hits)

show("'normaliz' in notebook code cells")
hits = 0
# for notebook in sorted((root / "notebooks").glob("*.ipynb")):
#     cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
#     for cell_index, cell in enumerate(cells):
#         if cell.get("cell_type") != "code":
#             continue
#         source = "".join(cell.get("source", []))
#         matching = [ln.strip() for ln in source.splitlines() if "normaliz" in ln.lower()]
#         if matching and hits < 25:
#             print(f"{notebook.name} | cell {cell_index}: {matching[0][:140]}")
#             hits += 1
print("matches shown:", hits)


──────── pyproject.toml ────────
[project]
name = "digikala-review-transformer"
version = "0.1.0"
description = "Persian Digikala review classification: TF-IDF baseline vs fine-tuned Transformer"
readme = "README.md"
authors = [
    { name = "AliirezzaaZiarati", email = "arzxii1998@gmail.com" }
]
requires-python = ">=3.12"

# Training dependencies only: this is everything the training machine needs
dependencies = [
    "huggingface-hub>=2.4.0",
    "numpy>=2.5.4",
    "pandas>=3.1.1",
    "protobuf",
    "pyarrow>=26.0.1",
    "pyyaml>=6.0.3",
    "scikit-learn>=1.9.1",
    "sentencepiece",
    "torch>=2.15.0",
    "tqdm>=4.70.1",
    "transformers>=5.22.2",
]

[build-system]
requires = ["uv_build>=0.13.6,<0.14.0"]
build-backend = "uv_build"

[dependency-groups]
dev = [
    "ipykernel>=7.4.0",
]
# Analysis, charts and data-preparation tools (not needed on the training machine)
analysis = [
    "arabic-reshaper>=3.1.0",
    "duckdb>=2.0.0",
    "joblib>=1.6.0",
    "matplotlib>=3.11.2"

In [ ]:
# ── Read-only inspection, part 1b: normalization search in notebook code cells ──
import json  # noqa: F811
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)

hits = 0
unreadable = []
for notebook in sorted((root / "notebooks").glob("*.ipynb")):
    # Skip notebooks that are empty or not valid JSON, and remember them
    try:
        cells = json.loads(notebook.read_text(encoding="utf-8")).get("cells", [])
    except json.JSONDecodeError:
        unreadable.append((notebook.name, notebook.stat().st_size))
        continue
    for cell_index, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        matching = [ln.strip() for ln in source.splitlines() if "normaliz" in ln.lower()]
        if matching and hits < 25:
            print(f"{notebook.name} | cell {cell_index}: {matching[0][:140]}")
            hits += 1

print("matches shown:", hits)
print("unreadable notebooks (name, size in bytes):", unreadable or "none")

00_data_ingest_quality_gate.ipynb | cell 42: "body text normalization; original text stays in raw"
00_data_ingest_quality_gate.ipynb | cell 50: # Total number of rows before normalization
00_data_ingest_quality_gate.ipynb | cell 53: # Replace the old body with the normalized one
00_data_ingest_quality_gate.ipynb | cell 87: fig.text(0.04, 0.865, "Rows whose exact (normalized) review text appears more than once in clean_base",
00_data_ingest_quality_gate.ipynb | cell 91: bg.text(M, 0.17, f"Base: {total:,} reviews, all of clean_base. Words are whitespace-separated tokens of the normalized body. "
01_more_data_ingest_analysis.ipynb | cell 16: # from parsivar import Normalizer
01_more_data_ingest_analysis.ipynb | cell 22: # # Spacing fixes that only the SpellCheck+guard route found, not the Normalizer
01_more_data_ingest_analysis.ipynb | cell 43: print(df["recommendation_status"].value_counts(dropna=False, normalize=True).round(4))
04_label_correction_step1.ipynb | cell 15: # Kneedle core: 

# Inspect: Imports, Labels, Config, Data Columns

In [ ]:
%load_ext autoreload
%autoreload 2

# ── Read-only inspection, part 2: imports, labels, config, data columns ──
import ast
import inspect
import subprocess  # noqa: F811
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

from digikala_review_transformer import config, data_io, labels, utils

root = Path(config.ROOT)
package_dir = root / "src" / "digikala_review_transformer"
processed_dir = Path(config.DATA_PROCESSED)
predictions_dir = root / "reports" / "predictions"


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def run_git(*args):
    result = subprocess.run(["git", "-C", str(root), *args], check=False, capture_output=True, text=True)
    return (result.stdout or result.stderr).strip()


# ── A) Which third-party packages does each module import? (light-import check) ──
show("A) imports per module (top-level / inside functions)")
training_ok = {
    "torch", "transformers", "huggingface_hub", "pandas", "pyarrow", "sklearn", "yaml",
    "tqdm", "sentencepiece", "google", "numpy", "tokenizers", "safetensors",
    "digikala_review_transformer",
}


def root_names(nodes):
    names = set()
    for node in nodes:
        if isinstance(node, ast.Import):
            names.update(alias.name.split(".")[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:
            names.add(node.module.split(".")[0])
    return names


for module_name in ["config", "labels", "utils", "data_io", "metrics", "predictions", "plot_style"]:
    tree = ast.parse((package_dir / f"{module_name}.py").read_text(encoding="utf-8"))
    top_level = root_names(tree.body)
    everywhere = root_names(ast.walk(tree))
    nested_only = everywhere - top_level
    outside = sorted(n for n in everywhere if n not in sys.stdlib_module_names and n not in training_ok)
    print(f"{module_name:12s} top={sorted(top_level)} nested_only={sorted(nested_only)}")
    print(f"{'':12s} NOT in training set: {outside or 'none'}")

# ── B) Seed-like constants and simple values in config ──
show("B) config: simple constants")
for name in sorted(vars(config)):
    value = getattr(config, name)
    if name.isupper() and isinstance(value, (int, float, str, bool)):
        print(f"{name} = {value!r}")
path_names = [n for n in vars(config) if n.isupper() and isinstance(getattr(config, n), Path)]
print("number of Path constants:", len(path_names))
print("lines containing 'seed' in config.py:")
for line_number, line in enumerate((package_dir / "config.py").read_text(encoding="utf-8").splitlines(), 1):
    if "seed" in line.lower():
        print(f"  {line_number}: {line.strip()[:140]}")

# ── C) labels: constants and function signatures; data_io.read_split ──
show("C) labels module")
for name in sorted(n for n in vars(labels) if not n.startswith("_")):
    obj = getattr(labels, name)
    if inspect.isfunction(obj) and obj.__module__ == labels.__name__:
        print(f"def {name}{inspect.signature(obj)}")
    elif not (inspect.ismodule(obj) or inspect.isfunction(obj) or inspect.isclass(obj)):
        print(f"{name} = {repr(obj)[:300]}")

show("C2) data_io.read_split and utils.timer")
print("read_split", inspect.signature(data_io.read_split))
print((inspect.getdoc(data_io.read_split) or "(no docstring)")[:400])
print("timer", inspect.signature(utils.timer))

# ── D) Parquet schemas (no data loaded) ──
show("D) parquet schemas")
parquet_paths = [predictions_dir / f"tfidf12_{head}_B__{split}.parquet"
                 for head in ("rec", "band") for split in ("validation", "test")]
parquet_paths.append(processed_dir / "step2_sample_weights.parquet")
for path in parquet_paths:
    parquet_file = pq.ParquetFile(path)
    columns = [f"{field.name}:{field.type}" for field in parquet_file.schema_arrow]
    print(f"{path.name}  rows={parquet_file.metadata.num_rows:,}")
    print("   ", columns)

# ── E) CSV headers and unique label values (label columns only) ──
show("E) CSV headers and unique label values")
for split_name in ["train_supervised", "validation", "test"]:
    csv_path = processed_dir / f"{split_name}.csv"
    header = list(pd.read_csv(csv_path, nrows=0).columns)
    print(f"\n{split_name}.csv columns: {header}")
    label_columns = [c for c in ["recommendation_status", "step1_rec_label",
                                 "rating_band", "step1_rate_band_label"] if c in header]
    label_frame = pd.read_csv(csv_path, usecols=label_columns, dtype=str, keep_default_na=False)
    print(f"  rows: {len(label_frame):,}")
    for column in label_columns:
        print(f"  {column}: {label_frame[column].value_counts(dropna=False).to_dict()}")

# ── F) Git: the empty notebook 06 and tracked files under models/ ──
show("F) git: notebook 06 and models/")
notebook_06 = "notebooks/06_masking_and_artifacts.ipynb"
print("size on disk:", (root / notebook_06).stat().st_size)
print("size in HEAD:", run_git("cat-file", "-s", f"HEAD:{notebook_06}"))
print("last commits touching it:", run_git("log", "--oneline", "-3", "--", notebook_06) or "(none)")
print("git status of it:", run_git("status", "--short", "--", notebook_06) or "(clean)")
print("tracked files under models/:", run_git("ls-files", "models")[:600] or "(none)")


──────── A) imports per module (top-level / inside functions) ────────
config       top=['pathlib'] nested_only=[]
             NOT in training set: none
labels       top=['__future__', 'numpy', 'pandas'] nested_only=[]
             NOT in training set: none
utils        top=['__future__', 'contextlib', 'csv', 'datetime', 'hashlib', 'json', 'pathlib', 'time'] nested_only=[]
             NOT in training set: none
data_io      top=['__future__', 'itertools', 'numpy', 'pandas'] nested_only=[]
             NOT in training set: none
metrics      top=['numpy', 'pandas', 'sklearn'] nested_only=[]
             NOT in training set: none
predictions  top=['datetime', 'numpy', 'pandas', 'pathlib', 'zoneinfo'] nested_only=[]
             NOT in training set: none
plot_style   top=['__future__', 'arabic_reshaper', 'colorsys', 'matplotlib', 'numpy', 'pathlib', 're', 'struct'] nested_only=['bidi']
             NOT in training set: ['arabic_reshaper', 'bidi', 'matplotlib']

──────── B) config: simple

# Inspect: Labels and Metrics Source

In [ ]:
# ── Read-only inspection, part 3: labels and metrics source and behavior ──
import inspect

import numpy as np
import pandas as pd

from digikala_review_transformer import labels, metrics


def show(title):
    print(f"\n{'─' * 8} {title} {'─' * 8}")


def print_source(obj_name, module):
    # Print the source of a function, or a note if it does not exist
    obj = getattr(module, obj_name, None)
    show(f"{module.__name__.split('.')[-1]}.{obj_name}")
    print(inspect.getsource(obj) if obj is not None else "NOT FOUND")


print_source("class_weights", labels)
print_source("encode", labels)
print_source("_valid", metrics)
print_source("evaluate", metrics)

show("metrics.per_class_table signature and doc")
print(inspect.signature(metrics.per_class_table))
print(inspect.getdoc(metrics.per_class_table))

# ── Toy calls: one missing label and a prediction that is wrong once ──
show("toy calls")
toy_true = pd.array([0, 1, 2, 1, None], dtype="Int8")
toy_pred = np.array([0, 1, 1, 1, 2])
toy_calls = {
    "class_weights": lambda: labels.class_weights(toy_true),
    "evaluate": lambda: metrics.evaluate(toy_true, toy_pred),
    "per_class_table": lambda: metrics.per_class_table(toy_true, toy_pred, "rec"),
}
for name, call in toy_calls.items():
    try:
        print(f"{name}:\n{call()}\n")
    except Exception as error:  # noqa: BLE001
        print(f"{name}: {type(error).__name__}: {error}\n")


──────── labels.class_weights ────────
def class_weights(y, n_classes=3):
    """Balanced weights n / (k * n_c) as {code: weight}, computed from training labels only."""
    y = np.asarray(y, dtype=int)
    counts = np.bincount(y, minlength=n_classes)
    return {i: float(len(y) / (n_classes * c)) for i, c in enumerate(counts)}


──────── labels.encode ────────
def encode(series, head):
    """Map label strings to int codes (nullable Int8). Missing stays <NA>; unknown values raise."""
    out = series.map(label_map(head))
    bad = series.notna() & out.isna()
    if bad.any():
        raise ValueError(f"unknown labels for head '{head}': {series[bad].unique()[:10]}")
    return out.astype("Int8")


──────── metrics._valid ────────
def _valid(y_true, y_pred):
    """Keep only rows with a valid true label; return int arrays."""
    yt = pd.Series(y_true).reset_index(drop=True)
    yp = np.asarray(y_pred)
    assert len(yt) == len(yp), "y_true and y_pred length differ"
    m = yt.notna().

# U2: pyproject.toml، .gitignore و git

In [5]:
# terminal prompts

# Inspect: Windows Wheels in uv.lock and config.py Paths

In [6]:
# ── Read-only inspection, part 4: Windows CUDA wheels in the lock file and config.py ──
import tomllib
from pathlib import Path

from digikala_review_transformer import config

root = Path(config.ROOT)

# ── A) torch / torchvision entries in uv.lock and their Windows cp312 wheels ──
lock = tomllib.loads((root / "uv.lock").read_text(encoding="utf-8"))
for package in lock["package"]:
    if package["name"] in ("torch", "torchvision"):
        wheel_urls = [w.get("url", "") for w in package.get("wheels", [])]
        windows_wheels = [u.rsplit("/", 1)[-1] for u in wheel_urls if "win_amd64" in u and "cp312" in u]
        print(package["name"], package["version"], "| source:", package.get("source"))
        print("   windows cp312 wheels:", windows_wheels or "NONE")

# ── B) How config.py defines paths (first 60 lines) ──
print("\n--- config.py (first 60 lines) ---")
config_path = root / "src" / "digikala_review_transformer" / "config.py"
for line_number, line in enumerate(config_path.read_text(encoding="utf-8").splitlines()[:60], 1):
    print(f"{line_number:3d}: {line}")

torch 2.15.0 | source: {'registry': 'https://pypi.org/simple'}
   windows cp312 wheels: NONE
torch 2.15.0+cu134 | source: {'registry': 'https://download.pytorch.org/whl/cu134'}
   windows cp312 wheels: ['torch-2.15.0%2Bcu134-cp312-cp312-win_amd64.whl']
torchvision 0.29.2 | source: {'registry': 'https://pypi.org/simple'}
   windows cp312 wheels: NONE
torchvision 0.29.2+cu134 | source: {'registry': 'https://download.pytorch.org/whl/cu134'}
   windows cp312 wheels: NONE

--- config.py (first 60 lines) ---
  1: from pathlib import Path
  2: 
  3: # Project root: src/digikala_review_transformer/config.py -> three levels up
  4: ROOT = Path(__file__).resolve().parents[2]
  5: 
  6: # Folders (data and artifacts contents are not tracked by Git)
  7: DATA_RAW = ROOT / "data" / "raw"
  8: DATA_INTERIM = ROOT / "data" / "interim"
  9: DATA_PROCESSED = ROOT / "data" / "processed"
 10: ARTIFACTS = ROOT / "artifacts"
 11: REPORTS = ROOT / "reports"
 12: 
 13: # Cleaned data: DuckDB file lives direc

# Test: Run Config Loader, Hash and Validation Errors

In [7]:
# ── Test: run_config loading, hashing and validation errors ──
import copy
import tempfile
from pathlib import Path

import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

configs_dir = Path(config.ROOT) / "configs" / "transformer"
base_path = configs_dir / "base.yaml"
tooka_path = configs_dir / "tooka_B.yaml"
failed = []


def report(name, passed, detail=""):
    print(f"[{'PASS' if passed else 'FAIL'}] {name} {detail}")
    if not passed:
        failed.append(name)


# ── 1) A valid config loads, with the project seed and normalized floats ──
cfg = run_config.load_run_config(tooka_path)
report("tooka_B loads", cfg["run_name"] == "tooka_B" and cfg["training"]["variant"] == "B")
report("seed comes from project config", cfg["seed"] == config.SPLIT_SEED, f"(seed={cfg['seed']})")
report("'extends' is removed", "extends" not in cfg)
report("floats are floats", isinstance(cfg["training"]["lr_encoder"], float))
print("run dir:", run_config.run_dir(cfg))

# ── 2) Hash: deterministic, ignores runtime-only keys, reacts to real changes ──
hash_a = run_config.config_hash(cfg)
report("hash is deterministic", hash_a == run_config.config_hash(run_config.load_run_config(tooka_path))
       and len(hash_a) == 64, f"({hash_a[:12]})")
runtime_changed = copy.deepcopy(cfg)
runtime_changed["training"]["num_workers"] = 0
runtime_changed["training"]["eval_batch_size"] = 64
report("hash ignores runtime-only keys", run_config.config_hash(runtime_changed) == hash_a)
batch_changed = copy.deepcopy(cfg)
batch_changed["training"]["batch_size"] = 16
report("hash changes with batch_size", run_config.config_hash(batch_changed) != hash_a)
variant_changed = copy.deepcopy(cfg)
variant_changed["training"]["variant"] = "C"
report("hash changes with variant", run_config.config_hash(variant_changed) != hash_a)

# ── 3) Resolved config can be written and read back unchanged ──
with tempfile.TemporaryDirectory() as tmp_name:
    tmp = Path(tmp_name)
    run_config.dump_resolved_config(cfg, tmp / "sub" / "config_resolved.yaml")
    reloaded = yaml.safe_load((tmp / "sub" / "config_resolved.yaml").read_text(encoding="utf-8"))
    report("resolved config round-trips", reloaded == cfg)

    # ── 4) Bad configs must be rejected with a clear message ──
    def write_child(name, training=None, top=None):
        training_lines = {"variant": "B"} | (training or {})
        lines = [f"extends: {base_path.as_posix()}", "run_name: t", "model:",
                 "  pretrained_path: p", "  max_length: 128", "training:"]
        lines += [f"  {key}: {value}" for key, value in training_lines.items()]
        lines += [f"{key}: {value}" for key, value in (top or {}).items()]
        path = tmp / name
        path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        return path

    # Control: the helper itself produces a valid config
    try:
        run_config.load_run_config(write_child("control.yaml"))
        report("control child config loads", True)
    except run_config.ConfigError as error:
        report("control child config loads", False, f"\n{error}")

    bad_cases = [
        ("seed in YAML is rejected", {"top": {"seed": 7}}, "seed"),
        ("unknown key is rejected", {"training": {"lr_encodr": "1.0e-5"}}, "lr_encodr"),
        ("encoder LR above head LR is rejected", {"training": {"lr_encoder": "3.0e-4"}}, "lr_encoder"),
        ("unknown variant is rejected", {"training": {"variant": "D"}}, "variant"),
        ("1e-4 read as text is rejected", {"training": {"lr_heads": "1e-4"}}, "lr_heads"),
    ]
    for index, (name, kwargs, snippet) in enumerate(bad_cases):
        path = write_child(f"bad_{index}.yaml", **kwargs)
        try:
            run_config.load_run_config(path)
            report(name, False, "(no error was raised)")
        except run_config.ConfigError as error:
            message = str(error)
            report(name, "[FAIL]" in message and snippet in message)

print("\nALL PASSED" if not failed else f"\nFAILED: {failed}")

[PASS] tooka_B loads 
[PASS] seed comes from project config (seed=2026)
[PASS] 'extends' is removed 
[PASS] floats are floats 
run dir: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/artifacts/transformer_runs/tooka_B
[PASS] hash is deterministic (4d353853f2f5)
[PASS] hash ignores runtime-only keys 
[PASS] hash changes with batch_size 
[PASS] hash changes with variant 
[PASS] resolved config round-trips 
[PASS] control child config loads 
[PASS] seed in YAML is rejected 
[PASS] unknown key is rejected 
[PASS] encoder LR above head LR is rejected 
[PASS] unknown variant is rejected 
[PASS] 1e-4 read as text is rejected 

ALL PASSED


# Inspect: Source Files for the Training Pack

In [ ]:
# ---- Section: Imports ----
import inspect
import subprocess  # noqa: F811
from pathlib import Path

import pyarrow.parquet as pq

from digikala_review_transformer import config, data_io, labels

# ---- Section: Helpers ----
SKIP_DIR_NAMES = {"tfidf_cache", "token_cache", "__pycache__", ".ipynb_checkpoints"}


def list_files(base, suffixes):
    """Return (path, size_mb) for files under base, skipping cache folders."""
    found = []
    if not base.exists():
        return found
    for p in sorted(base.rglob("*")):
        if p.is_file() and p.suffix.lower() in suffixes:  # noqa: SIM102
            if SKIP_DIR_NAMES.isdisjoint(p.parts):
                found.append((p, p.stat().st_size / 1e6))
    return found


# ---- Section: CSV and parquet files in data/ ----
print("=== data/ (csv, parquet) ===")
for base in (config.DATA_RAW, config.DATA_INTERIM, config.DATA_PROCESSED):
    for p, mb in list_files(base, {".csv", ".parquet"}):
        print(f"{p.relative_to(config.ROOT)}  [{mb:,.1f} MB]")

# ---- Section: CSV headers of split-like files ----
print("\n=== CSV header lines ===")
for base in (config.DATA_INTERIM, config.DATA_PROCESSED):
    for p, _ in list_files(base, {".csv"}):
        with open(p, encoding="utf-8") as f:
            print(f"{p.relative_to(config.ROOT)}: {f.readline().strip()}")

# ---- Section: Step 2 weight files anywhere in the project ----
print("\n=== Files with 'weight' or 'step2' in the name ===")
for base in (config.DATA_INTERIM, config.DATA_PROCESSED, config.ARTIFACTS, config.REPORTS):
    for p, mb in list_files(base, {".parquet", ".csv", ".json"}):
        low = p.name.lower()
        if "weight" in low or "step2" in low:
            line = f"{p.relative_to(config.ROOT)}  [{mb:,.1f} MB]"
            if p.suffix == ".parquet":
                pf = pq.ParquetFile(p)
                line += f"  rows={pf.metadata.num_rows:,}  cols={pf.schema_arrow.names}"
            print(line)

# ---- Section: Function signatures I will call ----
print("\n=== Signatures ===")
print("data_io.read_split", inspect.signature(data_io.read_split))
print("labels.encode", inspect.signature(labels.encode))
print("labels.class_weights", inspect.signature(labels.class_weights))

# ---- Section: Git commit (for the manifest) ----
print("\n=== Git ===")
try:
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=config.ROOT,
                          capture_output=True, text=True, check=True).stdout.strip()
    print("HEAD:", head)
except Exception as exc:  # noqa: BLE001
    print("git not usable:", exc)

=== data/ (csv, parquet) ===
data/raw/digikala-comments.csv  [1,278.5 MB]
data/raw/digikala-products.csv  [258.7 MB]
data/interim/body_groups_rating_band.parquet  [0.2 MB]
data/interim/body_groups_recommendation_status.parquet  [0.2 MB]
data/interim/cap_dropped_rows.parquet  [6.1 MB]
data/interim/cap_overcap_bodies.parquet  [0.0 MB]
data/interim/dropped_tables_archive/body_backup_latin.parquet  [0.1 MB]
data/interim/dropped_tables_archive/defect_manifest.parquet  [10.3 MB]
data/interim/dropped_tables_archive/dup_plan.parquet  [0.1 MB]
data/interim/dropped_tables_archive/dup_plan_s3.parquet  [0.0 MB]
data/interim/dropped_tables_archive/len_sample.parquet  [17.7 MB]
data/interim/dropped_tables_archive/rule_summary.parquet  [0.0 MB]
data/interim/explorer_candidates_tfidf12/tfidf12_band_B.parquet  [0.8 MB]
data/interim/explorer_candidates_tfidf12/tfidf12_rec_B.parquet  [0.9 MB]
data/interim/step2_oof/oof_band_fold0.parquet  [12.9 MB]
data/interim/step2_oof/oof_band_fold1.parquet  [12.9 MB]

# Inspect: Local Models and Tokenizer Behavior

In [ ]:
# ---- Section: Imports ----
import gc
import time

import numpy as np
import pandas as pd
from transformers import AutoConfig, AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Parameters ----
MODELS = [
    ("TookaBERT-Base", 128),
    ("roberta-fa-zwnj-base", 128),
    ("mdeberta-v3-base", 160),
]
SAMPLE_ROWS = 20_000
PERCENTILES = [50, 90, 95, 99, 99.9]
TOTAL_TRAIN_ROWS = 2_860_236

models_dir = config.ROOT / "models" / "pretrained"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])

# ---- Section: What is on disk ----
print("=== models/pretrained ===")
if not models_dir.exists():
    print("MISSING:", models_dir)
else:
    for d in sorted(p for p in models_dir.iterdir() if p.is_dir()):
        files = sorted(f for f in d.iterdir() if f.is_file())
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f"{d.name}: {len(files)} files, {total_mb:,.0f} MB")
        print("   ", [f.name for f in files])

# ---- Section: Sample of training bodies ----
train_bodies = pd.read_parquet(pack_dir / "train.parquet", columns=["body"])["body"]
sample = train_bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del train_bodies
gc.collect()
print(f"\nsample size: {len(sample):,}")

# ---- Section: Per-model tokenizer and config facts ----
for name, max_length in MODELS:
    path = models_dir / name
    print(f"\n=== {name} (max_length={max_length}) ===")
    if not path.exists():
        print("MISSING -> model not downloaded on this machine")
        continue
    try:
        tok = AutoTokenizer.from_pretrained(path)
        mcfg = AutoConfig.from_pretrained(path)
    except Exception as exc:  # noqa: BLE001
        print("LOAD ERROR:", type(exc).__name__, str(exc)[:300])
        continue

    print("tokenizer class:", type(tok).__name__, "| is_fast:", tok.is_fast)
    print("len(tokenizer):", len(tok), "| model_max_length:", tok.model_max_length,
          "| padding_side:", tok.padding_side)
    print("special ids -> cls:", tok.cls_token_id, "sep:", tok.sep_token_id, "pad:", tok.pad_token_id,
          "bos:", tok.bos_token_id, "eos:", tok.eos_token_id, "unk:", tok.unk_token_id)
    print("model_type:", mcfg.model_type, "| architectures:", getattr(mcfg, "architectures", None))
    print("hidden_size:", mcfg.hidden_size, "| config.vocab_size:", mcfg.vocab_size,
          "| max_position_embeddings:", getattr(mcfg, "max_position_embeddings", None),
          "| pad_token_id:", getattr(mcfg, "pad_token_id", None))

    # Keys and ids of one truncated example (ids only, no text).
    one = tok(sample[0], truncation=True, max_length=max_length)
    ids = one["input_ids"]
    print("encoding keys:", list(one.keys()))
    print("first 8 ids:", ids[:8], "| last 3 ids:", ids[-3:], "| len:", len(ids))
    print("starts with cls:", ids[0] == tok.cls_token_id, "| ends with sep:", ids[-1] == tok.sep_token_id)

    # Length distribution without truncation, to see how much gets cut.
    start = time.perf_counter()
    enc = tok(sample, truncation=False, add_special_tokens=True, return_attention_mask=False)
    seconds = time.perf_counter() - start
    lengths = np.array([len(x) for x in enc["input_ids"]])
    pct = np.percentile(lengths, PERCENTILES)
    print("length percentiles", dict(zip(PERCENTILES, pct.round(1))), "| max:", lengths.max())
    print(f"truncated at {max_length}: {(lengths > max_length).mean() * 100:.2f}% of rows")
    print(f"max id seen: {max(max(x) for x in enc['input_ids'])} (must be < vocab size)")
    print(f"tokenize speed: {SAMPLE_ROWS / seconds:,.0f} rows/s -> "
          f"~{TOTAL_TRAIN_ROWS / (SAMPLE_ROWS / seconds) / 60:.1f} min for the full train (single call)")

    del tok, enc
    gc.collect()

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== models/pretrained ===
TookaBERT-Base: 9 files, 986 MB
    ['.gitattributes', 'README.md', 'config.json', 'generation_config.json', 'model.safetensors', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json']
mdeberta-v3-base: 8 files, 2,286 MB
    ['.gitattributes', 'README.md', 'config.json', 'generator_config.json', 'pytorch_model.bin', 'pytorch_model.generator.bin', 'spm.model', 'tokenizer_config.json']
roberta-fa-zwnj-base: 9 files, 478 MB
    ['.gitattributes', 'README.md', 'config.json', 'merges.txt', 'pytorch_model.bin', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']

sample size: 20,000

=== TookaBERT-Base (max_length=128) ===
tokenizer class: TokenizersBackend | is_fast: True
len(tokenizer): 48000 | model_max_length: 512 | padding_side: right
special ids -> cls: None sep: 3 pad: 0 bos: 2 eos: 3 unk: 1
model_type: bert | architectures: ['BertForMaskedLM']
hidden_size: 768 | config.vocab_size: 48000 | ma

[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


tokenizer class: DebertaV2Tokenizer | is_fast: True
len(tokenizer): 250102 | model_max_length: 1000000000000000019884624838656 | padding_side: right
special ids -> cls: 1 sep: 2 pad: 0 bos: 1 eos: 2 unk: 3
model_type: deberta-v2 | architectures: None
hidden_size: 768 | config.vocab_size: 251000 | max_position_embeddings: 512 | pad_token_id: 0
encoding keys: ['input_ids', 'token_type_ids', 'attention_mask']
first 8 ids: [1, 260, 22464, 2144, 44146, 13019, 1506, 12989] | last 3 ids: [49123, 377, 2] | len: 44
starts with cls: True | ends with sep: True
length percentiles {50: np.float64(17.0), 90: np.float64(48.0), 95: np.float64(66.0), 99: np.float64(118.0), 99.9: np.float64(280.0)} | max: 694
truncated at 160: 0.46% of rows
max id seen: 249556 (must be < vocab size)
tokenize speed: 100,734 rows/s -> ~0.5 min for the full train (single call)


# Inspect: Tokenizer Correctness and the Mistral Regex Warning

In [ ]:
# ---- Section: Imports ----
import gc

import numpy as np
import pandas as pd
import sentencepiece as spm
from transformers import AutoConfig, AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Parameters ----
MODELS = [
    ("TookaBERT-Base", 128),
    ("roberta-fa-zwnj-base", 128),
    ("mdeberta-v3-base", 160),
]
SAMPLE_ROWS = 20_000
SHOW_MISMATCHES = 3
PROBE_TEXT = "x"

models_dir = config.ROOT / "models" / "pretrained"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])

# ---- Section: Same sample as the previous inspection ----
train_bodies = pd.read_parquet(pack_dir / "train.parquet", columns=["body"])["body"]
sample = train_bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del train_bodies
gc.collect()


# ---- Section: Helpers ----
def encode_ids(tok, texts, max_length):
    """Return input_ids lists; max_length=None means no truncation."""
    kwargs = {"return_attention_mask": False, "return_token_type_ids": False}
    if max_length is None:
        return tok(texts, truncation=False, **kwargs)["input_ids"]
    return tok(texts, truncation=True, max_length=max_length, **kwargs)["input_ids"]


def report(ok, name, hint=""):
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


# ---- Section: Checks per model ----
for name, max_length in MODELS:
    path = models_dir / name
    print(f"\n=== {name} (max_length={max_length}) ===")
    tok = AutoTokenizer.from_pretrained(path)
    mcfg = AutoConfig.from_pretrained(path)

    # First/last ids come from a probe text, never from cls_token_id.
    probe = tok(PROBE_TEXT, return_token_type_ids=False)["input_ids"]
    bare = tok(PROBE_TEXT, add_special_tokens=False, return_token_type_ids=False)["input_ids"]
    first_id, last_id = probe[0], probe[-1]
    print(f"probe ids: {probe} | special tokens added: {len(probe) - len(bare)} "
          f"| first_id={first_id} last_id={last_id}")
    print("config.type_vocab_size:", getattr(mcfg, "type_vocab_size", None))

    full = encode_ids(tok, sample, None)
    cut = encode_ids(tok, sample, max_length)
    full_len = np.array([len(x) for x in full])
    cut_len = np.array([len(x) for x in cut])
    needs_cut = full_len > max_length

    report(all(x[0] == first_id for x in cut), "every row starts with first_id", "tokenizer adds no start token on some rows")
    report(all(x[-1] == last_id for x in cut), "every row (also truncated ones) ends with last_id",
           "truncation removed the end token")
    report(bool((cut_len <= max_length).all()
                and (cut_len[needs_cut] == max_length).all()
                and (cut_len[~needs_cut] == full_len[~needs_cut]).all()),
           f"truncation gives exactly min(len, {max_length}); {int(needs_cut.sum())} rows were cut",
           "length logic differs from expectation")
    report(tok.pad_token_id == getattr(mcfg, "pad_token_id", None),
           f"tokenizer pad id ({tok.pad_token_id}) equals model config pad id",
           "padding would hit a real token")

    core = np.concatenate([np.asarray(x[1:-1]) for x in full])
    unk_rate = float((core == tok.unk_token_id).mean()) * 100
    print(f"unk rate: {unk_rate:.4f}% of {len(core):,} tokens | mean length: {full_len.mean():.2f}")

    # ---- Section: mdeberta only: warning flag and reference tokenizer ----
    if name == "mdeberta-v3-base":
        try:
            tok_fix = AutoTokenizer.from_pretrained(path, fix_mistral_regex=True)
            cut_fix = encode_ids(tok_fix, sample, max_length)
            same = sum(a == b for a, b in zip(cut, cut_fix))
            report(same == len(cut), f"ids identical with fix_mistral_regex=True ({same:,}/{len(cut):,} rows)",
                   "the flag changes tokenization; I must decide which one is correct")
            del tok_fix
        except Exception as exc:  # noqa: BLE001
            print("flag test could not run:", type(exc).__name__, str(exc)[:200])

        sp = spm.SentencePieceProcessor(model_file=str(path / "spm.model"))
        sp_ids = sp.encode(sample)
        fast_core = [x[1:-1] for x in full]
        bad = [i for i, (a, b) in enumerate(zip(fast_core, sp_ids)) if a != b]
        report(not bad, f"fast tokenizer equals raw sentencepiece ({len(sample) - len(bad):,}/{len(sample):,} rows)",
               "see mismatches below")
        for i in bad[:SHOW_MISMATCHES]:
            a, b = fast_core[i], sp_ids[i]
            pos = next((k for k in range(min(len(a), len(b))) if a[k] != b[k]), min(len(a), len(b)))
            print(f"  row {i}: fast_len={len(a)} sp_len={len(b)} first_diff_pos={pos} "
                  f"fast={a[max(0, pos - 2):pos + 3]} sp={b[max(0, pos - 2):pos + 3]}")
        del sp, sp_ids, fast_core

    del tok, full, cut
    gc.collect()


=== TookaBERT-Base (max_length=128) ===
probe ids: [2, 46517, 3] | special tokens added: 2 | first_id=2 last_id=3
config.type_vocab_size: 2
[PASS] every row starts with first_id
[PASS] every row (also truncated ones) ends with last_id
[PASS] truncation gives exactly min(len, 128); 48 rows were cut
[PASS] tokenizer pad id (0) equals model config pad id
unk rate: 0.0819% of 268,710 tokens | mean length: 15.44

=== roberta-fa-zwnj-base (max_length=128) ===
probe ids: [0, 98, 2] | special tokens added: 2 | first_id=0 last_id=2
config.type_vocab_size: 1
[PASS] every row starts with first_id
[PASS] every row (also truncated ones) ends with last_id
[PASS] truncation gives exactly min(len, 128); 50 rows were cut
[PASS] tokenizer pad id (1) equals model config pad id
unk rate: 0.0000% of 279,352 tokens | mean length: 15.97

=== mdeberta-v3-base (max_length=160) ===


[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


probe ids: [1, 260, 330, 2] | special tokens added: 2 | first_id=1 last_id=2
config.type_vocab_size: 0
[PASS] every row starts with first_id
[PASS] every row (also truncated ones) ends with last_id
[PASS] truncation gives exactly min(len, 160); 92 rows were cut
[PASS] tokenizer pad id (0) equals model config pad id
unk rate: 0.2480% of 447,595 tokens | mean length: 24.38
[FAIL] ids identical with fix_mistral_regex=True (0/20,000 rows) -> the flag changes tokenization; I must decide which one is correct
[FAIL] fast tokenizer equals raw sentencepiece (19,226/20,000 rows) -> see mismatches below
  row 19: fast_len=310 sp_len=309 first_diff_pos=122 fast=[377, 823, 3, 11490, 2409] sp=[377, 823, 260, 11490, 2409]
  row 27: fast_len=20 sp_len=19 first_diff_pos=15 fast=[17761, 377, 3, 1374, 260] sp=[17761, 377, 1374, 260, 11486]
  row 28: fast_len=45 sp_len=41 first_diff_pos=12 fast=[17036, 377, 3, 914, 7736] sp=[17036, 377, 1876, 7736, 260]


#  Inspect: Why Fast mDeBERTa Differs from SentencePiece

In [11]:
# ---- Section: Imports ----
import gc
import unicodedata
from collections import Counter

import pandas as pd
import sentencepiece as spm
from transformers import AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Parameters ----
SAMPLE_ROWS = 20_000
TOP_CHARS = 15
SHOW_ROWS = 3
HEAD_IDS = 10

path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])

# ---- Section: Same sample as before ----
train_bodies = pd.read_parquet(pack_dir / "train.parquet", columns=["body"])["body"]
sample = train_bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del train_bodies
gc.collect()

# ---- Section: Three tokenizers, no special tokens, no truncation ----
tok_default = AutoTokenizer.from_pretrained(path)
tok_flag = AutoTokenizer.from_pretrained(path, fix_mistral_regex=True)
sp = spm.SentencePieceProcessor(model_file=str(path / "spm.model"))
UNK = tok_default.unk_token_id
print("fast unk id:", UNK, "| sentencepiece unk id:", sp.unk_id())

kwargs = {"add_special_tokens": False, "truncation": False,
          "return_attention_mask": False, "return_token_type_ids": False}
enc_default = tok_default(sample, return_offsets_mapping=True, **kwargs)
ids_default = enc_default["input_ids"]
ids_flag = tok_flag(sample, **kwargs)["input_ids"]
ids_sp = sp.encode(sample)

# ---- Section: How the three versions compare ----
bad = [i for i in range(SAMPLE_ROWS) if ids_default[i] != ids_sp[i]]
bad_with_unk = [i for i in bad if UNK in ids_default[i]]
rows_with_unk = sum(UNK in x for x in ids_default)
sp_unk_rows = sum(sp.unk_id() in x for x in ids_sp)
print(f"\ndefault != sentencepiece: {len(bad):,} rows")
print(f"  of these, fast output contains unk: {len(bad_with_unk):,}")
print(f"  of these, no unk in fast output:    {len(bad) - len(bad_with_unk):,}")
print(f"rows with unk -> fast: {rows_with_unk:,} | sentencepiece: {sp_unk_rows:,}")
print(f"unk tokens -> fast: {sum(x.count(UNK) for x in ids_default):,} "
      f"| sentencepiece: {sum(x.count(sp.unk_id()) for x in ids_sp):,}")

same_flag_sp = sum(a == b for a, b in zip(ids_flag, ids_sp))
same_flag_default = sum(a == b for a, b in zip(ids_flag, ids_default))
print(f"\nflag == sentencepiece: {same_flag_sp:,}/{SAMPLE_ROWS:,}")
print(f"flag == default:       {same_flag_default:,}/{SAMPLE_ROWS:,}")
print(f"unk tokens with flag: {sum(x.count(UNK) for x in ids_flag):,}")
shortest = min(range(SAMPLE_ROWS), key=lambda i: len(ids_sp[i]))
for label, row in (("row 0", 0), ("shortest row", shortest)):
    print(f"{label}: default={ids_default[row][:HEAD_IDS]} flag={ids_flag[row][:HEAD_IDS]} "
          f"sp={ids_sp[row][:HEAD_IDS]} (lens {len(ids_default[row])}/{len(ids_flag[row])}/{len(ids_sp[row])})")

# ---- Section: Characters inside fast unk spans (code points only) ----
char_counter = Counter()
empty_spans = 0
for text, ids, offsets in zip(sample, ids_default, enc_default["offset_mapping"]):
    for token_id, (start, end) in zip(ids, offsets):
        if token_id != UNK:
            continue
        if start == end:
            empty_spans += 1
        for ch in text[start:end]:
            char_counter[ch] += 1
print(f"\nunk spans with empty offsets: {empty_spans:,}")
print(f"distinct characters inside unk spans: {len(char_counter)}")
print("code point | category | unicode name | count | fast ids alone | sentencepiece ids alone")
for ch, count in char_counter.most_common(TOP_CHARS):
    name = unicodedata.name(ch, "?")
    fast_alone = tok_default(ch, **{k: v for k, v in kwargs.items()})["input_ids"]
    print(f"U+{ord(ch):04X} | {unicodedata.category(ch)} | {name} | {count:,} | {fast_alone} | {sp.encode(ch)}")

# ---- Section: A few mismatched rows without unk in fast output ----
others = [i for i in bad if i not in set(bad_with_unk)]
for i in others[:SHOW_ROWS]:
    a, b = ids_default[i], ids_sp[i]
    pos = next((k for k in range(min(len(a), len(b))) if a[k] != b[k]), min(len(a), len(b)))
    print(f"row {i}: fast_len={len(a)} sp_len={len(b)} first_diff_pos={pos} "
          f"fast={a[max(0, pos - 2):pos + 3]} sp={b[max(0, pos - 2):pos + 3]}")

[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


fast unk id: 3 | sentencepiece unk id: 3

default != sentencepiece: 774 rows
  of these, fast output contains unk: 774
  of these, no unk in fast output:    0
rows with unk -> fast: 774 | sentencepiece: 0
unk tokens -> fast: 1,110 | sentencepiece: 0

flag == sentencepiece: 0/20,000
flag == default:       0/20,000
unk tokens with flag: 220,793
row 0: default=[260, 22464, 2144, 44146, 13019, 1506, 12989, 260, 11389, 3573] flag=[22464, 3, 1506, 44146, 3, 1441, 11805, 3, 377, 2409] sp=[260, 22464, 2144, 44146, 13019, 1506, 12989, 260, 11389, 3573] (lens 42/61/42)
shortest row: default=[555] flag=[5624] sp=[555] (lens 1/1/1)

unk spans with empty offsets: 0
distinct characters inside unk spans: 20
code point | category | unicode name | count | fast ids alone | sentencepiece ids alone
U+200C | Cf | ZERO WIDTH NON-JOINER | 1,096 | [260, 3] | []
U+1F979 | So | FACE HOLDING BACK TEARS | 3 | [260, 3] | [260, 244, 163, 169, 189]
U+1FAE1 | So | SALUTING FACE | 2 | [260, 3] | [260, 244, 163, 175, 1

# Inspect: mDeBERTa SentencePiece Special Pieces

In [12]:
# ---- Section: Imports ----
import json

import sentencepiece as spm

from digikala_review_transformer import config

# ---- Section: Parameters ----
path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
SPECIAL_NAMES = ["[PAD]", "[CLS]", "[SEP]", "[UNK]", "[MASK]"]
FIRST_IDS = 6

# ---- Section: Raw sentencepiece facts ----
sp = spm.SentencePieceProcessor(model_file=str(path / "spm.model"))
print("piece size:", sp.get_piece_size())
print("bos/eos/pad/unk ids:", sp.bos_id(), sp.eos_id(), sp.pad_id(), sp.unk_id())
for i in range(FIRST_IDS):
    print(i, sp.id_to_piece(i), "| control:", sp.is_control(i), "| unknown:", sp.is_unknown(i))
for name in SPECIAL_NAMES:
    print(name, "->", sp.piece_to_id(name))
print("empty text ->", sp.encode(""))

# ---- Section: HF config files (no text, only structure) ----
tok_cfg = json.loads((path / "tokenizer_config.json").read_text(encoding="utf-8"))
print("\ntokenizer_config keys (without added_tokens_decoder):")
print({k: v for k, v in tok_cfg.items() if k != "added_tokens_decoder"})
print("added tokens:", {k: v.get("content") for k, v in tok_cfg.get("added_tokens_decoder", {}).items()})
model_cfg = json.loads((path / "config.json").read_text(encoding="utf-8"))
print("\nconfig.json:", {k: model_cfg.get(k) for k in
      ("vocab_size", "pad_token_id", "type_vocab_size", "max_position_embeddings", "position_buckets", "relative_attention")})

piece size: 250101
bos/eos/pad/unk ids: 1 2 0 3
0 [PAD] | control: True | unknown: False
1 [CLS] | control: True | unknown: False
2 [SEP] | control: True | unknown: False
3 [UNK] | control: False | unknown: True
4 <0x00> | control: False | unknown: False
5 <0x01> | control: False | unknown: False
[PAD] -> 0
[CLS] -> 1
[SEP] -> 2
[UNK] -> 3
[MASK] -> 3
empty text -> []

tokenizer_config keys (without added_tokens_decoder):
{'do_lower_case': False, 'vocab_type': 'spm'}
added tokens: {}

config.json: {'vocab_size': 251000, 'pad_token_id': None, 'type_vocab_size': 0, 'max_position_embeddings': 512, 'position_buckets': 256, 'relative_attention': True}


#  Test: tokenizer_backend Config Key

In [1]:
# ---- Section: Imports ----
import shutil
import tempfile
from pathlib import Path

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config

# ---- Section: Helpers ----
CONFIG_DIR = config.ROOT / "configs" / "transformer"
CHILD_TEMPLATE = """extends: base.yaml
run_name: tooka_B
model:
  pretrained_path: models/pretrained/TookaBERT-Base
  max_length: 128
  tokenizer_backend: {backend}
training:
  variant: B
"""
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def load_child(tmp_dir, backend):
    """Write a child config next to a copy of base.yaml and load it."""
    child = tmp_dir / "child.yaml"
    child.write_text(CHILD_TEMPLATE.format(backend=backend), encoding="utf-8")
    return run_config.load_run_config(child)


# ---- Section: Checks ----
reference = run_config.load_run_config(CONFIG_DIR / "tooka_B.yaml")
check(reference["model"]["tokenizer_backend"] == "hf_fast", "tooka_B uses hf_fast from base.yaml",
      "add 'tokenizer_backend: hf_fast' under model: in base.yaml")

with tempfile.TemporaryDirectory() as tmp:
    tmp_dir = Path(tmp)
    shutil.copy(CONFIG_DIR / "base.yaml", tmp_dir / "base.yaml")

    same = load_child(tmp_dir, "hf_fast")
    check(run_config.config_hash(same) == run_config.config_hash(reference),
          "control: identical settings give identical hash",
          "the child template differs from tooka_B.yaml; fix the test before trusting the next check")

    other = load_child(tmp_dir, "sentencepiece")
    check(other["model"]["tokenizer_backend"] == "sentencepiece", "sentencepiece value is accepted", "check schema edit")
    check(run_config.config_hash(other) != run_config.config_hash(same),
          "tokenizer_backend changes the config hash", "key is missing from the hash; check RUNTIME_ONLY_KEYS")

    try:
        load_child(tmp_dir, "fast")
        check(False, "unknown backend is rejected", "add the check in _check_values")
    except run_config.ConfigError as error:
        check("tokenizer_backend" in str(error), "unknown backend is rejected with a clear message",
              "the error does not name the key")

print("\nALL PASSED" if all(results) else "\nFAILED")

[PASS] tooka_B uses hf_fast from base.yaml
[PASS] control: identical settings give identical hash
[PASS] sentencepiece value is accepted
[PASS] tokenizer_backend changes the config hash
[PASS] unknown backend is rejected with a clear message

ALL PASSED


# Test: Tokenization Backends, Cache and Round Trip

In [ ]:
# ---- Section: Imports ----
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from digikala_review_transformer import config
from digikala_review_transformer.training import run_config
from digikala_review_transformer.training import tokenization as tk

# ---- Section: Parameters ----
SAMPLE_ROWS = 20_000
SHORT_ROWS = 200
CACHE_ROWS = 5_000
MODELS = {
    # name: (backend, max_length, (first, last, pad, unk)) taken from the inspection cells
    "TookaBERT-Base": ("hf_fast", 128, (2, 3, 0, 1)),
    "roberta-fa-zwnj-base": ("hf_fast", 128, (0, 2, 1, 3)),
    "mdeberta-v3-base": ("sentencepiece", 160, (1, 2, 0, 3)),
}
models_dir = config.ROOT / "models" / "pretrained"
cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=tk.TokenizationError):
    try:
        function()
    except error_type:
        return True
    return False


# ---- Section: Sample (validation split, same rows for every model) ----
bodies = pd.read_parquet(pack_dir / "validation.parquet", columns=["body"])["body"]
sample = bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
short = sample[:SHORT_ROWS]
del bodies

# ---- Section: Per-model checks ----
bundles = {}
for name, (backend, max_length, expected_ids) in MODELS.items():
    print(f"\n=== {name} ({backend}, max_length={max_length}) ===")
    bundle = tk.load_tokenizer(models_dir / name, backend, max_length)
    bundles[name] = bundle
    got = (bundle.first_id, bundle.last_id, bundle.pad_id, bundle.unk_id)
    check(got == expected_ids, f"special ids first/last/pad/unk = {got}", f"expected {expected_ids}")

    arrays = tk.encode_texts(bundle, sample)
    check(len(arrays) == SAMPLE_ROWS, "one row of ids per text", "row count differs")
    try:
        tk.validate_arrays(arrays, bundle)
        check(True, "all rows start/end with special ids and fit max_length")
    except tk.TokenizationError as error:
        check(False, "row structure", str(error))

    direct = bundle.encode(short)
    check(all(arrays.row(i).tolist() == direct[i] for i in range(SHORT_ROWS)),
          "flat arrays equal bundle.encode output", "chunk concatenation or offsets are wrong")
    stats = tk.token_stats(arrays, bundle)
    print("   stats:", {k: stats[k] for k in ("mean_length", "rows_at_max_length_pct", "unk_pct_of_tokens",
                                              "rows_with_unk", "max_token_id")})

# ---- Section: sentencepiece vs the default Hugging Face tokenizer (rows without unk) ----
name, (backend, max_length, _) = "mdeberta-v3-base", MODELS["mdeberta-v3-base"]
hf_default = AutoTokenizer.from_pretrained(models_dir / name)
hf_ids = hf_default(sample, truncation=True, max_length=max_length, return_attention_mask=False,
                    return_token_type_ids=False)["input_ids"]
sp_arrays = tk.encode_texts(bundles[name], sample)
hf_unk = bundles[name].unk_id
clean_rows = [i for i, row in enumerate(hf_ids) if hf_unk not in row]
same = sum(sp_arrays.row(i).tolist() == hf_ids[i] for i in clean_rows)
check(same == len(clean_rows),
      f"sentencepiece == HF fast on rows without unk ({same:,}/{len(clean_rows):,})",
      "the two backends differ even without unk; stop and inspect")
print(f"[INFO] rows where HF fast has unk (excluded): {SAMPLE_ROWS - len(clean_rows):,}; "
      f"unk tokens from sentencepiece: {tk.token_stats(sp_arrays, bundles[name])['unk_tokens']:,}")

# ---- Section: Cache ----
bundle = bundles["TookaBERT-Base"]
texts = sample[:CACHE_ROWS]
reference = tk.encode_texts(bundle, texts)
reference_stats = tk.token_stats(reference, bundle)
old_chunks = (tk.ENCODE_CHUNK_ROWS, tk.STATS_CHUNK_ROWS)
tk.ENCODE_CHUNK_ROWS, tk.STATS_CHUNK_ROWS = 1_500, 700  # force several chunks
try:
    chunked = tk.encode_texts(bundle, texts)
    chunked_stats = tk.token_stats(chunked, bundle)
finally:
    tk.ENCODE_CHUNK_ROWS, tk.STATS_CHUNK_ROWS = old_chunks
check(chunked.equals(reference), "chunked encoding equals one-shot encoding", "chunk boundaries break rows")
check(chunked_stats == reference_stats, "chunked stats equal one-shot stats", "stats loop is wrong")


def no_texts():
    raise AssertionError("load_texts must not be called when the cache is valid")


with tempfile.TemporaryDirectory() as tmp:
    cache_dir = Path(tmp) / "cache"
    first, stats1, origin1 = tk.get_split_tokens(bundle, "validation", "sha_a", lambda: texts, CACHE_ROWS, cache_dir,
                                                 show_progress=False)
    check(origin1 == "built", "first call builds the cache", f"origin was {origin1}")
    second, stats2, origin2 = tk.get_split_tokens(bundle, "validation", "sha_a", no_texts, CACHE_ROWS, cache_dir)
    check(origin2 == "cache" and second.equals(first) and stats2 == stats1,
          "second call reads the identical arrays from cache", "cache round trip changed the data")
    _, _, origin3 = tk.get_split_tokens(bundle, "validation", "sha_b", lambda: texts, CACHE_ROWS, cache_dir,
                                        show_progress=False)
    check(origin3 == "built", "a different source sha256 invalidates the cache", "stale cache was reused")
    check(raises(lambda: tk.get_split_tokens(bundle, "validation", "sha_b", no_texts, CACHE_ROWS + 1, cache_dir,
                                             show_progress=False), AssertionError),
          "expected_rows mismatch is not served from cache", "cache ignored the row count")

    # Damage the id file on purpose: the cache must be rejected and rebuilt (a [WARN] line is expected).
    np.save(cache_dir / "validation_ids.npy", first.ids[:-5], allow_pickle=False)
    rebuilt, _, origin4 = tk.get_split_tokens(bundle, "validation", "sha_b", lambda: texts, CACHE_ROWS, cache_dir,
                                              show_progress=False)
    check(origin4 == "built" and rebuilt.equals(first), "damaged cache is rebuilt, not trusted", "bad cache was used")
    _, _, origin5 = tk.get_split_tokens(bundle, "validation", "sha_b", no_texts, CACHE_ROWS, cache_dir)
    check(origin5 == "cache", "rebuilt cache is valid again", "cache was not rewritten")

    _, _, origin6 = tk.get_split_tokens(bundle, "validation", "sha_a", lambda: texts, CACHE_ROWS, None,
                                        show_progress=False)
    check(origin6 == "built", "cache_dir=None builds without a disk cache", "unexpected origin")
    check(raises(lambda: tk.get_split_tokens(bundle, "validation", "sha_a", lambda: texts[:-1], CACHE_ROWS, None,
                                             show_progress=False)),
          "built row count must match expected_rows", "length mismatch not detected")

# ---- Section: Save and reload ----
with tempfile.TemporaryDirectory() as tmp:
    for name, bundle in bundles.items():
        out_dir = Path(tmp) / name
        bundle.save(out_dir)
        reloaded = tk.load_saved_tokenizer(out_dir)
        check(reloaded.encode(short) == bundle.encode(short) and reloaded.info()["special_ids"] == bundle.info()["special_ids"],
              f"{name}: saved tokenizer reloads and encodes identically", "save/load changed the tokenizer")
        check((out_dir / tk.BACKEND_INFO_NAME).is_file(), f"{name}: tokenizer_backend.json written")
    check((Path(tmp) / "mdeberta-v3-base" / tk.SP_MODEL_NAME).is_file(), "mdeberta: spm.model copied", "copy failed")

# ---- Section: Error handling ----
check(raises(lambda: tk.load_tokenizer(models_dir / "TookaBERT-Base", "unknown", 128)), "unknown backend is rejected")
check(raises(lambda: tk.load_tokenizer(models_dir / "TookaBERT-Base", "sentencepiece", 128)),
      "sentencepiece backend on a folder without spm.model is rejected")
check(raises(lambda: tk.load_tokenizer(models_dir / "mdeberta-v3-base", "sentencepiece", 2)),
      "max_length < 3 is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== TookaBERT-Base (hf_fast, max_length=128) ===
[PASS] special ids first/last/pad/unk = (2, 3, 0, 1)
[PASS] one row of ids per text
[PASS] all rows start/end with special ids and fit max_length
[PASS] flat arrays equal bundle.encode output
   stats: {'mean_length': 13.097, 'rows_at_max_length_pct': 0.16, 'unk_pct_of_tokens': 0.06223, 'rows_with_unk': 157, 'max_token_id': 47982}

=== roberta-fa-zwnj-base (hf_fast, max_length=128) ===
[PASS] special ids first/last/pad/unk = (0, 2, 1, 3)
[PASS] one row of ids per text
[PASS] all rows start/end with special ids and fit max_length
[PASS] flat arrays equal bundle.encode output
   stats: {'mean_length': 13.566, 'rows_at_max_length_pct': 0.16, 'unk_pct_of_tokens': 0.0, 'rows_with_unk': 0, 'max_token_id': 41947}

=== mdeberta-v3-base (sentencepiece, max_length=160) ===
[PASS] special ids first/last/pad/unk = (1, 2, 0, 3)
[PASS] one row of ids per text
[PASS] all rows start/end with special ids and fit max_length
[PASS] flat arrays equal bundl

[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[PASS] sentencepiece == HF fast on rows without unk (19,372/19,372)
[INFO] rows where HF fast has unk (excluded): 628; unk tokens from sentencepiece: 0
[PASS] chunked encoding equals one-shot encoding
[PASS] chunked stats equal one-shot stats
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] first call builds the cache
[PASS] second call reads the identical arrays from cache
[WARN] validation: cache key differs (new pack, tokenizer or max_length) -> rebuilding
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] a different source sha256 invalidates the cache
[WARN] validation: cache unreadable: cache has 5,000 rows, expected 5,001 -> rebuilding
[PASS] expected_rows mismatch is not served from cache
[WARN] validation: cache unreadable: token count differs from meta.json -> rebuilding
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] damaged cache is rebuilt, not trusted
[PASS] rebuilt cache is valid again
[timer] tokenize_TookaBERT-Base_validation: 0.1s
[PASS] cache_dir=

#  Test: Data Module, Batch Order, Padding and Spawn Loader

In [ ]:
# ---- Section: Imports ----
import numpy as np
import pandas as pd
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import data, pack, run_config
from digikala_review_transformer.training import tokenization as tk

# ---- Section: Parameters ----
SUBSET_ROWS = 20_000
MONITOR_ROWS = 5_000
BATCH = 64
EVAL_BATCH = 512
MIN_SPREAD_RATIO = 0.5
KNOWN_TRAIN_COUNTS = {"rec": [1_834_524, 266_638, 290_995], "band": [549_788, 500_933, 1_809_515]}
SEED = config.SPLIT_SEED

cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(cfg["data"]["pack_dir"])
model_dir = run_config.project_path(cfg["model"]["pretrained_path"])
train_path = pack_dir / "train.parquet"
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=data.DataError):
    try:
        function()
    except error_type:
        return True
    return False


def same_batch(a, b):
    return a.keys() == b.keys() and all(torch.equal(a[k], b[k]) for k in a)


# ---- Section: Pack and class weights on the full train labels ----
manifest = data.open_pack(pack_dir)
check(True, "open_pack verified sha256, rows and columns")
labels_full = pd.read_parquet(train_path, columns=["y_rec", "y_band"])
for head in data.HEADS:
    codes = data.to_codes(labels_full[f"y_{head}"], head)
    counts = np.bincount(codes[codes >= 0], minlength=3).tolist()
    check(counts == KNOWN_TRAIN_COUNTS[head], f"{head}: train class counts equal the handoff {counts}",
          f"expected {KNOWN_TRAIN_COUNTS[head]}")
    vector = data.class_weight_vector(codes, head)
    expected = np.array([sum(counts) / (3 * c) for c in counts], dtype=np.float32)
    check(np.allclose(vector, expected, rtol=1e-6), f"{head}: class weights = n/(k*n_c) -> {vector.round(4).tolist()}",
          "labels.class_weights differs from the plan formula")
del labels_full, codes

# ---- Section: Subsets of train for B and C ----
bundle = tk.load_tokenizer(model_dir, cfg["model"]["tokenizer_backend"], cfg["model"]["max_length"])
split_b = data.load_split("train", bundle, pack_dir, manifest, "B", subset_rows=SUBSET_ROWS, show_progress=False)
split_c = data.load_split("train", bundle, pack_dir, manifest, "C", subset_rows=SUBSET_ROWS, show_progress=False)
n = split_b.n_rows
check(n == SUBSET_ROWS and split_b.origin == "built", "subset has the requested rows and is built without cache")
check(np.array_equal(split_b.source_rows, split_c.source_rows) and split_b.arrays.equals(split_c.arrays),
      "B and C use identical rows and tokens", "subset selection depends on the variant")

pack_labels = pd.read_parquet(train_path, columns=["id"] + pack.LABEL_COLUMNS).iloc[split_b.source_rows]
check(np.array_equal(split_b.ids, pack_labels["id"].to_numpy()), "ids follow source_rows")
for head in data.HEADS:
    raw_codes = data.to_codes(pack_labels[f"y_{head}_raw"], head)
    check(np.array_equal(split_b.y_raw[head], raw_codes), f"{head}: raw labels match the pack")
    check(np.array_equal(split_b.y[head], split_c.y[head]), f"{head}: corrected labels do not depend on the variant")
    check(int((split_b.y[head] == data.IGNORE_INDEX).sum()) == int(pack_labels[f"y_{head}"].isna().sum()),
          f"{head}: missing labels become {data.IGNORE_INDEX}")

pack_weights = pd.read_parquet(train_path, columns=data.WEIGHT_COLUMNS).iloc[split_b.source_rows]
for head in data.HEADS:
    check(bool((split_b.w[head] == 1).all()), f"{head}: variant B weights are all 1")
    original = pack_weights[f"w_{head}"].to_numpy(dtype="float32", na_value=np.nan)
    finite = ~np.isnan(original)
    check(np.isfinite(split_c.w[head]).all(), f"{head}: variant C has no NaN after replacement")
    check(np.array_equal(split_c.w[head][finite], original[finite]) and bool((split_c.w[head][~finite] == 1).all()),
          f"{head}: variant C keeps Step 2 weights and uses 1 for NaN ({int((~finite).sum()):,} NaN rows)",
          "weights were altered")
    check(0.1 - 1e-6 <= split_c.w[head][finite].min() and split_c.w[head].max() <= 1.0 + 1e-6,
          f"{head}: variant C weights lie in [0.1, 1]")

# ---- Section: Batch order ----
sampler_b = data.LengthBucketedBatches(split_b.lengths, BATCH, SEED)
sampler_c = data.LengthBucketedBatches(split_c.lengths, BATCH, SEED)
epoch0 = sampler_b.batches_for_epoch(0)
check(np.array_equal(np.sort(np.concatenate(epoch0)), np.arange(n)), "every row appears exactly once per epoch")
check(len(epoch0) == len(sampler_b) == -(-n // BATCH), "number of batches = ceil(rows / batch_size)")
check(all(len(b) == BATCH for b in epoch0[:-1]) or sum(len(b) != BATCH for b in epoch0) == 1,
      "at most one batch is smaller than batch_size")
check(data.batch_fingerprint(epoch0) == data.batch_fingerprint(sampler_b.batches_for_epoch(0)),
      "same seed and epoch give the same order")
check(data.batch_fingerprint(epoch0) != data.batch_fingerprint(sampler_b.batches_for_epoch(1)),
      "a different epoch gives a different order", "epoch is ignored")
check(data.batch_fingerprint(epoch0) == data.batch_fingerprint(sampler_c.batches_for_epoch(0)),
      "B and C see identical batches", "order depends on the variant")
check(data.batch_fingerprint(epoch0) != data.batch_fingerprint(
    data.LengthBucketedBatches(split_b.lengths, BATCH, SEED + 1).batches_for_epoch(0)),
      "a different seed gives a different order")

spread = np.mean([split_b.lengths[b].max() - split_b.lengths[b].min() for b in epoch0])
random_order = np.random.default_rng(0).permutation(n)
random_spread = np.mean([
    np.ptp(split_b.lengths[random_order[i:i + BATCH]])
    for i in range(0, n, BATCH)
])
check(spread < MIN_SPREAD_RATIO * random_spread,
      f"bucketing shrinks length spread inside batches ({spread:.1f} vs {random_spread:.1f} random)",
      "length sorting is not working")

# ---- Section: Padding and targets ----
dataset = split_c.dataset()
indices = epoch0[0]
batch = dataset[indices]
lengths = split_c.lengths[indices]
check(batch["input_ids"].shape == (len(indices), int(lengths.max())) and batch["input_ids"].dtype == torch.int64,
      "batch shape is (rows, longest row) with int64 ids")
rows_ok = pads_ok = mask_ok = True
for i, row in enumerate(indices):
    length = int(lengths[i])
    rows_ok &= batch["input_ids"][i, :length].tolist() == split_c.arrays.row(row).tolist()
    pads_ok &= bool((batch["input_ids"][i, length:] == split_c.pad_id).all())
    mask_ok &= int(batch["attention_mask"][i].sum()) == length and bool(batch["attention_mask"][i, :length].all())
check(rows_ok, "padded rows keep the exact token ids")
check(pads_ok, f"padding uses pad id {split_c.pad_id}")
check(mask_ok, "attention mask is 1 on tokens and 0 on padding")
check(torch.equal(batch["index"], torch.from_numpy(indices)), "batch carries the row indices")
check(all(torch.equal(batch[f"y_{h}"], torch.from_numpy(split_c.y[h][indices])) for h in data.HEADS),
      "targets are aligned with rows")
check(all(torch.equal(batch[f"w_{h}"], torch.from_numpy(split_c.w[h][indices])) for h in data.HEADS),
      "weights are aligned with rows")

# Rows without a band label (future semi-supervised use) must collate without any rewrite.
no_band = data.BatchedTokenDataset(split_c.arrays, {"rec": split_c.y["rec"], "band": np.full(n, data.IGNORE_INDEX)},
                                   split_c.w, split_c.pad_id)
check(bool((no_band[indices]["y_band"] == data.IGNORE_INDEX).all()), "rows without band label collate fine")

# ---- Section: Spawn loader with workers (also checks set_epoch with persistent workers) ----
sampler = data.LengthBucketedBatches(split_c.lengths, BATCH, SEED)
loader = data.make_loader(dataset, sampler, num_workers=2, persistent=True)
sampler.set_epoch(0)
got0 = list(loader)
check(len(got0) == len(sampler), "loader yields every batch")
check(all(same_batch(got0[i], dataset[sampler.batches_for_epoch(0)[i]]) for i in (0, 1, len(got0) - 1)),
      "spawn workers return the same batches as direct indexing", "pickling or collate differs in workers")
sampler.set_epoch(1)
got1 = list(loader)
check(same_batch(got1[0], dataset[sampler.batches_for_epoch(1)[0]]) and not same_batch(got0[0], got1[0]),
      "set_epoch changes the order of a persistent loader", "workers cache the old order")
del loader, got0, got1

# ---- Section: Evaluation batches ----
monitor = data.subset_row_indices(n, MONITOR_ROWS, SEED, "monitor")
smoke = data.subset_row_indices(n, MONITOR_ROWS, SEED, "smoke")
check(len(monitor) == MONITOR_ROWS and (np.diff(monitor) > 0).all(), "monitor subset: right size, sorted, unique")
check(np.array_equal(monitor, data.subset_row_indices(n, MONITOR_ROWS, SEED, "monitor")), "monitor subset is reproducible")
check(not np.array_equal(monitor, smoke), "monitor and smoke subsets use different streams")
eval_batches = data.sorted_eval_batches(split_c.lengths, monitor, EVAL_BATCH)
flat = np.concatenate(eval_batches)
check(np.array_equal(np.sort(flat), monitor) and max(len(b) for b in eval_batches) <= EVAL_BATCH,
      "eval batches cover the subset once and respect the batch size")
check(bool((np.diff(split_c.lengths[flat]) >= 0).all()), "eval rows are sorted by length")
eval_loader = data.make_loader(dataset, eval_batches, num_workers=0)
seen = np.concatenate([b["index"].numpy() for b in eval_loader])
check(np.array_equal(seen, flat), "eval loader returns rows in the planned order")

# ---- Section: Full validation split (real size, variant C must still give weight 1) ----
validation = data.load_split("validation", bundle, pack_dir, manifest, "C", show_progress=False)
entry = manifest["files"]["validation.parquet"]
check(validation.n_rows == entry["rows"] and pd.Series(validation.ids).is_unique,
      f"validation: {validation.n_rows:,} rows with unique ids")
check(all(int((validation.y[h] == data.IGNORE_INDEX).sum()) == entry["label_missing"][f"y_{h}"] for h in data.HEADS),
      "validation: missing label counts equal the manifest")
check(all(bool((validation.w[h] == 1).all()) for h in data.HEADS), "validation: weights are 1 (no weight columns)")
print("   validation token stats:", {k: validation.stats[k] for k in ("mean_length", "rows_at_max_length_pct", "unk_pct_of_tokens")})

# ---- Section: Error handling ----
check(raises(lambda: data.class_weight_vector(np.array([0, 0, 1, 0, data.IGNORE_INDEX]), "rec")),
      "class weights with a missing class are rejected")
check(raises(lambda: data.load_split("dev", bundle, pack_dir, manifest, "B")), "unknown split is rejected")
check(raises(lambda: data.load_split("validation", bundle, pack_dir, manifest, "D")), "unknown variant is rejected")
check(raises(lambda: data.load_split("validation", bundle, pack_dir, manifest, "B", subset_rows=10, cache_dir=tk.CACHE_ROOT)),
      "subset together with disk cache is rejected")
check(raises(lambda: data.subset_row_indices(10, 5, SEED, "other")), "unknown subset purpose is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")

[PASS] open_pack verified sha256, rows and columns
[PASS] rec: train class counts equal the handoff [1834524, 266638, 290995]
[PASS] rec: class weights = n/(k*n_c) -> [0.43470001220703125, 2.990499973297119, 2.7402000427246094]
[PASS] band: train class counts equal the handoff [549788, 500933, 1809515]
[PASS] band: class weights = n/(k*n_c) -> [1.7340999841690063, 1.9033000469207764, 0.5268999934196472]
[timer] tokenize_TookaBERT-Base_train: 0.2s
[timer] tokenize_TookaBERT-Base_train: 0.2s
[PASS] subset has the requested rows and is built without cache
[PASS] B and C use identical rows and tokens
[PASS] ids follow source_rows
[PASS] rec: raw labels match the pack
[PASS] rec: corrected labels do not depend on the variant
[PASS] rec: missing labels become -100
[PASS] band: raw labels match the pack
[PASS] band: corrected labels do not depend on the variant
[PASS] band: missing labels become -100
[PASS] rec: variant B weights are all 1
[PASS] rec: variant C has no NaN after replacement
[P

# Test: Model, Heads, Parameter Groups and Padding Invariance

In [ ]:
# ---- Section: Imports ----
import copy
import gc
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import data, run_config
from digikala_review_transformer.training import model as mdl
from digikala_review_transformer.training import tokenization as tk

# ---- Section: Parameters ----
MODELS = {
    # name: (backend, max_length)
    "TookaBERT-Base": ("hf_fast", 128),
    "roberta-fa-zwnj-base": ("hf_fast", 128),
    "mdeberta-v3-base": ("sentencepiece", 160),
}
SAMPLE_ROWS = 2_000
FORWARD_ROWS = 4
PAD_ATOL = 1e-3
STD_TOLERANCE = 0.2
LR_ENCODER, LR_HEADS, WEIGHT_DECAY = 2.0e-5, 1.0e-4, 0.01

base_cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")
pack_dir = run_config.project_path(base_cfg["data"]["pack_dir"])
models_dir = config.ROOT / "models" / "pretrained"
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=mdl.ModelError):
    try:
        function()
    except error_type:
        return True
    return False


def cfg_for(name, seed=None):
    cfg = copy.deepcopy(base_cfg)
    cfg["model"]["pretrained_path"] = str(models_dir / name)
    if seed is not None:
        cfg["seed"] = seed
    return cfg


def heads_equal(a, b):
    return all(torch.equal(x, y) for x, y in zip(a.heads.state_dict().values(), b.heads.state_dict().values()))


# ---- Section: Sample texts (validation, fixed seed) ----
bodies = pd.read_parquet(pack_dir / "validation.parquet", columns=["body"])["body"]
sample = bodies.sample(SAMPLE_ROWS, random_state=config.SPLIT_SEED).tolist()
del bodies

# ---- Section: Per-model checks ----
for name, (backend, max_length) in MODELS.items():
    print(f"\n=== {name} ===")
    cfg = cfg_for(name)
    bundle = tk.load_tokenizer(models_dir / name, backend, max_length)
    model, load_info = mdl.build_model(cfg)
    print("   load_info:", load_info, "| encoder:", type(model.encoder).__name__)

    hidden = model.encoder.config.hidden_size
    check(all(p.dtype == torch.float32 for p in model.parameters()), "all parameters are float32",
          "dtype was not forced to float32")
    check(getattr(model.encoder, "pooler", None) is None, "pooler is removed")
    check(model.dropout.p == cfg["model"]["head_dropout"], f"head dropout = {model.dropout.p}")

    counts = mdl.count_parameters(model)
    expected_heads = sum(hidden * len(mdl.labels.CLASSES[h]) + len(mdl.labels.CLASSES[h]) for h in mdl.HEADS)
    check(counts["heads"] == expected_heads and counts["total"] == counts["encoder"] + counts["heads"]
          and counts["trainable"] == counts["total"],
          f"parameter counts are consistent {counts}", "counts do not add up")

    stds = [layer.weight.std().item() for layer in model.heads.values()]
    check(all(abs(s - mdl.HEAD_INIT_STD) < STD_TOLERANCE * mdl.HEAD_INIT_STD for s in stds)
          and all(bool((layer.bias == 0).all()) for layer in model.heads.values()),
          f"head init: std {[round(s, 4) for s in stds]}, zero bias", "init differs from the plan")

    # Token ids vs embedding rows.
    arrays = tk.encode_texts(bundle, sample)
    stats = tk.token_stats(arrays, bundle)
    try:
        mdl.check_token_ids_fit(model, stats["max_token_id"], bundle.pad_id)
        check(True, f"token ids fit the embedding (max id {stats['max_token_id']})")
    except mdl.ModelError as error:
        check(False, "token ids fit the embedding", str(error))
    check(raises(lambda: mdl.check_token_ids_fit(model, 10 ** 9, bundle.pad_id)), "oversized token id is rejected")  # noqa: B023
    check(raises(lambda: mdl.check_token_ids_fit(model, 1, bundle.pad_id + 1))  # noqa: B023
          or getattr(model.encoder.config, "pad_token_id", None) is None,
          "wrong pad id is rejected (when the config has a pad id)")

    # Forward: padding invariance proves attention mask and position 0 are used correctly.
    zeros = {h: np.full(SAMPLE_ROWS, data.IGNORE_INDEX) for h in data.HEADS}
    ones = {h: np.ones(SAMPLE_ROWS, dtype=np.float32) for h in data.HEADS}
    dataset = data.BatchedTokenDataset(arrays, zeros, ones, bundle.pad_id)
    order = np.argsort(arrays.lengths)
    rows = np.array([order[0], order[SAMPLE_ROWS // 3], order[2 * SAMPLE_ROWS // 3], order[-1]])[:FORWARD_ROWS]
    model.eval()
    with torch.no_grad():
        batch = dataset[rows]
        together = model(batch["input_ids"], batch["attention_mask"])
        alone = [model(*(lambda b: (b["input_ids"], b["attention_mask"]))(dataset[rows[i:i + 1]])) for i in range(len(rows))]  # noqa: PLC3002
    check(all(together[h].shape == (len(rows), 3) and together[h].dtype == torch.float32
              and bool(torch.isfinite(together[h]).all()) for h in mdl.HEADS),
          "forward gives finite float32 logits of shape (rows, 3) per head")
    gap = max((together[h][i] - alone[i][h][0]).abs().max().item() for h in mdl.HEADS for i in range(len(rows)))
    check(gap < PAD_ATOL, f"logits do not depend on padding (max gap {gap:.2e}; lengths {arrays.lengths[rows].tolist()})",
          "attention mask or feature position is wrong")

    # Backward: heads get gradients; report encoder tensors without gradient.
    model.train()
    model.zero_grad(set_to_none=True)
    logits = model(batch["input_ids"], batch["attention_mask"])
    (logits["rec"].sum() + logits["band"].sum()).backward()
    check(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.heads.parameters()),
          "heads receive finite gradients")
    no_grad = [n for n, p in model.encoder.named_parameters() if p.grad is None]
    check(len(no_grad) / counts["encoder"] < 1e-3 or len(no_grad) < 5,
          f"almost every encoder tensor receives a gradient ({len(no_grad)} without: {no_grad[:3]})",
          "large part of the encoder is disconnected from the loss")

    # Parameter groups and one optimizer step.
    groups = mdl.parameter_groups(model, LR_ENCODER, LR_HEADS, WEIGHT_DECAY)
    flat = [p for g in groups for p in g["params"]]
    check(len(flat) == len({id(p) for p in flat}) == len(list(model.parameters())),
          f"every parameter is in exactly one group ({[g['name'] for g in groups]})")
    check(all((g["weight_decay"] > 0) == all(p.ndim >= 2 for p in g["params"]) for g in groups)
          and all(p.ndim < 2 for g in groups if g["weight_decay"] == 0 for p in g["params"]),
          "weight decay only on tensors with ndim >= 2")
    check(all(g["lr"] == (LR_ENCODER if g["name"].startswith("encoder") else LR_HEADS) for g in groups),
          "encoder and head learning rates are applied")
    before = model.heads["rec"].weight.detach().clone()
    torch.optim.AdamW(groups).step()
    check(not torch.equal(before, model.heads["rec"].weight.detach()), "an AdamW step changes the head weights")

    # Saved config.
    with tempfile.TemporaryDirectory() as tmp:
        target = Path(tmp) / "encoder_config.json"
        mdl.save_encoder_config(model, target)
        check(target.is_file() and f'"hidden_size": {hidden}' in target.read_text(encoding="utf-8"),
              "encoder_config.json is written")
    summary = mdl.model_summary(model, load_info)
    check(summary["parameters"] == counts and summary["dtype"] == "torch.float32", "model_summary is consistent")

    del model, bundle, arrays, dataset, batch, together, alone, logits, groups, flat
    gc.collect()

# ---- Section: Same seed -> same heads (variants B and C) ----
print("\n=== seeds ===")
first, _ = mdl.build_model(cfg_for("TookaBERT-Base"))
second, _ = mdl.build_model(cfg_for("TookaBERT-Base"))
third, _ = mdl.build_model(cfg_for("TookaBERT-Base", seed=config.SPLIT_SEED + 1))
check(heads_equal(first, second), "same seed gives identical head initialisation")
check(not heads_equal(first, third), "a different seed gives different heads")
check(all(torch.equal(a, b) for a, b in zip(first.encoder.state_dict().values(), second.encoder.state_dict().values())),
      "encoder weights are identical across builds")
del first, second, third
gc.collect()

# ---- Section: Error handling ----
bad_cfg = cfg_for("TookaBERT-Base")
bad_cfg["model"]["pretrained_path"] = str(models_dir / "does_not_exist")
check(raises(lambda: mdl.build_model(bad_cfg)), "missing model folder is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")


=== TookaBERT-Base ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8076.61it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

   load_info: {'unexpected_keys': 5, 'pooler_keys_missing': 2} | encoder: BertModel
[PASS] all parameters are float32
[PASS] pooler is removed
[PASS] head dropout = 0.1
[PASS] parameter counts are consistent {'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[PASS] head init: std [0.0197, 0.0199], zero bias
[PASS] token ids fit the embedding (max id 47187)
[PASS] oversized token id is rejected
[PASS] wrong pad id is rejected (when the config has a pad id)
[PASS] forward gives finite float32 logits of shape (rows, 3) per head
[PASS] logits do not depend on padding (max gap 1.05e-06; lengths [3, 6, 12, 128])
[PASS] heads receive finite gradients
[PASS] almost every encoder tensor receives a gradient (0 without: [])
[PASS] every parameter is in exactly one group (['encoder_decay', 'encoder_no_decay', 'heads_decay', 'heads_no_decay'])
[PASS] weight decay only on tensors with ndim >= 2
[PASS] encoder and head learning rates are applied
[PASS] an AdamW step cha

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 50475.13it/s]
[transformers] RobertaModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/roberta-fa-zwnj-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   load_info: {'unexpected_keys': 7, 'pooler_keys_missing': 2} | encoder: RobertaModel
[PASS] all parameters are float32
[PASS] pooler is removed
[PASS] head dropout = 0.1
[PASS] parameter counts are consistent {'total': 117712134, 'trainable': 117712134, 'encoder': 117707520, 'heads': 4614}
[PASS] head init: std [0.0197, 0.0199], zero bias
[PASS] token ids fit the embedding (max id 41944)
[PASS] oversized token id is rejected
[PASS] wrong pad id is rejected (when the config has a pad id)
[PASS] forward gives finite float32 logits of shape (rows, 3) per head
[PASS] logits do not depend on padding (max gap 5.36e-07; lengths [3, 7, 12, 128])
[PASS] heads receive finite gradients
[PASS] almost every encoder tensor receives a gradient (0 without: [])
[PASS] every parameter is in exactly one group (['encoder_decay', 'encoder_no_decay', 'heads_decay', 'heads_no_decay'])
[PASS] weight decay only on tensors with ndim >= 2
[PASS] encoder and head learning rates are applied
[PASS] an AdamW step 

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 865.30it/s]
[transformers] DebertaV2Model LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_prediction

   load_info: {'unexpected_keys': 12, 'pooler_keys_missing': 0} | encoder: DebertaV2Model
[PASS] all parameters are float32
[PASS] pooler is removed
[PASS] head dropout = 0.1
[PASS] parameter counts are consistent {'total': 278223366, 'trainable': 278223366, 'encoder': 278218752, 'heads': 4614}
[PASS] head init: std [0.0197, 0.0199], zero bias
[PASS] token ids fit the embedding (max id 242648)
[PASS] oversized token id is rejected
[PASS] wrong pad id is rejected (when the config has a pad id)
[PASS] forward gives finite float32 logits of shape (rows, 3) per head
[PASS] logits do not depend on padding (max gap 2.68e-06; lengths [3, 10, 19, 160])
[PASS] heads receive finite gradients
[PASS] almost every encoder tensor receives a gradient (0 without: [])
[PASS] every parameter is in exactly one group (['encoder_decay', 'encoder_no_decay', 'heads_decay', 'heads_no_decay'])
[PASS] weight decay only on tensors with ndim >= 2
[PASS] encoder and head learning rates are applied
[PASS] an AdamW 

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16316.70it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[PASS] same seed gives identical head initialisation
[PASS] a different seed gives different heads
[PASS] encoder weights are identical across builds
[PASS] missing model folder is rejected

ALL PASSED


#  Inspect: mDeBERTa Embedding Keys in the Checkpoint

In [7]:
# ---- Section: Imports ----
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import model as mdl

# ---- Section: Parameters ----
path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
KEY_FILTER = "word_embeddings"

# ---- Section: Embedding-related keys in the raw checkpoint ----
state = torch.load(path / "pytorch_model.bin", map_location="cpu", weights_only=True)
keys = [k for k in state if KEY_FILTER in k]
print("keys containing", KEY_FILTER, ":")
for k in keys:
    t = state[k]
    print(f"  {k} | shape={tuple(t.shape)} | dtype={t.dtype} | mean={t.float().mean():.5f} | std={t.float().std():.5f}")

weight_key = next((k for k in keys if k.endswith("word_embeddings.weight")), None)
extra_key = next((k for k in keys if k.endswith("word_embeddings._weight")), None)
print("\nweight key:", weight_key, "| extra key:", extra_key)

# ---- Section: Compare the two tensors ----
if weight_key and extra_key:
    a, b = state[weight_key].float(), state[extra_key].float()
    if a.shape == b.shape:
        diff = (a - b).abs()
        print(f"max |weight - _weight| = {diff.max():.6f} | mean = {diff.mean():.6f} | identical: {torch.equal(a, b)}")
        print(f"norm(weight) = {a.norm():.2f} | norm(_weight) = {b.norm():.2f}")
    else:
        print("shapes differ:", tuple(a.shape), tuple(b.shape))

# ---- Section: What the loaded model actually uses ----
encoder, info = mdl.load_encoder(path)
loaded = encoder.get_input_embeddings().weight.detach().float()
if weight_key:
    print("\nmodel embedding == checkpoint .weight:", torch.equal(loaded, state[weight_key].float()))
if extra_key and state[extra_key].shape == loaded.shape:
    print("model embedding == checkpoint ._weight:", torch.equal(loaded, state[extra_key].float()))
print("model embedding shape:", tuple(loaded.shape), "| load_info:", info)

keys containing word_embeddings :
  deberta.embeddings.word_embeddings.weight | shape=(251000, 768) | dtype=torch.float16 | mean=-0.01234 | std=0.14396
  deberta.embeddings.word_embeddings._weight | shape=(251000, 768) | dtype=torch.float32 | mean=0.01802 | std=0.15646

weight key: deberta.embeddings.word_embeddings.weight | extra key: deberta.embeddings.word_embeddings._weight
max |weight - _weight| = 1.315626 | mean = 0.054561 | identical: False
norm(weight) = 1957.61 | norm(_weight) = 2129.99


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1042.28it/s]
[transformers] DebertaV2Model LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.bias           | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.L


model embedding == checkpoint .weight: True
model embedding == checkpoint ._weight: False
model embedding shape: (251000, 768) | load_info: {'unexpected_keys': 12, 'pooler_keys_missing': 0}


# Inspect: mDeBERTa Embedding vs Generator Embedding

In [8]:
# ---- Section: Imports ----
from collections import Counter

import torch

from digikala_review_transformer import config

# ---- Section: Parameters ----
path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
EMB = "deberta.embeddings.word_embeddings"
ROW_QUANTILES = torch.tensor([0.5, 0.9, 0.99, 1.0])
SAMPLE_PREVIEW = 3


# ---- Section: Helpers ----
def describe(tensor):
    t = tensor.float()
    return f"shape={tuple(tensor.shape)} dtype={tensor.dtype} mean={t.mean():.5f} std={t.std():.5f} norm={t.norm():.2f}"


def compare(name_a, a, name_b, b):
    """Print numeric closeness of two tensors of the same shape."""
    if a.shape != b.shape:
        print(f"{name_a} vs {name_b}: shapes differ {tuple(a.shape)} {tuple(b.shape)}")
        return
    a, b = a.float(), b.float()
    diff = (a - b).abs()
    cosine = torch.nn.functional.cosine_similarity(a, b, dim=1)
    row_gap = (a - b).norm(dim=1) / a.norm(dim=1).clamp_min(1e-8)
    print(f"{name_a} vs {name_b}: identical={torch.equal(a, b)} | max|diff|={diff.max():.6f} | mean|diff|={diff.mean():.6f}")
    print(f"   row cosine mean={cosine.mean():.4f} min={cosine.min():.4f} | "
          f"relative row gap quantiles (50/90/99/100%)={[round(v, 4) for v in torch.quantile(row_gap, ROW_QUANTILES).tolist()]}")


# ---- Section: Discriminator checkpoint ----
disc = torch.load(path / "pytorch_model.bin", map_location="cpu", weights_only=True)
print("discriminator tensors by dtype:", dict(Counter(str(t.dtype) for t in disc.values())))
weight, extra = disc[f"{EMB}.weight"], disc[f"{EMB}._weight"]
print("weight :", describe(weight))
print("_weight:", describe(extra))
print("weight has non-finite values:", not bool(torch.isfinite(weight.float()).all()),
      "| _weight has non-finite values:", not bool(torch.isfinite(extra).all()))

# ---- Section: Generator checkpoint ----
gen = torch.load(path / "pytorch_model.generator.bin", map_location="cpu", weights_only=True)
print("\ngenerator tensors by dtype:", dict(Counter(str(t.dtype) for t in gen.values())))
gen_keys = [k for k in gen if "word_embeddings" in k]
print("generator embedding keys:", gen_keys)
for key in gen_keys:
    print(" ", key, describe(gen[key]))

# ---- Section: Comparisons ----
print()
compare("discriminator .weight", weight, "discriminator ._weight", extra)
for key in gen_keys:
    compare("discriminator .weight", weight, f"generator {key.split('.')[-1]}", gen[key])
    compare("discriminator ._weight", extra, f"generator {key.split('.')[-1]}", gen[key])

discriminator tensors by dtype: {'torch.float16': 210, 'torch.float32': 2}
weight : shape=(251000, 768) dtype=torch.float16 mean=-0.01234 std=0.14396 norm=1957.61
_weight: shape=(251000, 768) dtype=torch.float32 mean=0.01802 std=0.15646 norm=2129.99
weight has non-finite values: False | _weight has non-finite values: False

generator tensors by dtype: {'torch.float32': 108}
generator embedding keys: ['deberta.embeddings.word_embeddings.weight']
  deberta.embeddings.word_embeddings.weight shape=(251000, 768) dtype=torch.float32 mean=-0.03036 std=0.06194 norm=933.96

discriminator .weight vs discriminator ._weight: identical=False | max|diff|=1.315626 | mean|diff|=0.054561
   row cosine mean=0.8913 min=0.3470 | relative row gap quantiles (50/90/99/100%)=[0.4706, 0.5431, 2.219, 2.3229]
discriminator .weight vs generator weight: identical=False | max|diff|=1.957709 | mean|diff|=0.124575
   row cosine mean=0.0402 min=-0.1387 | relative row gap quantiles (50/90/99/100%)=[1.0881, 1.1209, 2.40

# Test: Losses, Masking, Class Weights and Accumulator

In [9]:
# ---- Section: Imports ----
import numpy as np
import torch
import torch.nn.functional as F

from digikala_review_transformer.training import data, losses

# ---- Section: Parameters ----
N_ROWS = 12
SMOOTHING = 0.1
TOLERANCE = 1e-5
CLASS_WEIGHTS = {"rec": [0.6, 1.7, 2.1], "band": [1.8, 1.3, 0.5]}
IGN = data.IGNORE_INDEX
HEADS = data.HEADS
results = []


def check(ok, name, hint=""):
    results.append(bool(ok))
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else f" -> {hint}"))


def raises(function, error_type=losses.LossError):
    try:
        function()
    except error_type:
        return True
    return False


def make_batch(n=N_ROWS, rec_missing=(1, 5), seed=0):
    g = torch.Generator().manual_seed(seed)
    logits = {h: torch.randn(n, 3, generator=g).requires_grad_() for h in HEADS}
    y_rec = torch.randint(0, 3, (n,), generator=g)
    y_rec[list(rec_missing)] = IGN
    batch = {
        "y_rec": y_rec,
        "y_band": torch.randint(0, 3, (n,), generator=g),
        "w_rec": 0.1 + 0.9 * torch.rand(n, generator=g),
        "w_band": 0.1 + 0.9 * torch.rand(n, generator=g),
    }
    return logits, batch


def reference_head_loss(logits, target, w, class_weight, eps):
    """Slow row-by-row version of the plan formula (double precision)."""
    logp = F.log_softmax(logits.detach().double(), dim=1)
    numerator = denominator = 0.0
    for i in range(len(target)):
        t = int(target[i])
        if t == IGN:
            continue
        ce = (1 - eps) * (-logp[i, t]) + eps * (-logp[i].mean())
        e = float(w[i]) * class_weight[t]
        numerator += e * float(ce)
        denominator += e
    return numerator / denominator if denominator > 0 else 0.0


def make_loss(smoothing=SMOOTHING, lam=1.0, weights=CLASS_WEIGHTS):
    return losses.TwoHeadLoss(weights, smoothing, lam, "cpu")


# ---- Section: Formula against a hand-written reference ----
logits, batch = make_batch()
output = make_loss(lam=2.0)(logits, batch)
for head in HEADS:
    expected = reference_head_loss(logits[head], batch[f"y_{head}"], batch[f"w_{head}"], CLASS_WEIGHTS[head], SMOOTHING)
    check(abs(float(output.head_loss[head]) - expected) < TOLERANCE,
          f"{head}: loss equals the row-by-row reference ({float(output.head_loss[head]):.6f})",
          "formula differs from the plan")
check(abs(float(output.total) - (float(output.head_loss["rec"]) + 2.0 * float(output.head_loss["band"]))) < TOLERANCE,
      "total = loss_rec + loss_weight_band * loss_band", "head combination is wrong")
check(output.total.dtype == torch.float32 and output.total.requires_grad, "total is float32 and keeps the graph")
check(not any(v.requires_grad for v in output.head_loss.values()), "per-head values are detached")

# ---- Section: Special cases of the weighting ----
ones = {h: [1.0, 1.0, 1.0] for h in HEADS}
flat = make_loss(weights=ones)
batch_ones = {**batch, "w_rec": torch.ones(N_ROWS), "w_band": torch.ones(N_ROWS)}
out_flat = flat(logits, batch_ones)
valid = batch["y_rec"] != IGN
mean_ce = F.cross_entropy(logits["rec"][valid], batch["y_rec"][valid], label_smoothing=SMOOTHING)
check(abs(float(out_flat.head_loss["rec"]) - float(mean_ce)) < TOLERANCE,
      "w=1 and class weights 1 give the plain smoothed mean CE over valid rows", "normalisation is wrong")

scaled = make_loss()(logits, {**batch, "w_rec": batch["w_rec"] * 3.7, "w_band": batch["w_band"] * 3.7})
check(all(abs(float(scaled.head_loss[h]) - float(output.head_loss[h])) < TOLERANCE for h in HEADS),
      "multiplying all sample weights by a constant does not change the loss", "denominator is not normalising")

# ---- Section: Masking ----
grad_rec = torch.autograd.grad(make_loss()(logits, batch).total, logits["rec"])[0]
masked_rows = (batch["y_rec"] == IGN).nonzero().flatten()
check(bool((grad_rec[masked_rows] == 0).all()) and bool((grad_rec[~(batch["y_rec"] == IGN)] != 0).any()),
      f"rows without rec label get zero gradient ({len(masked_rows)} masked rows)", "masked rows leak into the loss")
band_same = torch.allclose(
    make_loss()(logits, {**batch, "y_rec": torch.full((N_ROWS,), IGN)}).head_loss["band"], output.head_loss["band"])
check(band_same, "band loss is unchanged when rec is fully masked", "heads are not independent")

logits_empty, batch_empty = make_batch()
batch_empty["y_rec"] = torch.full((N_ROWS,), IGN)
batch_empty["y_band"] = torch.full((N_ROWS,), IGN)
out_empty = make_loss()(logits_empty, batch_empty)
out_empty.total.backward()
check(float(out_empty.total) == 0.0 and all(bool(torch.isfinite(l.grad).all()) and bool((l.grad == 0).all())
                                            for l in logits_empty.values()),
      "batch with no valid label gives loss 0 and zero finite gradients", "empty batch produces NaN or garbage")
check(all(int(out_empty.n_valid[h]) == 0 and float(out_empty.plain_ce_sum[h]) == 0.0 for h in HEADS),
      "empty batch: plain CE sum 0 and count 0")

# ---- Section: Plain cross entropy ----
for head in HEADS:
    target = batch[f"y_{head}"]
    ok_rows = target != IGN
    expected = float(F.cross_entropy(logits[head][ok_rows], target[ok_rows]))
    got = float(output.plain_ce_sum[head]) / int(output.n_valid[head])
    check(abs(got - expected) < TOLERANCE, f"{head}: plain CE ignores weights and smoothing ({got:.5f})")

# ---- Section: Numeric types ----
bf16_out = make_loss()({h: l.detach().bfloat16() for h, l in logits.items()}, batch)
check(bf16_out.total.dtype == torch.float32 and bool(torch.isfinite(bf16_out.total)),
      "bf16 logits are handled in float32")

# ---- Section: Accumulator ----
acc = losses.LossAccumulator("cpu")
loss_fn = make_loss()
parts = []
for seed in (1, 2):
    l, b = make_batch(seed=seed)
    out = loss_fn(l, b)
    acc.add(out)
    parts.append((l, b, out))
summary = acc.summary()
check(abs(summary["objective"] - np.mean([float(p[2].total) for p in parts])) < TOLERANCE and summary["steps"] == 2,
      "accumulator objective = mean of step losses")
pooled = {h: float(sum(p[2].plain_ce_sum[h] for p in parts)) / int(sum(p[2].n_valid[h] for p in parts)) for h in HEADS}
check(all(abs(summary[f"plain_ce_{h}"] - pooled[h]) < TOLERANCE for h in HEADS),
      "accumulator plain CE is the pooled mean over valid rows (not a mean of batch means)")
acc_eval = losses.LossAccumulator("cpu")
for l, b, _ in parts:
    acc_eval.add_logits(l, b)
eval_summary = acc_eval.summary()
check(all(abs(eval_summary[f"plain_ce_{h}"] - pooled[h]) < TOLERANCE for h in HEADS) and np.isnan(eval_summary["objective"]),
      "add_logits gives the same plain CE and no objective")
acc.reset()
fresh = acc.summary()
check(fresh["steps"] == 0 and all(np.isnan(fresh[k]) for k in ("objective", "plain_ce_rec", "plain_ce_band")),
      "reset clears the sums (empty summary is NaN, not 0)")

# ---- Section: build_loss from config and labels ----
cfg = {"training": {"use_class_weights": True, "label_smoothing": SMOOTHING, "loss_weight_band": 1.0}}
y_toy = {"rec": np.array([0, 0, 0, 0, 1, 2, IGN]), "band": np.array([0, 1, 2, 2, 2, 2, 1])}
built = losses.build_loss(cfg, y_toy, "cpu")
check(np.allclose(built.class_weights["rec"].numpy(), [6 / (3 * 4), 6 / (3 * 1), 6 / (3 * 1)]),
      "build_loss: class weights come from the labels of each head (masked rows excluded)", "wrong counts")
cfg_off = {"training": {**cfg["training"], "use_class_weights": False}}
check(bool((losses.build_loss(cfg_off, y_toy, "cpu").class_weights["band"] == 1).all()),
      "use_class_weights=false gives all-ones class weights")

# ---- Section: Error handling ----
check(raises(lambda: make_loss(lam=0.0)), "loss_weight_band = 0 is rejected")
check(raises(lambda: make_loss(smoothing=1.0)), "label_smoothing = 1 is rejected")
check(raises(lambda: make_loss(weights={"rec": [1, 1, 1]})), "missing head weights are rejected")
check(raises(lambda: make_loss(weights={"rec": [1, 1], "band": [1, 1, 1]})), "wrong weight vector length is rejected")
check(raises(lambda: make_loss(weights={"rec": [1, 0, 1], "band": [1, 1, 1]})), "zero class weight is rejected")
check(raises(lambda: make_loss(weights={"rec": [1, float("nan"), 1], "band": [1, 1, 1]})), "NaN class weight is rejected")

print("\nALL PASSED" if all(results) else "\nFAILED")

[PASS] rec: loss equals the row-by-row reference (1.074777)
[PASS] band: loss equals the row-by-row reference (1.487795)
[PASS] total = loss_rec + loss_weight_band * loss_band
[PASS] total is float32 and keeps the graph
[PASS] per-head values are detached
[PASS] w=1 and class weights 1 give the plain smoothed mean CE over valid rows
[PASS] multiplying all sample weights by a constant does not change the loss
[PASS] rows without rec label get zero gradient (2 masked rows)
[PASS] band loss is unchanged when rec is fully masked
[PASS] batch with no valid label gives loss 0 and zero finite gradients
[PASS] empty batch: plain CE sum 0 and count 0
[PASS] rec: plain CE ignores weights and smoothing (1.00999)
[PASS] band: plain CE ignores weights and smoothing (1.52831)
[PASS] bf16 logits are handled in float32
[PASS] accumulator objective = mean of step losses
[PASS] accumulator plain CE is the pooled mean over valid rows (not a mean of batch means)
[PASS] add_logits gives the same plain CE a

/tmp/ipykernel_237443/3684321815.py:72: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  check(abs(float(output.total) - (float(output.head_loss["rec"]) + 2.0 * float(output.head_loss["band"]))) < TOLERANCE,


# Test: Checkpoint Save, Resume, RNG and Run Status

In [10]:
# Test: Checkpoint Save, Resume, RNG and Run Status

import copy
import json
import random
import tempfile
from collections.abc import Mapping
from pathlib import Path
from unittest.mock import patch

import numpy as np
import torch
from torch import nn

from digikala_review_transformer.training import checkpoint

# ---- Section: Test Helpers ----
# Keep the notebook's RNG state unchanged after running this test cell.
_checkpoint_initial_rng = checkpoint.capture_rng()
_checkpoint_failures = []


def _checkpoint_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _checkpoint_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


def _assert_nested_equal(left, right, path="root"):
    """Assert equality for nested checkpoint state, including tensors."""
    if isinstance(left, torch.Tensor):
        assert isinstance(right, torch.Tensor), f"{path}: tensor type mismatch"
        assert left.dtype == right.dtype, f"{path}: tensor dtype mismatch"
        assert torch.equal(left.cpu(), right.cpu()), f"{path}: tensor value mismatch"
        return

    if isinstance(left, np.ndarray):
        assert isinstance(right, np.ndarray), f"{path}: array type mismatch"
        assert np.array_equal(left, right), f"{path}: array value mismatch"
        return

    if isinstance(left, Mapping):
        assert isinstance(right, Mapping), f"{path}: mapping type mismatch"
        assert set(left) == set(right), f"{path}: mapping keys differ"
        for key in left:
            _assert_nested_equal(left[key], right[key], f"{path}.{key}")
        return

    if isinstance(left, (list, tuple)):
        assert isinstance(right, (list, tuple)), f"{path}: sequence type mismatch"
        assert len(left) == len(right), f"{path}: sequence lengths differ"
        for index, (left_item, right_item) in enumerate(zip(left, right)):
            _assert_nested_equal(left_item, right_item, f"{path}[{index}]")
        return

    assert left == right, f"{path}: values differ ({left!r} != {right!r})"


def _build_training_bundle(seed):
    """Build matching CPU model, optimizer, and scheduler objects."""
    torch.manual_seed(seed)

    model = nn.Sequential(
        nn.Linear(3, 5),
        nn.Tanh(),
        nn.Linear(5, 2),
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1.0e-3,
        weight_decay=0.01,
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: max(0.0, 1.0 - step / 10.0),
    )
    return model, optimizer, scheduler


_checkpoint_tmp = tempfile.TemporaryDirectory(
    prefix="transformer_checkpoint_test_"
)
_checkpoint_root = Path(_checkpoint_tmp.name)
_checkpoint_fixture = {}


# ---- Section: Last Checkpoint Round-Trip ----
# Create optimizer state, save a checkpoint, and restore it into fresh objects.
def _test_last_checkpoint_roundtrip():
    source_model, source_optimizer, source_scheduler = _build_training_bundle(101)

    inputs = torch.tensor(
        [
            [0.2, -0.4, 0.8],
            [1.0, 0.3, -0.5],
        ],
        dtype=torch.float32,
    )
    source_optimizer.zero_grad(set_to_none=True)
    loss = source_model(inputs).square().mean()
    loss.backward()
    source_optimizer.step()
    source_scheduler.step()

    state = checkpoint.TrainState(
        epochs_done=1,
        global_step=1,
        best_score=0.41,
        best_epoch=1,
        bad_epochs=0,
        history=[
            {
                "epoch": 1,
                "train_objective": float(loss.detach().item()),
            }
        ],
        train_seconds=2.5,
    )

    expected_model = {
        name: value.detach().cpu().clone()
        for name, value in source_model.state_dict().items()
    }
    expected_optimizer = copy.deepcopy(source_optimizer.state_dict())
    expected_scheduler = copy.deepcopy(source_scheduler.state_dict())

    last_path = _checkpoint_root / "roundtrip" / "last.pt"
    checkpoint.save_last(
        last_path,
        source_model,
        source_optimizer,
        source_scheduler,
        state,
        config_hash="checkpoint-test-hash",
    )

    payload = checkpoint.load_last(
        last_path,
        expected_hash="checkpoint-test-hash",
    )

    target_model, target_optimizer, target_scheduler = _build_training_bundle(202)
    restored_state = checkpoint.restore_last(
        payload,
        target_model,
        target_optimizer,
        target_scheduler,
    )

    for name, expected_value in expected_model.items():
        actual_value = target_model.state_dict()[name].detach().cpu()
        assert torch.equal(actual_value, expected_value), (
            f"Model tensor {name!r} was not restored exactly."
        )

    _assert_nested_equal(
        expected_optimizer,
        target_optimizer.state_dict(),
        "optimizer_state_dict",
    )
    _assert_nested_equal(
        expected_scheduler,
        target_scheduler.state_dict(),
        "scheduler_state_dict",
    )
    _assert_nested_equal(
        state.to_dict(),
        restored_state.to_dict(),
        "train_state",
    )

    assert payload["config_hash"] == "checkpoint-test-hash"
    _checkpoint_fixture["last_path"] = last_path


_checkpoint_check(
    "last.pt round-trip restores model, optimizer, scheduler, and TrainState",
    _test_last_checkpoint_roundtrip,
    "check save_last/load_last/restore_last and resolve the first reported mismatch",
)


# ---- Section: Configuration Hash Guard ----
# A checkpoint made for another resolved configuration must not be resumed.
def _test_config_hash_mismatch():
    last_path = _checkpoint_fixture.get("last_path")
    assert last_path is not None, "The round-trip checkpoint was not created."

    try:
        checkpoint.load_last(
            last_path,
            expected_hash="deliberately-wrong-hash",
        )
    except checkpoint.CheckpointError as error:
        assert "config hash" in str(error).lower(), (
            "The error should clearly identify the configuration hash mismatch."
        )
    else:
        raise AssertionError("A mismatched config hash was accepted.")


_checkpoint_check(
    "load_last rejects a mismatched configuration hash",
    _test_config_hash_mismatch,
    "ensure the run uses the original config hash or a new run directory",
)


# ---- Section: Best Weights ----
# Check fp32 floating weights, preserved integer buffers, and strict loading.
def _test_best_weights():
    source_model = nn.Linear(3, 2)
    source_model.register_buffer(
        "integer_marker",
        torch.tensor([17], dtype=torch.int64),
    )
    source_model = source_model.half()

    target_model = nn.Linear(3, 2)
    target_model.register_buffer(
        "integer_marker",
        torch.tensor([0], dtype=torch.int64),
    )

    best_path = _checkpoint_root / "best_model.pt"
    checkpoint.save_best_weights(best_path, source_model)

    saved_state = torch.load(
        best_path,
        map_location="cpu",
        weights_only=True,
    )

    for name, value in saved_state.items():
        if torch.is_floating_point(value):
            assert value.dtype == torch.float32, (
                f"Floating tensor {name!r} was not saved as fp32."
            )

    assert saved_state["integer_marker"].dtype == torch.int64
    checkpoint.load_best_weights(best_path, target_model)

    for name, source_value in source_model.state_dict().items():
        expected = source_value.detach().cpu()
        if torch.is_floating_point(expected):
            expected = expected.to(dtype=torch.float32)

        actual = target_model.state_dict()[name].detach().cpu()
        assert torch.equal(actual, expected), (
            f"Best-model tensor {name!r} did not load as expected."
        )


_checkpoint_check(
    "best_model.pt stores floating weights as fp32 and loads strictly",
    _test_best_weights,
    "check save_best_weights/load_best_weights and the model state_dict keys",
)


# ---- Section: Atomic Replacement and Temporary Files ----
# Simulate transient Windows file locks and verify failed saves preserve old data.
def _test_atomic_replace_retry():
    target_path = _checkpoint_root / "retry" / "checkpoint.pt"
    real_replace = checkpoint.os.replace
    attempts = {"count": 0}

    def fail_twice_then_replace(source, destination):
        attempts["count"] += 1
        if attempts["count"] <= 2:
            raise PermissionError("simulated temporary file lock")
        return real_replace(source, destination)

    with patch.object(
        checkpoint.os,
        "replace",
        new=fail_twice_then_replace,
    ):
        checkpoint.atomic_torch_save(
            {"sentinel": 23},
            target_path,
            max_attempts=4,
            retry_delay_seconds=0,
        )

    assert attempts["count"] == 3, (
        f"Expected 3 replacement attempts; got {attempts['count']}."
    )
    saved = torch.load(
        target_path,
        map_location="cpu",
        weights_only=True,
    )
    assert saved == {"sentinel": 23}


_checkpoint_check(
    "atomic_torch_save retries transient replacement errors",
    _test_atomic_replace_retry,
    "check the retry policy and same-directory temporary-file replacement",
)


def _test_failed_replace_preserves_existing_checkpoint():
    target_path = _checkpoint_root / "preserve" / "checkpoint.pt"
    checkpoint.atomic_torch_save(
        {"version": "old"},
        target_path,
        retry_delay_seconds=0,
    )

    real_replace = checkpoint.os.replace  # noqa: F841
    attempts = {"count": 0}

    def always_fail_replace(source, destination):
        attempts["count"] += 1
        raise PermissionError("simulated persistent file lock")

    try:
        with patch.object(
            checkpoint.os,
            "replace",
            new=always_fail_replace,
        ):
            checkpoint.atomic_torch_save(
                {"version": "new"},
                target_path,
                max_attempts=2,
                retry_delay_seconds=0,
            )
    except checkpoint.CheckpointError:
        pass
    else:
        raise AssertionError("A permanently blocked replacement did not fail.")

    assert attempts["count"] == 2
    saved = torch.load(
        target_path,
        map_location="cpu",
        weights_only=True,
    )
    assert saved == {"version": "old"}, (
        "The existing checkpoint was changed after replacement failed."
    )

    temporary_files = list(
        target_path.parent.glob(f".{target_path.name}.*.tmp")
    )
    assert not temporary_files, (
        f"Temporary files were left behind: {temporary_files}"
    )


_checkpoint_check(
    "failed replacement preserves the old checkpoint and removes the temp file",
    _test_failed_replace_preserves_existing_checkpoint,
    "check atomic replacement cleanup and ensure the original file is never overwritten on failure",
)


def _test_serialization_failure_cleans_temp_file():
    target_path = _checkpoint_root / "serialization_failure" / "broken.pt"

    try:
        checkpoint.atomic_torch_save(
            {"not_serializable": lambda value: value},
            target_path,
            retry_delay_seconds=0,
        )
    except checkpoint.CheckpointError:
        pass
    else:
        raise AssertionError("An unserializable payload did not raise CheckpointError.")

    assert not target_path.exists(), "A target checkpoint was created unexpectedly."
    temporary_files = list(
        target_path.parent.glob(f".{target_path.name}.*.tmp")
    )
    assert not temporary_files, (
        f"Temporary files were left behind: {temporary_files}"
    )


_checkpoint_check(
    "serialization failure removes the incomplete temporary file",
    _test_serialization_failure_cleans_temp_file,
    "check atomic_torch_save cleanup when torch.save fails",
)


# ---- Section: RNG Round-Trip ----
# Verify Python, NumPy, PyTorch CPU, and available CUDA RNG states are restored.
def _test_rng_roundtrip():
    original_state = checkpoint.capture_rng()

    try:
        expected_state = checkpoint.capture_rng()

        expected_python = random.random()
        expected_numpy = np.random.random(4)
        expected_cpu = torch.rand(5, device="cpu")

        expected_cuda = None
        if torch.cuda.is_available():
            expected_cuda = torch.rand(5, device="cuda").cpu()

        checkpoint.restore_rng(expected_state)

        actual_python = random.random()
        actual_numpy = np.random.random(4)
        actual_cpu = torch.rand(5, device="cpu")

        assert actual_python == expected_python
        assert np.array_equal(actual_numpy, expected_numpy)
        assert torch.equal(actual_cpu, expected_cpu)

        if expected_cuda is not None:
            actual_cuda = torch.rand(5, device="cuda").cpu()
            assert torch.equal(actual_cuda, expected_cuda)

    finally:
        checkpoint.restore_rng(original_state)


_checkpoint_check(
    "capture_rng/restore_rng reproduce Python, NumPy, CPU, and available CUDA draws",
    _test_rng_roundtrip,
    "check capture_rng/restore_rng and run the test on the same CUDA device setup",
)


# ---- Section: Run Status and Guarded Cleanup ----
# Only a complete run may remove its resumable last.pt checkpoint.
def _test_run_status_and_cleanup():
    run_dir = _checkpoint_root / "status_test"
    last_path = run_dir / "last.pt"
    summary_path = run_dir / "run_summary.json"

    assert checkpoint.run_status(run_dir) == "fresh"

    run_dir.mkdir(parents=True, exist_ok=True)
    last_path.write_bytes(b"resume marker")
    assert checkpoint.run_status(run_dir) == "resumable"

    try:
        checkpoint.cleanup_after_complete(run_dir)
    except checkpoint.CheckpointError:
        pass
    else:
        raise AssertionError("Cleanup was allowed before the run was complete.")

    assert last_path.is_file(), "last.pt was removed from an incomplete run."

    summary_path.write_text(
        json.dumps({"status": "complete"}),
        encoding="utf-8",
    )
    assert checkpoint.run_status(run_dir) == "complete"

    checkpoint.cleanup_after_complete(run_dir)
    assert not last_path.exists(), "last.pt was not removed after completion."
    assert checkpoint.run_status(run_dir) == "complete"


_checkpoint_check(
    "run_status detects fresh/resumable/complete and guards last.pt cleanup",
    _test_run_status_and_cleanup,
    "check run_summary.json status handling and cleanup_after_complete",
)


# ---- Section: Cleanup and Final Result ----
# Restore the notebook RNG and remove all temporary test files.
_checkpoint_check(
    "notebook RNG state is restored after the tests",
    lambda: checkpoint.restore_rng(_checkpoint_initial_rng),
    "check restore_rng and the saved notebook RNG state",
)

_checkpoint_tmp.cleanup()

if _checkpoint_failures:
    print(
        f"[FAIL] {len(_checkpoint_failures)} checkpoint test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL CHECKPOINT TESTS PASSED")

[PASS] last.pt round-trip restores model, optimizer, scheduler, and TrainState
[PASS] load_last rejects a mismatched configuration hash
[PASS] best_model.pt stores floating weights as fp32 and loads strictly
[PASS] atomic_torch_save retries transient replacement errors
[PASS] failed replacement preserves the old checkpoint and removes the temp file
[PASS] serialization failure removes the incomplete temporary file
[PASS] capture_rng/restore_rng reproduce Python, NumPy, CPU, and available CUDA draws
[PASS] run_status detects fresh/resumable/complete and guards last.pt cleanup
[PASS] notebook RNG state is restored after the tests
[PASS] ALL CHECKPOINT TESTS PASSED


# Inspect: Training Loop Dependencies (Read-Only)

In [ ]:
# Inspect: Training Loop Dependencies (Read-Only)

import ast
from pathlib import Path

# ---- Section: Locate the Project ----
# Find the repository root without importing or modifying project modules.
_current_dir = Path.cwd()
_project_root = next(
    (
        candidate
        for candidate in (_current_dir, *_current_dir.parents)
        if (candidate / "src" / "digikala_review_transformer").is_dir()
    ),
    None,
)

if _project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "'src/digikala_review_transformer'. Run this cell from the project."
    )


# ---- Section: Inspection Targets ----
# Inspect only interfaces and constants needed to connect the training loop.
_targets = [
    (
        "src/digikala_review_transformer/config.py",
        {"ROOT", "SPLIT_SEED", "ARTIFACTS"},
    ),
    (
        "src/digikala_review_transformer/training/run_config.py",
        {
            "CONFIG_SCHEMA",
            "RUNTIME_ONLY_KEYS",
            "load_run_config",
            "config_hash",
            "project_path",
            "run_dir",
            "dump_resolved_config",
        },
    ),
    (
        "src/digikala_review_transformer/training/data.py",
        {
            "HEADS",
            "IGNORE_INDEX",
            "SplitData",
            "BatchedTokenDataset",
            "open_pack",
            "class_weight_vector",
            "subset_row_indices",
            "load_split",
            "LengthBucketedBatches",
            "sorted_eval_batches",
            "make_loader",
        },
    ),
    (
        "src/digikala_review_transformer/training/tokenization.py",
        {
            "load_tokenizer",
            "get_split_tokens",
            "default_cache_dir",
        },
    ),
    (
        "src/digikala_review_transformer/training/model.py",
        {
            "TwoHeadClassifier",
            "build_model",
            "count_parameters",
            "model_summary",
            "parameter_groups",
        },
    ),
    (
        "src/digikala_review_transformer/training/losses.py",
        {
            "LossOutput",
            "TwoHeadLoss",
            "build_loss",
            "LossAccumulator",
        },
    ),
    (
        "src/digikala_review_transformer/training/checkpoint.py",
        {
            "TrainState",
            "save_last",
            "load_last",
            "restore_last",
            "save_best_weights",
            "load_best_weights",
            "run_status",
            "cleanup_after_complete",
        },
    ),
    (
        "src/digikala_review_transformer/labels.py",
        {
            "CLASSES",
            "CORR_COL",
            "RAW_COL",
            "class_weights",
            "decode",
            "encode",
            "label_map",
        },
    ),
    (
        "src/digikala_review_transformer/metrics.py",
        {
            "evaluate",
            "per_class_table",
        },
    ),
    (
        "src/digikala_review_transformer/utils.py",
        {"timer"},
    ),
]


# ---- Section: AST Helpers ----
# Report source-level interfaces without executing any project code.
def _assigned_names(target):
    if isinstance(target, ast.Name):
        return {target.id}
    if isinstance(target, (ast.Tuple, ast.List)):
        names = set()
        for item in target.elts:
            names.update(_assigned_names(item))
        return names
    return set()


def _node_names(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        return {node.name}
    if isinstance(node, ast.Assign):
        names = set()
        for target in node.targets:
            names.update(_assigned_names(target))
        return names
    if isinstance(node, ast.AnnAssign):
        return _assigned_names(node.target)
    return set()


def _format_function_signature(node):
    prefix = "async def" if isinstance(node, ast.AsyncFunctionDef) else "def"
    arguments = ast.unparse(node.args)
    returns = (
        f" -> {ast.unparse(node.returns)}"
        if node.returns is not None
        else ""
    )
    decorators = [
        f"@{ast.unparse(decorator)}"
        for decorator in node.decorator_list
    ]
    signature = f"{prefix} {node.name}({arguments}){returns}"
    return "\n".join([*decorators, signature])


def _print_class_interface(node):
    decorators = [
        f"@{ast.unparse(decorator)}"
        for decorator in node.decorator_list
    ]
    bases = ", ".join(ast.unparse(base) for base in node.bases)
    class_line = f"class {node.name}({bases}):" if bases else f"class {node.name}:"

    for decorator in decorators:
        print(decorator)
    print(class_line)

    docstring = ast.get_docstring(node)
    if docstring:
        print(f'  doc: {docstring.splitlines()[0]}')

    for child in node.body:
        if isinstance(child, ast.AnnAssign):
            field_name = ast.unparse(child.target)
            annotation = ast.unparse(child.annotation)
            default = (
                f" = {ast.unparse(child.value)}"
                if child.value is not None
                else ""
            )
            print(f"  field: {field_name}: {annotation}{default}")

        elif isinstance(child, ast.Assign):
            source = ast.get_source_segment(_source_text, child)
            if source:
                print(f"  class assignment: {source.strip()}")

        elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
            signature = _format_function_signature(child).replace("\n", "\n  ")
            print(f"  {signature}")


# ---- Section: Print Selected Interfaces ----
# Print selected constants, function signatures, and class fields/method signatures.
for _relative_path, _selected_names in _targets:
    _file_path = _project_root / _relative_path
    print(f"\n===== {_relative_path} =====")

    if not _file_path.is_file():
        print("[MISSING] File not found at this path.")
        continue

    _source_text = _file_path.read_text(encoding="utf-8")
    _module = ast.parse(_source_text, filename=str(_file_path))
    _found_names = set()

    for _node in _module.body:
        _matched = _node_names(_node) & _selected_names
        if not _matched:
            continue

        _found_names.update(_matched)

        if isinstance(_node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(_format_function_signature(_node))
            _docstring = ast.get_docstring(_node)
            if _docstring:
                print(f"  doc: {_docstring.splitlines()[0]}")

        elif isinstance(_node, ast.ClassDef):
            _print_class_interface(_node)

        else:
            _source_segment = ast.get_source_segment(_source_text, _node)
            if _source_segment:
                print(_source_segment.strip())

    _missing_names = _selected_names - _found_names
    if _missing_names:
        print(f"[NOT FOUND] {sorted(_missing_names)}")


===== src/digikala_review_transformer/config.py =====
ROOT = Path(__file__).resolve().parents[2]
ARTIFACTS = ROOT / "artifacts"
SPLIT_SEED = 2026

===== src/digikala_review_transformer/training/run_config.py =====
CONFIG_SCHEMA = {
    "run_name": str,
    "data": {"pack_dir": str},
    "model": {
        "pretrained_path": str,
        "max_length": int,
        "head_dropout": float,
        "tokenizer_backend": str,
    },    "training": {
        "variant": str,
        "max_epochs": int,
        "early_stopping_patience": int,
        "early_stopping_min_delta": float,
        "batch_size": int,
        "eval_batch_size": int,
        "lr_encoder": float,
        "lr_heads": float,
        "weight_decay": float,
        "warmup_ratio": float,
        "max_grad_norm": float,
        "label_smoothing": float,
        "loss_weight_band": float,
        "use_class_weights": bool,
        "bf16": bool,
        "num_workers": int,
        "log_every_steps": int,
        "train_subset_e

# Test: Training Loop, Evaluation Alignment, Resume and Early Stopping

In [2]:
# Test: Training Loop, Evaluation Alignment, Resume and Early Stopping

import copy
import json
import math
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn  # noqa: F811

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    run_config,
    tokenization,
)
from digikala_review_transformer.training import (
    model as model_module,
)

# ---- Section: Test Helpers ----
# Restore the notebook RNG after exercising training, checkpointing, and resume.
_loop_initial_rng = checkpoint.capture_rng()
_loop_failures = []


def _loop_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _loop_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


class _ToyEncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _ToyEncoder(nn.Module):
    def __init__(self, vocab_size=64, hidden_size=12):
        super().__init__()
        self.config = type("ToyConfig", (), {"hidden_size": hidden_size})()
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        hidden = self.embeddings(input_ids)
        hidden = torch.tanh(self.projection(hidden))
        hidden = self.norm(hidden)
        return _ToyEncoderOutput(last_hidden_state=hidden)


def _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.1):
    model_module.seed_everything(seed)
    encoder = _ToyEncoder()
    model_module.seed_everything(seed)
    return model_module.TwoHeadClassifier(encoder, head_dropout=head_dropout)


def _make_token_arrays(sequences):
    offsets = [0]
    flat_ids = []
    for sequence in sequences:
        token_ids = [int(value) for value in sequence]
        flat_ids.extend(token_ids)
        offsets.append(len(flat_ids))
    return tokenization.TokenArrays(
        ids=np.asarray(flat_ids, dtype=np.int32),
        offsets=np.asarray(offsets, dtype=np.int64),
    )


def _make_split(
    split_name,
    n_rows,
    *,
    seed,
    id_start,
    variant="B",
    mask_rec_every=5,
):
    rng = np.random.default_rng(seed)
    sequences = []
    rec_corr = np.empty(n_rows, dtype=np.int64)
    band_corr = np.empty(n_rows, dtype=np.int64)
    rec_raw = np.empty(n_rows, dtype=np.int64)
    band_raw = np.empty(n_rows, dtype=np.int64)

    for row_index in range(n_rows):
        rec_class = row_index % 3
        band_class = (row_index // 3 + row_index) % 3
        extra_length = int((row_index * 5 + 3) % 9) + 2

        anchor_rec = 4 + rec_class
        anchor_band = 10 + band_class
        body_tokens = rng.integers(16, 48, size=extra_length).tolist()
        sequence = [2, anchor_rec, anchor_band, *body_tokens, 3]
        sequences.append(sequence)

        rec_corr[row_index] = rec_class
        band_corr[row_index] = band_class
        rec_raw[row_index] = (
            rec_class if row_index % 7 != 0 else (rec_class + 1) % 3
        )
        band_raw[row_index] = (
            band_class if row_index % 9 != 0 else (band_class + 1) % 3
        )

    if mask_rec_every is not None:
        for row_index in range(0, n_rows, mask_rec_every):
            rec_corr[row_index] = data.IGNORE_INDEX
            rec_raw[row_index] = data.IGNORE_INDEX

    if variant == "C":
        w_rec = np.linspace(0.6, 1.0, n_rows, dtype=np.float32)
        w_band = np.linspace(1.0, 0.7, n_rows, dtype=np.float32)
    else:
        w_rec = np.ones(n_rows, dtype=np.float32)
        w_band = np.ones(n_rows, dtype=np.float32)

    arrays = _make_token_arrays(sequences)
    ids = np.arange(id_start, id_start + n_rows, dtype=np.int64)

    return data.SplitData(
        split=split_name,
        ids=ids,
        arrays=arrays,
        pad_id=0,
        y={"rec": rec_corr, "band": band_corr},
        y_raw={"rec": rec_raw, "band": band_raw},
        w={"rec": w_rec, "band": w_band},
        stats={"rows": n_rows},
        origin="test",
        source_rows=n_rows,
    )


def _make_config(
    output_root,
    *,
    run_name="loop_test",
    variant="B",
    max_epochs=2,
    patience=2,
    min_delta=0.0,
    batch_size=8,
    eval_batch_size=7,
    lr_encoder=2.0e-4,
    lr_heads=1.0e-3,
    log_every_steps=2,
):
    return {
        "run_name": run_name,
        "seed": config.SPLIT_SEED,
        "data": {"pack_dir": "data/training_pack"},
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 32,
            "head_dropout": 0.1,
            "tokenizer_backend": "hf_fast",
        },
        "training": {
            "variant": variant,
            "max_epochs": max_epochs,
            "early_stopping_patience": patience,
            "early_stopping_min_delta": min_delta,
            "batch_size": batch_size,
            "eval_batch_size": eval_batch_size,
            "lr_encoder": lr_encoder,
            "lr_heads": lr_heads,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
            "max_grad_norm": 1.0,
            "label_smoothing": 0.1,
            "loss_weight_band": 1.0,
            "use_class_weights": True,
            "bf16": False,
            "num_workers": 0,
            "log_every_steps": log_every_steps,
            "train_subset_eval_rows": 12,
        },
        "output": {"root": str(output_root)},
    }


def _manual_probabilities(model, split):
    model.eval()
    dataset = split.dataset()
    batch = dataset[np.arange(split.n_rows, dtype=np.int64)]
    with torch.inference_mode():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        return {
            head: torch.softmax(logits[head].float(), dim=-1).cpu().numpy()
            for head in data.HEADS
        }


_loop_tmp = tempfile.TemporaryDirectory(prefix="transformer_loop_test_")
_loop_root = Path(_loop_tmp.name)


# ---- Section: Scheduler and Evaluation Order ----
# Check the 6% warmup/decay schedule and length-sorted evaluation restoration.
def _test_linear_scheduler():
    model = _build_toy_model()
    groups = model_module.parameter_groups(
        model,
        lr_encoder=2.0e-5,
        lr_heads=1.0e-4,
        weight_decay=0.01,
    )
    optimizer = torch.optim.AdamW(groups)
    scheduler, warmup_steps = loop.make_linear_scheduler(
        optimizer,
        total_steps=100,
        warmup_ratio=0.06,
    )

    assert warmup_steps == 6
    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)

    for _ in range(warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 2.0e-5, rel_tol=1e-6)
    assert math.isclose(heads_lr, 1.0e-4, rel_tol=1e-6)

    for _ in range(100 - warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)


_loop_check(
    "linear scheduler warms up for 6% of total steps and decays to zero",
    _test_linear_scheduler,
    "check make_linear_scheduler warmup_steps and decay denominator",
)


def _test_eval_restores_original_order_and_schema():
    split = _make_split("validation", 23, seed=11, id_start=5000)
    model = _build_toy_model(seed=19)

    expected = _manual_probabilities(model, split)
    evaluation = loop.evaluate_split(
        model,
        split,
        batch_size=5,
        device="cpu",
        bf16=False,
        include_raw=True,
    )

    for head in data.HEADS:
        assert np.allclose(
            evaluation.probabilities[head],
            expected[head],
            rtol=1e-5,
            atol=1e-6,
        ), f"Restored probabilities for {head} do not match row order."
        assert math.isfinite(evaluation.plain_ce[head])
        assert math.isfinite(evaluation.corrected_metrics[head]["macro_f1"])
        assert math.isfinite(evaluation.raw_metrics[head]["macro_f1"])

    frame = loop.probability_frame(split, evaluation.probabilities)
    expected_columns = [
        "id",
        *(f"p_{name}" for name in labels.CLASSES["rec"]),
        *(f"p_{name}" for name in labels.CLASSES["band"]),
    ]
    assert list(frame.columns) == expected_columns
    assert np.array_equal(frame["id"].to_numpy(), split.ids)
    for column in expected_columns[1:]:
        assert frame[column].dtype == np.float32


_loop_check(
    "sorted evaluation restores original id order and float32 probability columns",
    _test_eval_restores_original_order_and_schema,
    "check sorted_eval_batches index restoration and probability_frame columns",
)


# ---- Section: Full Fit, Pause/Resume, and Hash Guard ----
# A 1-epoch pause followed by resume must match an uninterrupted 2-epoch run.
def _test_fit_and_exact_resume():
    train_split = _make_split("train", 24, seed=101, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=202, id_start=1000)
    subset_split = _make_split(
        "train_subset",
        12,
        seed=303,
        id_start=3000,
        mask_rec_every=6,
    )

    direct_cfg = _make_config(
        _loop_root / "direct",
        run_name="direct_two_epochs",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    direct_model = _build_toy_model(seed=config.SPLIT_SEED)
    direct_result = loop.fit(
        direct_cfg,
        direct_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert direct_result.status == "trained"
    assert direct_result.epochs_done == 2
    assert direct_result.global_step == 6

    resume_cfg = _make_config(
        _loop_root / "resumed",
        run_name="paused_then_resumed",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    paused_model = _build_toy_model(seed=config.SPLIT_SEED)
    paused_result = loop.fit(
        resume_cfg,
        paused_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
        max_epochs_this_call=1,
    )

    assert paused_result.status == "paused"
    assert paused_result.stop_reason == "paused_by_call_limit"
    assert paused_result.epochs_done == 1
    assert (paused_result.run_directory / "last.pt").is_file()

    # Append an uncommitted step row to simulate a crash during epoch 2.
    train_log_path = paused_result.run_directory / "train_log.csv"
    log_df = pd.read_csv(train_log_path)
    extra_row = log_df.iloc[[-1]].copy()
    extra_row["global_step"] = 999
    pd.concat([log_df, extra_row], ignore_index=True).to_csv(
        train_log_path,
        index=False,
    )

    wrong_cfg = copy.deepcopy(resume_cfg)
    wrong_cfg["training"]["lr_heads"] = 9.0e-4
    try:
        loop.fit(
            wrong_cfg,
            _build_toy_model(seed=config.SPLIT_SEED),
            train_split,
            val_split,
            subset_split,
            run_directory=paused_result.run_directory,
            device="cpu",
        )
    except checkpoint.CheckpointError as error:
        assert "config hash" in str(error).lower()
    else:
        raise AssertionError("Mismatched config hash did not fail on resume.")

    resumed_model = _build_toy_model(seed=999)
    resumed_result = loop.fit(
        resume_cfg,
        resumed_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert resumed_result.status == "trained"
    assert resumed_result.epochs_done == 2
    assert resumed_result.global_step == 6

    for name, direct_tensor in direct_model.state_dict().items():
        resumed_tensor = resumed_model.state_dict()[name]
        assert torch.allclose(
            direct_tensor,
            resumed_tensor,
            rtol=1e-6,
            atol=1e-6,
        ), f"Final best weight {name} differed after resume."

    direct_history = pd.read_csv(direct_result.run_directory / "history.csv")
    resumed_history = pd.read_csv(resumed_result.run_directory / "history.csv")
    required_history_columns = {
        "train_objective",
        "train_plain_ce_rec",
        "train_plain_ce_band",
        "train_subset_plain_ce_rec",
        "train_subset_plain_ce_band",
        "val_plain_ce_rec",
        "val_plain_ce_band",
        "val_acc_rec",
        "val_acc_band",
        "val_macro_f1_rec",
        "val_macro_f1_band",
        "val_macro_f1_rec_raw",
        "val_macro_f1_band_raw",
        "train_subset_macro_f1_rec",
        "train_subset_macro_f1_band",
        "val_mean_macro_f1",
        "lr_end",
        "epoch_seconds",
        "is_best",
    }
    assert required_history_columns.issubset(direct_history.columns)

    compared_columns = [
        column
        for column in direct_history.columns
        if column != "epoch_seconds"
    ]
    pd.testing.assert_frame_equal(
        direct_history[compared_columns],
        resumed_history[compared_columns],
        rtol=1e-5,
        atol=1e-6,
    )

    repaired_log = pd.read_csv(train_log_path)
    assert int(repaired_log["global_step"].max()) == 6
    assert (resumed_result.run_directory / "best_model.pt").is_file()
    assert (resumed_result.run_directory / "val_probs_best.parquet").is_file()
    assert (resumed_result.run_directory / "train.log").is_file()


_loop_check(
    "fit supports pause/resume, prunes uncommitted logs, and rejects hash changes",
    _test_fit_and_exact_resume,
    "check save_last/restore_last integration and epoch-boundary state checks",
)


# ---- Section: Shared Early Stopping and Best Weight Reload ----
# When epoch 2 does not beat min_delta, patience=1 stops and reloads epoch 1.
def _test_early_stopping_and_best_reload():
    train_split = _make_split("train", 16, seed=404, id_start=400)
    val_split = _make_split("validation", 15, seed=505, id_start=1400)
    subset_split = _make_split("train_subset", 10, seed=606, id_start=2400)

    cfg = _make_config(
        _loop_root / "early_stop",
        run_name="early_stop_run",
        max_epochs=4,
        patience=1,
        min_delta=2.0,
        lr_encoder=5.0e-3,
        lr_heads=1.0e-2,
    )
    model = _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.0)
    result = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert result.status == "trained"
    assert result.stop_reason == "early_stopping_patience"
    assert result.epochs_done == 2
    assert result.best_epoch == 1
    assert [row["is_best"] for row in result.history] == [True, False]

    best_state = torch.load(
        result.best_model_path,
        map_location="cpu",
        weights_only=True,
    )
    last_payload = checkpoint.load_last(
        result.run_directory / "last.pt",
        expected_hash=run_config.config_hash(cfg),
    )

    for name, best_tensor in best_state.items():
        assert torch.equal(model.state_dict()[name].cpu(), best_tensor)

    changed_after_best = any(
        not torch.equal(
            best_state[name].cpu(),
            last_payload["model_state_dict"][name].cpu(),
        )
        for name in best_state
    )
    assert changed_after_best, "Epoch 2 did not change weights from epoch 1."

    # Mark the run complete and verify a second call skips without overwriting.
    (result.run_directory / "run_summary.json").write_text(
        json.dumps({"status": "complete"}),
        encoding="utf-8",
    )
    skipped = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    assert skipped.skipped is True
    assert skipped.status == "complete"


_loop_check(
    "shared early stopping stops on patience and reloads best_model.pt",
    _test_early_stopping_and_best_reload,
    "check val_mean_macro_f1 improvement logic and load_best_weights at the end of fit",
)


# ---- Section: Cleanup and Final Result ----
# Restore the notebook RNG and remove temporary test artifacts.
_loop_check(
    "notebook RNG state is restored after training loop tests",
    lambda: checkpoint.restore_rng(_loop_initial_rng),
    "check capture_rng/restore_rng around the test cell",
)

_loop_tmp.cleanup()

if _loop_failures:
    print(
        f"[FAIL] {len(_loop_failures)} training loop test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL TRAINING LOOP TESTS PASSED")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] linear scheduler warms up for 6% of total steps and decays to zero
[PASS] sorted evaluation restores original id order and float32 probability columns
[run] start run=direct_two_epochs device=cpu variant=C seed=2026
[model] parameter_counts={'total': 1026, 'trainable': 1026, 'encoder': 948, 'heads': 78}
[optimizer] fused=False steps_per_epoch=3 total_steps=6 warmup_steps=0
[train] run=direct_two_epochs epoch=1/2 step=2/3 global_step=2/6 progress=33.33% objective=2.208625 lr_encoder=1.333e-04 lr_heads=6.667e-04 samples_per_second=4217.61 ETA=0.0 min
[train] run=direct_two_epochs epoch=1/2 step=3/3 global_step=3/6 progress=50.00% objective=2.223766 lr_encoder=1.000e-04 lr_heads=5.000e-04 samples_per_second=4231.15 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.0s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.213672 val_mean_macro_f1=0.157407 best_epoch=1 is_best=True bad_epochs=0/2 epoch_seconds=0.

# Inspect: Evaluation Root Cause and Helper Bodies (Read-Only)

In [1]:
# Inspect: Evaluation Root Cause and Helper Bodies (Read-Only)

import ast
import traceback
from pathlib import Path

# ---- Section: Locate Project Root ----
# Read source files directly without changing anything on disk.
_current_dir = Path.cwd()
_project_root = next(
    (
        candidate
        for candidate in (_current_dir, *_current_dir.parents)
        if (candidate / "src" / "digikala_review_transformer").is_dir()
    ),
    None,
)

if _project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "'src/digikala_review_transformer'."
    )


# ---- Section: Print Exact Helper Bodies ----
# Inspect the bodies of the helpers called by _collect_split_outputs.
_source_targets = [
    (
        "src/digikala_review_transformer/training/data.py",
        {
            "BatchedTokenDataset",
            "SplitData",
            "sorted_eval_batches",
            "make_loader",
        },
    ),
    (
        "src/digikala_review_transformer/training/losses.py",
        {"plain_ce_sum"},
    ),
    (
        "src/digikala_review_transformer/metrics.py",
        {"evaluate"},
    ),
]

for _relative_path, _names in _source_targets:
    _path = _project_root / _relative_path
    _text = _path.read_text(encoding="utf-8")
    _tree = ast.parse(_text, filename=str(_path))
    print(f"\n===== {_relative_path} =====")
    for _node in _tree.body:
        if isinstance(_node, (ast.FunctionDef, ast.ClassDef)) and _node.name in _names:
            print(ast.get_source_segment(_text, _node))
            print()


# ---- Section: Print Underlying Evaluation Exception ----
# Run the existing toy evaluation call once and print __cause__ if it fails.
if "_make_split" in globals() and "_build_toy_model" in globals():
    from digikala_review_transformer.training import loop

    print("===== Underlying exception from evaluate_split =====")
    try:
        _debug_split = _make_split("validation", 23, seed=11, id_start=5000)
        _debug_model = _build_toy_model(seed=19)
        loop.evaluate_split(
            _debug_model,
            _debug_split,
            batch_size=5,
            device="cpu",
            bf16=False,
            include_raw=True,
        )
    except Exception as _error:  # noqa: BLE001
        print(f"Top-level: {type(_error).__name__}: {_error}")
        if _error.__cause__ is not None:
            print(
                f"Direct cause: "
                f"{type(_error.__cause__).__name__}: {_error.__cause__}"
            )
            traceback.print_exception(
                type(_error.__cause__),
                _error.__cause__,
                _error.__cause__.__traceback__,
            )
        else:
            traceback.print_exc()


===== src/digikala_review_transformer/training/data.py =====
class BatchedTokenDataset(Dataset):
    """Flat token arrays + targets + weights. dataset[row_indices] returns one padded batch."""

    def __init__(self, arrays, y, w, pad_id):
        self.ids = arrays.ids
        self.offsets = arrays.offsets
        self.lengths = arrays.lengths
        n_rows = len(self.lengths)
        self.y = {head: np.ascontiguousarray(y[head], dtype=np.int64) for head in HEADS}
        self.w = {head: np.ascontiguousarray(w[head], dtype=np.float32) for head in HEADS}
        for head in HEADS:
            if len(self.y[head]) != n_rows or len(self.w[head]) != n_rows:
                raise DataError(f"'{head}' targets/weights have a different length than the token arrays")
        self.pad_id = int(pad_id)

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, index):
        rows = np.asarray(index, dtype=np.int64)
        lengths = self.lengths[rows]
        steps = n

# test again

In [3]:
# Test: Training Loop, Evaluation Alignment, Resume and Early Stopping

import copy
import json
import math  # noqa: F811
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    run_config,
    tokenization,  # noqa: F811
)
from digikala_review_transformer.training import (
    model as model_module,  # noqa: F811
)

# ---- Section: Test Helpers ----
# Restore the notebook RNG after exercising training, checkpointing, and resume.
_loop_initial_rng = checkpoint.capture_rng()
_loop_failures = []


def _loop_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _loop_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


class _ToyEncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _ToyEncoder(nn.Module):
    def __init__(self, vocab_size=64, hidden_size=12):
        super().__init__()
        self.config = type("ToyConfig", (), {"hidden_size": hidden_size})()
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        hidden = self.embeddings(input_ids)
        hidden = torch.tanh(self.projection(hidden))
        hidden = self.norm(hidden)
        return _ToyEncoderOutput(last_hidden_state=hidden)


def _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.1):
    model_module.seed_everything(seed)
    encoder = _ToyEncoder()
    model_module.seed_everything(seed)
    return model_module.TwoHeadClassifier(encoder, head_dropout=head_dropout)


def _make_token_arrays(sequences):
    offsets = [0]
    flat_ids = []
    for sequence in sequences:
        token_ids = [int(value) for value in sequence]
        flat_ids.extend(token_ids)
        offsets.append(len(flat_ids))
    return tokenization.TokenArrays(
        ids=np.asarray(flat_ids, dtype=np.int32),
        offsets=np.asarray(offsets, dtype=np.int64),
    )


def _make_split(
    split_name,
    n_rows,
    *,
    seed,
    id_start,
    variant="B",
    mask_rec_every=5,
):
    rng = np.random.default_rng(seed)
    sequences = []
    rec_corr = np.empty(n_rows, dtype=np.int64)
    band_corr = np.empty(n_rows, dtype=np.int64)
    rec_raw = np.empty(n_rows, dtype=np.int64)
    band_raw = np.empty(n_rows, dtype=np.int64)

    for row_index in range(n_rows):
        rec_class = row_index % 3
        band_class = (row_index // 3 + row_index) % 3
        extra_length = int((row_index * 5 + 3) % 9) + 2

        anchor_rec = 4 + rec_class
        anchor_band = 10 + band_class
        body_tokens = rng.integers(16, 48, size=extra_length).tolist()
        sequence = [2, anchor_rec, anchor_band, *body_tokens, 3]
        sequences.append(sequence)

        rec_corr[row_index] = rec_class
        band_corr[row_index] = band_class
        rec_raw[row_index] = (
            rec_class if row_index % 7 != 0 else (rec_class + 1) % 3
        )
        band_raw[row_index] = (
            band_class if row_index % 9 != 0 else (band_class + 1) % 3
        )

    if mask_rec_every is not None:
        for row_index in range(0, n_rows, mask_rec_every):
            rec_corr[row_index] = data.IGNORE_INDEX
            rec_raw[row_index] = data.IGNORE_INDEX

    if variant == "C":
        w_rec = np.linspace(0.6, 1.0, n_rows, dtype=np.float32)
        w_band = np.linspace(1.0, 0.7, n_rows, dtype=np.float32)
    else:
        w_rec = np.ones(n_rows, dtype=np.float32)
        w_band = np.ones(n_rows, dtype=np.float32)

    arrays = _make_token_arrays(sequences)
    ids = np.arange(id_start, id_start + n_rows, dtype=np.int64)

    return data.SplitData(
        split=split_name,
        ids=ids,
        arrays=arrays,
        pad_id=0,
        y={"rec": rec_corr, "band": band_corr},
        y_raw={"rec": rec_raw, "band": band_raw},
        w={"rec": w_rec, "band": w_band},
        stats={"rows": n_rows},
        origin="test",
        source_rows=n_rows,
    )


def _make_config(
    output_root,
    *,
    run_name="loop_test",
    variant="B",
    max_epochs=2,
    patience=2,
    min_delta=0.0,
    batch_size=8,
    eval_batch_size=7,
    lr_encoder=2.0e-4,
    lr_heads=1.0e-3,
    log_every_steps=2,
):
    return {
        "run_name": run_name,
        "seed": config.SPLIT_SEED,
        "data": {"pack_dir": "data/training_pack"},
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 32,
            "head_dropout": 0.1,
            "tokenizer_backend": "hf_fast",
        },
        "training": {
            "variant": variant,
            "max_epochs": max_epochs,
            "early_stopping_patience": patience,
            "early_stopping_min_delta": min_delta,
            "batch_size": batch_size,
            "eval_batch_size": eval_batch_size,
            "lr_encoder": lr_encoder,
            "lr_heads": lr_heads,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
            "max_grad_norm": 1.0,
            "label_smoothing": 0.1,
            "loss_weight_band": 1.0,
            "use_class_weights": True,
            "bf16": False,
            "num_workers": 0,
            "log_every_steps": log_every_steps,
            "train_subset_eval_rows": 12,
        },
        "output": {"root": str(output_root)},
    }


def _manual_probabilities(model, split):
    model.eval()
    dataset = split.dataset()
    batch = dataset[np.arange(split.n_rows, dtype=np.int64)]
    with torch.inference_mode():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        return {
            head: torch.softmax(logits[head].float(), dim=-1).cpu().numpy()
            for head in data.HEADS
        }


_loop_tmp = tempfile.TemporaryDirectory(prefix="transformer_loop_test_")
_loop_root = Path(_loop_tmp.name)


# ---- Section: Scheduler and Evaluation Order ----
# Check the 6% warmup/decay schedule and length-sorted evaluation restoration.
def _test_linear_scheduler():
    model = _build_toy_model()
    groups = model_module.parameter_groups(
        model,
        lr_encoder=2.0e-5,
        lr_heads=1.0e-4,
        weight_decay=0.01,
    )
    optimizer = torch.optim.AdamW(groups)
    scheduler, warmup_steps = loop.make_linear_scheduler(
        optimizer,
        total_steps=100,
        warmup_ratio=0.06,
    )

    assert warmup_steps == 6
    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)

    for _ in range(warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 2.0e-5, rel_tol=1e-6)
    assert math.isclose(heads_lr, 1.0e-4, rel_tol=1e-6)

    for _ in range(100 - warmup_steps):
        optimizer.step()
        scheduler.step()

    encoder_lr, heads_lr = loop._learning_rates(optimizer)
    assert math.isclose(encoder_lr, 0.0, abs_tol=1e-12)
    assert math.isclose(heads_lr, 0.0, abs_tol=1e-12)


_loop_check(
    "linear scheduler warms up for 6% of total steps and decays to zero",
    _test_linear_scheduler,
    "check make_linear_scheduler warmup_steps and decay denominator",
)


def _test_eval_restores_original_order_and_schema():
    split = _make_split("validation", 23, seed=11, id_start=5000)
    model = _build_toy_model(seed=19)

    expected = _manual_probabilities(model, split)
    evaluation = loop.evaluate_split(
        model,
        split,
        batch_size=5,
        device="cpu",
        bf16=False,
        include_raw=True,
    )

    for head in data.HEADS:
        assert np.allclose(
            evaluation.probabilities[head],
            expected[head],
            rtol=1e-5,
            atol=1e-6,
        ), f"Restored probabilities for {head} do not match row order."
        assert math.isfinite(evaluation.plain_ce[head])
        assert math.isfinite(evaluation.corrected_metrics[head]["macro_f1"])
        assert math.isfinite(evaluation.raw_metrics[head]["macro_f1"])

    frame = loop.probability_frame(split, evaluation.probabilities)
    expected_columns = [
        "id",
        *(f"p_{name}" for name in labels.CLASSES["rec"]),
        *(f"p_{name}" for name in labels.CLASSES["band"]),
    ]
    assert list(frame.columns) == expected_columns
    assert np.array_equal(frame["id"].to_numpy(), split.ids)
    for column in expected_columns[1:]:
        assert frame[column].dtype == np.float32


_loop_check(
    "sorted evaluation restores original id order and float32 probability columns",
    _test_eval_restores_original_order_and_schema,
    "check sorted_eval_batches index restoration and probability_frame columns",
)


# ---- Section: Full Fit, Pause/Resume, and Hash Guard ----
# A 1-epoch pause followed by resume must match an uninterrupted 2-epoch run.
def _test_fit_and_exact_resume():
    train_split = _make_split("train", 24, seed=101, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=202, id_start=1000)
    subset_split = _make_split(
        "train_subset",
        12,
        seed=303,
        id_start=3000,
        mask_rec_every=6,
    )

    direct_cfg = _make_config(
        _loop_root / "direct",
        run_name="direct_two_epochs",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    direct_model = _build_toy_model(seed=config.SPLIT_SEED)
    direct_result = loop.fit(
        direct_cfg,
        direct_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert direct_result.status == "trained"
    assert direct_result.epochs_done == 2
    assert direct_result.global_step == 6

    resume_cfg = _make_config(
        _loop_root / "resumed",
        run_name="paused_then_resumed",
        variant="C",
        max_epochs=2,
        patience=2,
    )
    paused_model = _build_toy_model(seed=config.SPLIT_SEED)
    paused_result = loop.fit(
        resume_cfg,
        paused_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
        max_epochs_this_call=1,
    )

    assert paused_result.status == "paused"
    assert paused_result.stop_reason == "paused_by_call_limit"
    assert paused_result.epochs_done == 1
    assert (paused_result.run_directory / "last.pt").is_file()

    # Append an uncommitted step row to simulate a crash during epoch 2.
    train_log_path = paused_result.run_directory / "train_log.csv"
    log_df = pd.read_csv(train_log_path)
    extra_row = log_df.iloc[[-1]].copy()
    extra_row["global_step"] = 999
    pd.concat([log_df, extra_row], ignore_index=True).to_csv(
        train_log_path,
        index=False,
    )

    wrong_cfg = copy.deepcopy(resume_cfg)
    wrong_cfg["training"]["lr_heads"] = 9.0e-4
    try:
        loop.fit(
            wrong_cfg,
            _build_toy_model(seed=config.SPLIT_SEED),
            train_split,
            val_split,
            subset_split,
            run_directory=paused_result.run_directory,
            device="cpu",
        )
    except checkpoint.CheckpointError as error:
        assert "config hash" in str(error).lower()
    else:
        raise AssertionError("Mismatched config hash did not fail on resume.")

    resumed_model = _build_toy_model(seed=999)
    resumed_result = loop.fit(
        resume_cfg,
        resumed_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert resumed_result.status == "trained"
    assert resumed_result.epochs_done == 2
    assert resumed_result.global_step == 6

    for name, direct_tensor in direct_model.state_dict().items():
        resumed_tensor = resumed_model.state_dict()[name]
        assert torch.allclose(
            direct_tensor,
            resumed_tensor,
            rtol=1e-6,
            atol=1e-6,
        ), f"Final best weight {name} differed after resume."

    direct_history = pd.read_csv(direct_result.run_directory / "history.csv")
    resumed_history = pd.read_csv(resumed_result.run_directory / "history.csv")
    required_history_columns = {
        "train_objective",
        "train_plain_ce_rec",
        "train_plain_ce_band",
        "train_subset_plain_ce_rec",
        "train_subset_plain_ce_band",
        "val_plain_ce_rec",
        "val_plain_ce_band",
        "val_acc_rec",
        "val_acc_band",
        "val_macro_f1_rec",
        "val_macro_f1_band",
        "val_macro_f1_rec_raw",
        "val_macro_f1_band_raw",
        "train_subset_macro_f1_rec",
        "train_subset_macro_f1_band",
        "val_mean_macro_f1",
        "lr_end",
        "epoch_seconds",
        "is_best",
    }
    assert required_history_columns.issubset(direct_history.columns)

    compared_columns = [
        column
        for column in direct_history.columns
        if column != "epoch_seconds"
    ]
    pd.testing.assert_frame_equal(
        direct_history[compared_columns],
        resumed_history[compared_columns],
        rtol=1e-5,
        atol=1e-6,
    )

    repaired_log = pd.read_csv(train_log_path)
    assert int(repaired_log["global_step"].max()) == 6
    assert (resumed_result.run_directory / "best_model.pt").is_file()
    assert (resumed_result.run_directory / "val_probs_best.parquet").is_file()
    assert (resumed_result.run_directory / "train.log").is_file()


_loop_check(
    "fit supports pause/resume, prunes uncommitted logs, and rejects hash changes",
    _test_fit_and_exact_resume,
    "check save_last/restore_last integration and epoch-boundary state checks",
)


# ---- Section: Shared Early Stopping and Best Weight Reload ----
# When epoch 2 does not beat min_delta, patience=1 stops and reloads epoch 1.
def _test_early_stopping_and_best_reload():
    train_split = _make_split("train", 16, seed=404, id_start=400)
    val_split = _make_split("validation", 15, seed=505, id_start=1400)
    subset_split = _make_split("train_subset", 10, seed=606, id_start=2400)

    cfg = _make_config(
        _loop_root / "early_stop",
        run_name="early_stop_run",
        max_epochs=4,
        patience=1,
        min_delta=2.0,
        lr_encoder=5.0e-3,
        lr_heads=1.0e-2,
    )
    model = _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.0)
    result = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )

    assert result.status == "trained"
    assert result.stop_reason == "early_stopping_patience"
    assert result.epochs_done == 2
    assert result.best_epoch == 1
    assert [row["is_best"] for row in result.history] == [True, False]

    best_state = torch.load(
        result.best_model_path,
        map_location="cpu",
        weights_only=True,
    )
    last_payload = checkpoint.load_last(
        result.run_directory / "last.pt",
        expected_hash=run_config.config_hash(cfg),
    )

    for name, best_tensor in best_state.items():
        assert torch.equal(model.state_dict()[name].cpu(), best_tensor)

    changed_after_best = any(
        not torch.equal(
            best_state[name].cpu(),
            last_payload["model_state_dict"][name].cpu(),
        )
        for name in best_state
    )
    assert changed_after_best, "Epoch 2 did not change weights from epoch 1."

    # Mark the run complete and verify a second call skips without overwriting.
    (result.run_directory / "run_summary.json").write_text(
        json.dumps({"status": "complete"}),
        encoding="utf-8",
    )
    skipped = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    assert skipped.skipped is True
    assert skipped.status == "complete"


_loop_check(
    "shared early stopping stops on patience and reloads best_model.pt",
    _test_early_stopping_and_best_reload,
    "check val_mean_macro_f1 improvement logic and load_best_weights at the end of fit",
)


# ---- Section: Cleanup and Final Result ----
# Restore the notebook RNG and remove temporary test artifacts.
_loop_check(
    "notebook RNG state is restored after training loop tests",
    lambda: checkpoint.restore_rng(_loop_initial_rng),
    "check capture_rng/restore_rng around the test cell",
)

_loop_tmp.cleanup()

if _loop_failures:
    print(
        f"[FAIL] {len(_loop_failures)} training loop test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL TRAINING LOOP TESTS PASSED")

[PASS] linear scheduler warms up for 6% of total steps and decays to zero
[PASS] sorted evaluation restores original id order and float32 probability columns
[run] start run=direct_two_epochs device=cpu variant=C seed=2026
[model] parameter_counts={'total': 1026, 'trainable': 1026, 'encoder': 948, 'heads': 78}
[optimizer] fused=False steps_per_epoch=3 total_steps=6 warmup_steps=0
[train] run=direct_two_epochs epoch=1/2 step=2/3 global_step=2/6 progress=33.33% objective=2.208625 lr_encoder=1.333e-04 lr_heads=6.667e-04 samples_per_second=5289.47 ETA=0.0 min
[train] run=direct_two_epochs epoch=1/2 step=3/3 global_step=3/6 progress=50.00% objective=2.223766 lr_encoder=1.000e-04 lr_heads=5.000e-04 samples_per_second=5274.60 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.0s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.213672 val_mean_macro_f1=0.157407 best_epoch=1 is_best=True bad_epochs=0/2 epoch_seconds=0.

# Inspect: Predict and Run-Summary Interfaces (Read-Only)# 

In [4]:
# Inspect: Predict and Run-Summary Interfaces (Read-Only)

import ast
from pathlib import Path

# ---- Section: Locate Project Root ----
# Read module sources directly without modifying anything on disk.
_current_dir = Path.cwd()
_project_root = next(
    (
        candidate
        for candidate in (_current_dir, *_current_dir.parents)
        if (candidate / "src" / "digikala_review_transformer").is_dir()
    ),
    None,
)

if _project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "'src/digikala_review_transformer'."
    )


# ---- Section: Print Target Definitions ----
# Inspect exact signatures and return keys used by predict.py.
_targets = [
    (
        "src/digikala_review_transformer/training/pack.py",
        {"sha256_file", "pack_fingerprint", "pack_file_name"},
    ),
    (
        "src/digikala_review_transformer/training/tokenization.py",
        {
            "HfFastBundle",
            "SentencePieceBundle",
            "load_saved_tokenizer",
            "token_stats",
        },
    ),
    (
        "src/digikala_review_transformer/training/model.py",
        {"model_summary", "save_encoder_config", "count_parameters"},
    ),
    (
        "src/digikala_review_transformer/training/run_config.py",
        {"dump_resolved_config", "config_hash", "run_dir", "project_path"},
    ),
]

for _relative_path, _selected in _targets:
    _path = _project_root / _relative_path
    _text = _path.read_text(encoding="utf-8")
    _tree = ast.parse(_text, filename=str(_path))
    print(f"\n===== {_relative_path} =====")
    for _node in _tree.body:
        if isinstance(_node, (ast.FunctionDef, ast.ClassDef)) and _node.name in _selected:
            print(ast.get_source_segment(_text, _node))
            print()


===== src/digikala_review_transformer/training/pack.py =====
def pack_file_name(split):
    return f"{split}.parquet"

def sha256_file(path, chunk_bytes=1 << 20):
    """Return the sha256 hex digest of a file, read in chunks."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def pack_fingerprint(manifest):
    """Short id of the pack contents (used in token-cache keys)."""
    joined = "".join(manifest["files"][pack_file_name(s)]["sha256"] for s in SPLITS)
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:16]


===== src/digikala_review_transformer/training/tokenization.py =====
class HfFastBundle(TokenizerBundle):
    backend = "hf_fast"
    structure = "tokenizer adds its own special tokens; truncation keeps the end token"

    def __init__(self, source_dir, max_length):
     

# Test: Predict, Probability Alignment, Run Summary and Final Cleanup

In [5]:
# Test: Predict, Probability Alignment, Run Summary and Final Cleanup

import copy
import json
import math  # noqa: F811
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import (
    checkpoint,
    data,
    loop,
    pack,
    predict,
    run_config,
    tokenization,  # noqa: F811
)
from digikala_review_transformer.training import (
    model as model_module,  # noqa: F811
)

# ---- Section: Test Helpers ----
# Keep the notebook RNG state unchanged after running prediction and finalization tests.
_predict_initial_rng = checkpoint.capture_rng()
_predict_failures = []


def _predict_check(name, test_function, fix):
    try:
        test_function()
        print(f"[PASS] {name}")
    except Exception as error:  # noqa: BLE001
        message = f"{type(error).__name__}: {error}"
        _predict_failures.append((name, message))
        print(f"[FAIL] {name}: {message} -> {fix}")


class _ToyEncoderConfig:
    def __init__(self, hidden_size=12, vocab_size=64):
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        self.model_type = "toy_encoder"

    def to_dict(self):
        return {
            "hidden_size": self.hidden_size,
            "vocab_size": self.vocab_size,
            "model_type": self.model_type,
        }


class _ToyEncoderOutput:
    def __init__(self, last_hidden_state):
        self.last_hidden_state = last_hidden_state


class _ToyEncoder(nn.Module):
    def __init__(self, vocab_size=64, hidden_size=12):
        super().__init__()
        self.config = _ToyEncoderConfig(
            hidden_size=hidden_size,
            vocab_size=vocab_size,
        )
        self.embeddings = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.projection = nn.Linear(hidden_size, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, input_ids, attention_mask=None):
        hidden = self.embeddings(input_ids)
        hidden = torch.tanh(self.projection(hidden))
        hidden = self.norm(hidden)
        return _ToyEncoderOutput(last_hidden_state=hidden)


class _ToyTokenizerBundle:
    backend = "hf_fast"
    name = "ToyBERT"
    max_length = 32
    first_id = 2
    last_id = 3
    pad_id = 0
    unk_id = 1

    def info(self):
        return {
            "name": self.name,
            "backend": self.backend,
            "max_length": self.max_length,
            "fingerprint": "toy-tokenizer-fp",
            "special_ids": {
                "first": self.first_id,
                "last": self.last_id,
                "pad": self.pad_id,
                "unk": self.unk_id,
            },
        }

    def save(self, out_dir):
        out_path = Path(out_dir)
        out_path.mkdir(parents=True, exist_ok=True)
        (out_path / "tokenizer_backend.json").write_text(
            json.dumps(self.info(), indent=2),
            encoding="utf-8",
        )
        (out_path / "vocab.txt").write_text(
            "[PAD]\n[UNK]\n[CLS]\n[SEP]\n",
            encoding="utf-8",
        )


def _build_toy_model(seed=config.SPLIT_SEED, head_dropout=0.0):
    model_module.seed_everything(seed)
    encoder = _ToyEncoder()
    model_module.seed_everything(seed)
    return model_module.TwoHeadClassifier(encoder, head_dropout=head_dropout)


def _make_token_arrays(sequences):
    offsets = [0]
    flat_ids = []
    for sequence in sequences:
        token_ids = [int(value) for value in sequence]
        flat_ids.extend(token_ids)
        offsets.append(len(flat_ids))
    return tokenization.TokenArrays(
        ids=np.asarray(flat_ids, dtype=np.int32),
        offsets=np.asarray(offsets, dtype=np.int64),
    )


def _make_split(
    split_name,
    n_rows,
    *,
    seed,
    id_start,
    variant="B",
    mask_rec_every=5,
):
    rng = np.random.default_rng(seed)
    bundle = _ToyTokenizerBundle()
    sequences = []
    rec_corr = np.empty(n_rows, dtype=np.int64)
    band_corr = np.empty(n_rows, dtype=np.int64)
    rec_raw = np.empty(n_rows, dtype=np.int64)
    band_raw = np.empty(n_rows, dtype=np.int64)

    for row_index in range(n_rows):
        rec_class = row_index % 3
        band_class = (row_index // 3 + row_index) % 3
        body_length = int((row_index * 7 + 5) % 11) + 2

        body_tokens = rng.integers(8, 48, size=body_length).tolist()
        if row_index % 6 == 0:
            body_tokens[0] = bundle.unk_id
        sequence = [
            bundle.first_id,
            4 + rec_class,
            10 + band_class,
            *body_tokens,
            bundle.last_id,
        ][: bundle.max_length]
        sequences.append(sequence)

        rec_corr[row_index] = rec_class
        band_corr[row_index] = band_class
        rec_raw[row_index] = (
            rec_class if row_index % 7 != 0 else (rec_class + 1) % 3
        )
        band_raw[row_index] = (
            band_class if row_index % 8 != 0 else (band_class + 1) % 3
        )

    if mask_rec_every is not None:
        for row_index in range(0, n_rows, mask_rec_every):
            rec_corr[row_index] = data.IGNORE_INDEX
            rec_raw[row_index] = data.IGNORE_INDEX

    if variant == "C":
        w_rec = np.linspace(0.7, 1.0, n_rows, dtype=np.float32)
        w_band = np.linspace(1.0, 0.8, n_rows, dtype=np.float32)
    else:
        w_rec = np.ones(n_rows, dtype=np.float32)
        w_band = np.ones(n_rows, dtype=np.float32)

    arrays = _make_token_arrays(sequences)
    stats = tokenization.token_stats(arrays, bundle)
    # Give splits non-monotonic, realistic ids so order bugs are caught immediately.
    ids = np.asarray(
        [id_start + ((index * 37 + 13) % 997) + index * 1000 for index in range(n_rows)],
        dtype=np.int64,
    )

    return data.SplitData(
        split=split_name,
        ids=ids,
        arrays=arrays,
        pad_id=bundle.pad_id,
        y={"rec": rec_corr, "band": band_corr},
        y_raw={"rec": rec_raw, "band": band_raw},
        w={"rec": w_rec, "band": w_band},
        stats=stats,
        origin="test",
        source_rows=n_rows,
    )


def _make_manifest():
    return {
        "format_version": 1,
        "files": {
            "train.parquet": {
                "sha256": "a" * 64,
                "rows": 24,
                "size_mb": 0.1,
            },
            "validation.parquet": {
                "sha256": "b" * 64,
                "rows": 18,
                "size_mb": 0.05,
            },
            "test.parquet": {
                "sha256": "c" * 64,
                "rows": 21,
                "size_mb": 0.06,
            },
        },
    }


def _make_config(output_root, *, run_name="predict_test_run"):
    return {
        "run_name": run_name,
        "seed": config.SPLIT_SEED,
        "data": {"pack_dir": "data/training_pack"},
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 32,
            "head_dropout": 0.0,
            "tokenizer_backend": "hf_fast",
        },
        "training": {
            "variant": "C",
            "max_epochs": 2,
            "early_stopping_patience": 1,
            "early_stopping_min_delta": 0.0,
            "batch_size": 8,
            "eval_batch_size": 6,
            "lr_encoder": 2.0e-4,
            "lr_heads": 1.0e-3,
            "weight_decay": 0.01,
            "warmup_ratio": 0.06,
            "max_grad_norm": 1.0,
            "label_smoothing": 0.1,
            "loss_weight_band": 1.0,
            "use_class_weights": True,
            "bf16": False,
            "num_workers": 0,
            "log_every_steps": 2,
            "train_subset_eval_rows": 12,
        },
        "output": {"root": str(output_root)},
    }


def _direct_probabilities(model, split):
    model.eval()
    batch = split.dataset()[np.arange(split.n_rows, dtype=np.int64)]
    with torch.inference_mode():
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        return {
            head: torch.softmax(logits[head].float(), dim=-1).cpu().numpy()
            for head in data.HEADS
        }


_predict_tmp = tempfile.TemporaryDirectory(prefix="transformer_predict_test_")
_predict_root = Path(_predict_tmp.name)


# ---- Section: Probability Schema, Order Restoration, and Corruption Guards ----
# Verify length-sorted prediction writes exact id order and rejects corrupted tables.
def _test_predict_split_and_verify_guards():
    test_split = _make_split("test", 21, seed=17, id_start=9000)
    model = _build_toy_model(seed=31)
    parquet_path = _predict_root / "standalone" / "test_probs.parquet"

    frame, probabilities = predict.predict_split_to_parquet(
        model,
        test_split,
        parquet_path,
        batch_size=4,
        device="cpu",
        bf16=False,
    )
    expected = _direct_probabilities(model, test_split)

    assert parquet_path.is_file()
    assert list(frame.columns) == [
        "id",
        "p_recommended",
        "p_not_recommended",
        "p_no_idea",
        "p_low",
        "p_medium",
        "p_high",
    ]
    assert np.array_equal(frame["id"].to_numpy(), test_split.ids)

    for head in data.HEADS:
        assert np.allclose(
            probabilities[head],
            expected[head],
            rtol=1.0e-5,
            atol=1.0e-6,
        )

    reloaded_frame, _ = predict.read_and_verify_probability_parquet(
        parquet_path,
        test_split,
    )
    for column_name in predict.expected_probability_columns()[1:]:
        assert reloaded_frame[column_name].dtype == np.float32

    swapped = reloaded_frame.copy()
    swapped.iloc[[0, 1]] = swapped.iloc[[1, 0]].to_numpy()
    try:
        predict.verify_probability_frame(swapped, test_split)
    except predict.PredictionError as error:
        assert "aligned" in str(error).lower()
    else:
        raise AssertionError("Misaligned ids were accepted.")

    wrong_dtype = reloaded_frame.copy()
    wrong_dtype["p_low"] = wrong_dtype["p_low"].astype(np.float64)
    try:
        predict.verify_probability_frame(wrong_dtype, test_split)
    except predict.PredictionError as error:
        assert "float32" in str(error).lower()
    else:
        raise AssertionError("float64 probability column was accepted.")

    bad_sum = reloaded_frame.copy()
    bad_sum.loc[0, "p_low"] = np.float32(0.95)
    bad_sum.loc[0, "p_medium"] = np.float32(0.95)
    try:
        predict.verify_probability_frame(bad_sum, test_split)
    except predict.PredictionError as error:
        assert "sum to 1" in str(error).lower()
    else:
        raise AssertionError("Probabilities not summing to 1 were accepted.")


_predict_check(
    "predict_split_to_parquet restores id order and rejects misaligned/bad probability tables",
    _test_predict_split_and_verify_guards,
    "check predict_split_to_parquet and verify_probability_frame",
)


# ---- Section: Both-View Quick Metrics and Incomplete Run Guard ----
# Both corrected and raw views must be reported; paused runs must not be finalized.
def _test_quick_metrics_and_paused_guard():
    train_split = _make_split("train", 24, seed=101, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=202, id_start=1000)
    test_split = _make_split("test", 21, seed=303, id_start=2000)
    subset_split = _make_split("train_subset", 12, seed=404, id_start=3000)

    paused_cfg = _make_config(
        _predict_root / "paused_guard",
        run_name="paused_run",
    )
    paused_model = _build_toy_model(seed=config.SPLIT_SEED)
    paused_loop = loop.fit(
        paused_cfg,
        paused_model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
        max_epochs_this_call=1,
    )

    try:
        predict.finalize_run(
            paused_cfg,
            paused_model,
            _ToyTokenizerBundle(),
            paused_loop,
            train_split=train_split,
            validation_split=val_split,
            test_split=test_split,
            train_subset_split=subset_split,
            manifest=_make_manifest(),
            load_info={"unexpected_keys": [], "missing_keys": []},
            device="cpu",
        )
    except predict.PredictionError as error:
        assert "paused" in str(error).lower()
    else:
        raise AssertionError("A paused run was allowed to finalize.")

    assert (paused_loop.run_directory / "last.pt").is_file()


_predict_check(
    "finalize_run refuses paused runs and leaves last.pt intact",
    _test_quick_metrics_and_paused_guard,
    "check finalize_run status validation before writing run_summary.json",
)


# ---- Section: Full End-to-End Finalize, Best Weight Reload, Summary, and Cleanup ----
# Corrupt in-memory weights before finalize_run to prove best_model.pt is reloaded.
def _test_full_finalize_and_idempotent_rerun():
    train_split = _make_split("train", 24, seed=501, id_start=100, variant="C")
    val_split = _make_split("validation", 18, seed=502, id_start=1000)
    test_split = _make_split("test", 21, seed=503, id_start=2000)
    subset_split = _make_split("train_subset", 12, seed=504, id_start=3000)
    manifest = _make_manifest()
    load_info = {
        "unexpected_keys": ["cls.predictions.bias"],
        "missing_keys": ["pooler.dense.weight", "pooler.dense.bias"],
        "mismatched_keys": [],
        "error_msgs": [],
    }

    cfg = _make_config(
        _predict_root / "complete_flow",
        run_name="tooka_toy_C",
    )
    model = _build_toy_model(seed=config.SPLIT_SEED)
    loop_result = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    run_directory = loop_result.run_directory
    assert (run_directory / "last.pt").is_file()

    reference_model = _build_toy_model(seed=777)
    checkpoint.load_best_weights(run_directory / "best_model.pt", reference_model)
    expected_test_probs = _direct_probabilities(reference_model, test_split)

    # Deliberately corrupt the in-memory model before finalization.
    with torch.no_grad():
        for parameter in model.parameters():
            parameter.fill_(0.0)

    finalized = predict.finalize_run(
        cfg,
        model,
        _ToyTokenizerBundle(),
        loop_result,
        train_split=train_split,
        validation_split=val_split,
        test_split=test_split,
        train_subset_split=subset_split,
        manifest=manifest,
        load_info=load_info,
        device="cpu",
    )

    assert not (run_directory / "last.pt").exists(), "last.pt was not deleted."
    assert checkpoint.run_status(run_directory) == "complete"

    required_files = [
        "config_resolved.yaml",
        "history.csv",
        "train_log.csv",
        "train.log",
        "run_summary.json",
        "val_probs_best.parquet",
        "test_probs.parquet",
        "best_model.pt",
        "encoder_config.json",
        "tokenizer/tokenizer_backend.json",
        "tokenizer/vocab.txt",
    ]
    for relative_path in required_files:
        assert (run_directory / relative_path).is_file(), (
            f"Missing required artifact: {relative_path}"
        )

    _, saved_test_probs = predict.read_and_verify_probability_parquet(
        finalized.test_probabilities_path,
        test_split,
    )
    for head in data.HEADS:
        assert np.allclose(
            saved_test_probs[head],
            expected_test_probs[head],
            rtol=1.0e-5,
            atol=1.0e-6,
        ), f"Test predictions for {head} did not use reloaded best_model.pt."

    summary = json.loads(
        (run_directory / "run_summary.json").read_text(encoding="utf-8")
    )
    assert summary["status"] == "complete"
    assert summary["run_name"] == "tooka_toy_C"
    assert summary["seed"] == config.SPLIT_SEED
    assert summary["variant"] == "C"
    assert summary["config_hash"] == run_config.config_hash(cfg)
    assert summary["tokenizer_backend"] == "hf_fast"
    assert summary["encoder_load_info"] == load_info
    assert summary["selection_and_stopping"]["test_used_for_any_decision"] is False
    assert summary["selection_and_stopping"]["labels_used_for_selection"] == (
        "step1_corrected_only"
    )
    assert summary["selection_and_stopping"]["best_epoch"] == loop_result.best_epoch

    for split_name in ("train", "validation", "test", "train_subset"):
        assert "unk_pct_of_tokens" in summary["token_stats"][split_name]
        assert "rows_at_max_length_pct" in summary["token_stats"][split_name]

    for block_name in ("validation_best", "test_quick"):
        block = summary[block_name]
        for view in ("corrected", "raw"):
            assert view in block
            assert math.isfinite(block[view]["mean_macro_f1"])
            for head in data.HEADS:
                assert math.isfinite(block[view][head]["macro_f1"])
                assert math.isfinite(block[view][head]["accuracy"])
                assert math.isfinite(block[view][head]["balanced_accuracy"])
            assert "mean_abs_distance" in block[view]["band"]
            assert "neighbor_error_pct" in block[view]["band"]
            assert "extreme_error_pct" in block[view]["band"]

    assert summary["pack"]["fingerprint"] == pack.pack_fingerprint(manifest)
    for relative_name, recorded_hash in summary["artifact_sha256"].items():
        actual_hash = pack.sha256_file(run_directory / relative_name)
        assert recorded_hash == actual_hash, (
            f"SHA-256 mismatch for {relative_name}"
        )

    # Simulate a second call on an already completed run.
    skipped_loop = loop.fit(
        cfg,
        model,
        train_split,
        val_split,
        subset_split,
        device="cpu",
    )
    assert skipped_loop.skipped is True
    finalized_again = predict.finalize_run(
        cfg,
        model,
        _ToyTokenizerBundle(),
        skipped_loop,
        train_split=train_split,
        validation_split=val_split,
        test_split=test_split,
        train_subset_split=subset_split,
        manifest=manifest,
        load_info=load_info,
        device="cpu",
    )
    assert finalized_again.summary["config_hash"] == summary["config_hash"]


_predict_check(
    "finalize_run reloads best_model.pt, writes complete artifacts/hashes, and deletes last.pt",
    _test_full_finalize_and_idempotent_rerun,
    "check finalize_run artifact creation, test_quick calculation, and cleanup_after_complete",
)


# ---- Section: Cleanup and Final Result ----
# Restore notebook RNG state and clean temporary files.
_predict_check(
    "notebook RNG state is restored after predict tests",
    lambda: checkpoint.restore_rng(_predict_initial_rng),
    "check capture_rng/restore_rng around the test cell",
)

_predict_tmp.cleanup()

if _predict_failures:
    print(
        f"[FAIL] {len(_predict_failures)} predict test(s) failed; "
        "see the failure details above and apply the listed fixes."
    )
else:
    print("[PASS] ALL PREDICT TESTS PASSED")

[PASS] predict_split_to_parquet restores id order and rejects misaligned/bad probability tables
[run] start run=paused_run device=cpu variant=C seed=2026
[model] parameter_counts={'total': 1026, 'trainable': 1026, 'encoder': 948, 'heads': 78}
[optimizer] fused=False steps_per_epoch=3 total_steps=6 warmup_steps=0
[train] run=paused_run epoch=1/2 step=2/3 global_step=2/6 progress=33.33% objective=2.222551 lr_encoder=1.333e-04 lr_heads=6.667e-04 samples_per_second=4585.09 ETA=0.0 min
[train] run=paused_run epoch=1/2 step=3/3 global_step=3/6 progress=50.00% objective=2.179136 lr_encoder=1.000e-04 lr_heads=5.000e-04 samples_per_second=4271.79 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.0s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.208079 val_mean_macro_f1=0.157407 best_epoch=1 is_best=True bad_epochs=0/1 epoch_seconds=0.0
[run] training_loop_status=paused stop_reason=paused_by_call_limit epochs_done=1 

# Cell Title: Inspect: Pack Manifest and Signatures Used by Smoke (Read-Only)

In [2]:
import inspect

from digikala_review_transformer import config
from digikala_review_transformer.training import data, losses, model, pack, tokenization  # noqa: F811

opened = data.open_pack(config.ROOT / "data" / "training_pack")
print("open_pack type & len:", type(opened), len(opened) if hasattr(opened, "__len__") else None)
if isinstance(opened, tuple):
    for i, item in enumerate(opened):
        print(f"  tuple[{i}]:", type(item), item if not isinstance(item, dict) else list(item.keys()))
elif isinstance(opened, dict):
    print("  dict keys:", list(opened.keys()))

for fn in (
    pack.verify_pack,
    tokenization.load_tokenizer,
    tokenization.get_split_tokens,
    data.load_split,
    data.subset_row_indices,
    data.LengthBucketedBatches,
    data.batch_fingerprint,
    model.build_model,
    model.check_token_ids_fit,
    model.count_parameters,
    model.parameter_groups,
    losses.build_loss,
):
    print(f"{fn.__module__.split('.')[-1]}.{fn.__name__}{inspect.signature(fn)}")

open_pack type & len: <class 'dict'> 7
  dict keys: ['format_version', 'code', 'label_codes', 'label_code_note', 'weights', 'files', 'sources']
pack.verify_pack(pack_dir, check_content=False)
tokenization.load_tokenizer(model_path, backend, max_length)
tokenization.get_split_tokens(bundle, split, source_sha256, load_texts, expected_rows, cache_dir=None, show_progress=True)
data.load_split(split, bundle, pack_dir, manifest, variant, subset_rows=None, subset_seed=2026, cache_dir=None, show_progress=True)
data.subset_row_indices(n_rows, size, seed, purpose)
data.LengthBucketedBatches(lengths, batch_size, seed, chunk_batches=50)
data.batch_fingerprint(batches, n_batches=5)
model.build_model(cfg)
model.check_token_ids_fit(model, max_token_id, pad_id)
model.count_parameters(model)
model.parameter_groups(model, lr_encoder, lr_heads, weight_decay)
losses.build_loss(cfg, y_train, device)


# Cell Title: Inspect: Manifest Files and Object Attributes for Smoke (Read-Only)

In [3]:
from digikala_review_transformer import config
from digikala_review_transformer.training import data, pack, tokenization

pack_dir = config.ROOT / "data" / "training_pack"
manifest = data.open_pack(pack_dir)
print("manifest['files']:", manifest["files"])

vp = pack.verify_pack(pack_dir, check_content=False)
print("verify_pack return:", type(vp), vp[:2] if isinstance(vp, list) else vp)

print("TokenArrays attrs:", [a for a in dir(tokenization.TokenArrays) if not a.startswith("_")])
print("SplitData attrs:", [a for a in dir(data.SplitData) if not a.startswith("_")])
print("LengthBucketedBatches attrs:", [a for a in dir(data.LengthBucketedBatches) if not a.startswith("_")])

manifest['files']: {'train.parquet': {'rows': 2860236, 'label_missing': {'y_rec': 468079, 'y_band': 0, 'y_rec_raw': 468079, 'y_band_raw': 0}, 'blank_body_rows': 0, 'columns': ['id', 'body', 'y_rec', 'y_band', 'y_rec_raw', 'y_band_raw', 'w_rec', 'w_band'], 'sha256': 'b770d123b261dc7526f3d003544029c60c95c4de3a639922a32dece275b26b4c', 'size_mb': 119.3}, 'validation.parquet': {'rows': 922794, 'label_missing': {'y_rec': 134117, 'y_band': 0, 'y_rec_raw': 134117, 'y_band_raw': 0}, 'blank_body_rows': 0, 'columns': ['id', 'body', 'y_rec', 'y_band', 'y_rec_raw', 'y_band_raw'], 'sha256': '2ff8bcf518abf62d1179449c565da322b94fb9accc38c9f27e8643770e6ebe30', 'size_mb': 29.5}, 'test.parquet': {'rows': 922805, 'label_missing': {'y_rec': 134960, 'y_band': 1, 'y_rec_raw': 134960, 'y_band_raw': 1}, 'blank_body_rows': 0, 'columns': ['id', 'body', 'y_rec', 'y_band', 'y_rec_raw', 'y_band_raw'], 'sha256': '6eee541ad46f1003fe6f8634ceddccce7399d6ce6e27afa77fce92bcd0084902', 'size_mb': 29.5}}
verify_pack return:

# Cell Title: Inspect: Remaining Interfaces Used in Smoke (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import data, losses, tokenization

print("SentencePieceBundle attrs:", [a for a in dir(tokenization.SentencePieceBundle) if not a.startswith("_")])
print("encode_texts sig:", inspect.signature(tokenization.encode_texts))
print("LengthBucketedBatches source:\n", inspect.getsource(data.LengthBucketedBatches))
print("batch_fingerprint source:\n", inspect.getsource(data.batch_fingerprint))
print("SplitData annotations:", data.SplitData.__annotations__)
# TwoHeadLoss may not define a `forward` method; inspect the class itself and
# fall back to `__call__` if needed.
print("TwoHeadLoss source:\n", inspect.getsource(losses.TwoHeadLoss))

if hasattr(losses.TwoHeadLoss, "forward"):
    print("TwoHeadLoss.forward source:\n", inspect.getsource(losses.TwoHeadLoss.forward))
else:
    print("TwoHeadLoss has no `forward` method; checking `__call__` instead.")
    if callable(losses.TwoHeadLoss):
        try:
            print("TwoHeadLoss.__call__ source:\n", inspect.getsource(losses.TwoHeadLoss.__call__))
        except (OSError, TypeError):
            print("Could not inspect __call__ source.")

SentencePieceBundle attrs: ['backend', 'encode', 'info', 'save', 'structure']
encode_texts sig: (bundle, texts, show_progress=False)
LengthBucketedBatches source:
 class LengthBucketedBatches:
    """Training batches. The order depends only on (seed, epoch, lengths), never on the variant."""

    def __init__(self, lengths, batch_size, seed, chunk_batches=CHUNK_BATCHES):
        self.lengths = np.asarray(lengths)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.chunk_batches = int(chunk_batches)
        self.epoch = 0
        if self.batch_size < 1 or self.chunk_batches < 1:
            raise DataError("batch_size and chunk_batches must be at least 1")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)

    def batches_for_epoch(self, epoch):
        """Shuffle rows, sort each chunk by length, cut into batches, shuffle batch order."""
        rng = np.

# Cell Title: Inspect: TokenizerBundle Attributes and Info (Read-Only)

In [4]:
from digikala_review_transformer import config
from digikala_review_transformer.training import tokenization

sp_b = tokenization.load_tokenizer(config.ROOT / "models/pretrained/mdeberta-v3-base", "sentencepiece", 160)
hf_b = tokenization.load_tokenizer(config.ROOT / "models/pretrained/TookaBERT-Base", "hf_fast", 128)
print("sp vars:", {k: v for k, v in vars(sp_b).items() if k != "sp"}, "info:", sp_b.info())
print("hf vars:", {k: v for k, v in vars(hf_b).items() if k != "tokenizer"}, "info:", hf_b.info())

sp vars: {'source_dir': PosixPath('/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base'), 'max_length': 160, 'name': 'mdeberta-v3-base', 'fingerprint': 'ceae13c4186e4ca6be9c90ba02c3bbe5aba12f7c224a313a68631703354bb1b5', 'first_id': 1, 'last_id': 2, 'pad_id': 0, 'unk_id': 3, 'processor': <sentencepiece.SentencePieceProcessor object at 0x76dbed3f43b0>} info: {'backend': 'sentencepiece', 'source_model': 'mdeberta-v3-base', 'max_length': 160, 'special_ids': {'first': 1, 'last': 2, 'pad': 0, 'unk': 3}, 'structure': '[CLS] + ids[:max_length-2] + [SEP]; attention mask is all ones', 'fingerprint': 'ceae13c4186e4ca6be9c90ba02c3bbe5aba12f7c224a313a68631703354bb1b5'}
hf vars: {'source_dir': PosixPath('/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base'), 'max_length': 128, 'name': 'TookaBERT-Base', 'fingerprint': 'd1a1da1f5d72c29284a049412da210f8a4c0640dcf8acb67574e8786676d

# Cell Title: Inspect: Failed Check in Smoke Report (Read-Only)

In [2]:
for c in report.checks:
    print(smoke.format_check_line(c))

[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=15.0 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)
[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[FAIL] 4. Temporary token cache build & re-read: ValueError: too many values to unpack (expected 2) -> check write permissions in system temp directory and tokenization.py
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=dbe9821e8f5f3922, ep1=5325511e61ab9ce3), max_token_id=46517 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(16, 64), loss=2.6952, peak_vram=2356.8 MB (2.30 GB)
[PASS] 7. 2-epoch training & checkpoint auto-resume: resumed epoch 1 -> 2 (steps=20), train_objective=(2.2685 -> 2.152

# Cell Title: Inspect: Return Signatures of get_split_tokens and read_and_verify_probability_parquet (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import predict, tokenization  # noqa: F811

print("get_split_tokens:\n", inspect.getsource(tokenization.get_split_tokens))
print("read_and_verify_probability_parquet:\n", inspect.getsource(predict.read_and_verify_probability_parquet))

get_split_tokens:
 def get_split_tokens(bundle, split, source_sha256, load_texts, expected_rows, cache_dir=None, show_progress=True):
    """Return (TokenArrays, stats, origin); origin is 'cache' or 'built'.

    load_texts is only called when the cache cannot be used. cache_dir=None means no disk cache.
    """
    key = cache_key(bundle, source_sha256)
    if cache_dir is not None:
        arrays, stats, reason = _read_cache(bundle, split, key, expected_rows, cache_dir)
        if arrays is not None:
            return arrays, stats, "cache"
        if reason != "no cache entry":
            print(f"[WARN] {split}: {reason} -> rebuilding")

    with utils.timer(f"tokenize_{bundle.name}_{split}"):
        arrays = encode_texts(bundle, load_texts(), show_progress=show_progress)
    if len(arrays) != expected_rows:
        raise TokenizationError(f"{split}: got {len(arrays):,} rows, expected {expected_rows:,} -> texts and manifest differ")
    validate_arrays(arrays, bundle)
    stats =

# Cell Title: Test: Smoke Runner and mDeBERTa Tokenizer Checks

In [1]:
import copy
import importlib
import tempfile
from pathlib import Path

import pyarrow.parquet as pq
import torch

from digikala_review_transformer import config
from digikala_review_transformer.training import (
    checkpoint,
    data,
    pack,
    predict,
    run_config,
    smoke,
    tokenization,
)

importlib.reload(smoke)

results: list[tuple[str, bool, str]] = []


def record(name: str, cond: bool, detail: str = "") -> None:
    results.append((name, bool(cond), detail))
    status = "[PASS]" if cond else "[FAIL]"
    suffix = f" ({detail})" if detail else ""
    print(f"{status} {name}{suffix}")


# ---- 1. Formatting and Failure Guard ----
pass_line = smoke.format_check_line(
    smoke.SmokeCheckResult(name="sample", passed=True, detail="ok")
)
fail_line = smoke.format_check_line(
    smoke.SmokeCheckResult(name="sample", passed=False, detail="bad", fix="do x")
)
record(
    "format_check_line follows [PASS]/[FAIL] convention",
    pass_line.startswith("[PASS]") and fail_line == "[FAIL] sample: bad -> do x",
    fail_line,
)

base_cfg = run_config.load_run_config(config.ROOT / "configs" / "transformer" / "tooka_B.yaml")

with tempfile.TemporaryDirectory(prefix="dk_smoke_guard_") as guard_dir:
    guard_report = smoke.run_smoke(
        base_cfg,
        smoke_directory=Path(guard_dir) / "smoke",
        train_rows=32,
        val_rows=16,
        test_rows=16,
        monitor_rows=16,
        num_workers=0,
        require_cuda=False,
        min_free_disk_gb=1e9,
    )
    record(
        "Disk space guard halts cleanly at Check 1 when threshold is unmet",
        (not guard_report.passed)
        and len(guard_report.checks) == 1
        and (not guard_report.checks[0].passed),
        guard_report.checks[0].detail,
    )

# ---- 2. mDeBERTa 3 Extra Tokenizer Checks ----
mdeb_path = config.ROOT / "models" / "pretrained" / "mdeberta-v3-base"
mdeb_bundle = tokenization.load_tokenizer(
    mdeb_path,
    backend="sentencepiece",
    max_length=160,
)
pack_dir = run_config.project_path(base_cfg["data"]["pack_dir"])
manifest = data.open_pack(pack_dir)
val_total_rows = int(manifest["files"][pack.pack_file_name("validation")]["rows"])
sample_idx = data.subset_row_indices(val_total_rows, 500, config.SPLIT_SEED, purpose="smoke")
val_table = pq.read_table(pack_dir / pack.pack_file_name("validation"), columns=["body"]).take(sample_idx)
sample_texts = [t if t is not None else "" for t in val_table.column("body").to_pylist()]

mdeb_checks = smoke.run_mdeberta_tokenizer_checks(mdeb_path, mdeb_bundle, sample_texts)
record(
    "mDeBERTa check 1 (special token IDs match HF)",
    len(mdeb_checks) == 3 and mdeb_checks[0].passed,
    mdeb_checks[0].detail if mdeb_checks else "missing",
)
record(
    "mDeBERTa check 2 (100% parity with HF fast on non-UNK rows)",
    len(mdeb_checks) == 3 and mdeb_checks[1].passed,
    mdeb_checks[1].detail if len(mdeb_checks) > 1 else "missing",
)
record(
    "mDeBERTa check 3 ([UNK] row count reported as warning)",
    len(mdeb_checks) == 3
    and mdeb_checks[2].passed
    and len(mdeb_checks[2].warnings) == 1
    and "sentencepiece=" in mdeb_checks[2].warnings[0]
    and "hf_fast=" in mdeb_checks[2].warnings[0],
    mdeb_checks[2].warnings[0] if (len(mdeb_checks) > 2 and mdeb_checks[2].warnings) else "missing",
)

# ---- 3. Full 9-Step End-to-End Smoke Run (TookaBERT in Temp Directory) ----
test_cfg = copy.deepcopy(base_cfg)
test_cfg["run_name"] = "smoke_unit_test"
test_cfg["model"]["max_length"] = 64
test_cfg["training"]["batch_size"] = 16
test_cfg["training"]["eval_batch_size"] = 32

with tempfile.TemporaryDirectory(prefix="dk_smoke_e2e_") as tmp_root:
    smoke_out_dir = Path(tmp_root) / "smoke"
    report = smoke.run_smoke(
        test_cfg,
        smoke_directory=smoke_out_dir,
        train_rows=160,
        val_rows=80,
        test_rows=80,
        monitor_rows=64,
        num_workers=0,
        require_cuda=torch.cuda.is_available(),
    )

    record(
        "End-to-end smoke report passed all 9 checks",
        report.passed and len(report.checks) == 9 and all(c.passed for c in report.checks),
        f"checks={len(report.checks)}, warnings={len(report.warnings)}",
    )
    record(
        "Worst-case batch recorded peak memory on CUDA",
        (not torch.cuda.is_available()) or (report.peak_memory_mb > 0.0),
        f"peak_memory_mb={report.peak_memory_mb:.1f}",
    )
    record(
        "Throughput and full-epoch wall-time estimate computed",
        report.train_samples_per_sec > 0.0
        and report.eval_samples_per_sec > 0.0
        and report.estimated_full_epoch_minutes > 0.0
        and report.estimated_full_run_minutes > 0.0,
        f"train_sps={report.train_samples_per_sec:.1f}, "
        f"est_epoch={report.estimated_full_epoch_minutes:.1f} min",
    )
    record(
        "Smoke directory finalized to status='complete' and last.pt cleaned up",
        checkpoint.run_status(smoke_out_dir) == "complete"
        and not (smoke_out_dir / predict.LAST_CHECKPOINT_NAME).exists()
        and (smoke_out_dir / predict.RUN_SUMMARY_NAME).exists()
        and (smoke_out_dir / predict.VAL_PROBS_NAME).exists()
        and (smoke_out_dir / predict.TEST_PROBS_NAME).exists()
        and (smoke_out_dir / predict.BEST_MODEL_NAME).exists(),
        f"status={checkpoint.run_status(smoke_out_dir)}",
    )

failed = [name for name, ok, _ in results if not ok]
print("-" * 60)
if failed:
    raise AssertionError(f"Failed checks ({len(failed)}): {failed}")
print(f"ALL PASSED ({len(results)}/{len(results)} checks)")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] format_check_line follows [PASS]/[FAIL] convention ([FAIL] sample: bad -> do x)
STARTING SMOKE TEST: run_name=tooka_B, model=TookaBERT-Base, variant=B, batch_size=128, max_length=128
Smoke output directory: /tmp/dk_smoke_guard_ua7f0lhd/smoke
[FAIL] 1. Environment & disk: free disk space is 15.00 GB (< 1000000000.00 GB required) -> free up disk space on the drive hosting the repository
------------------------------------------------------------------------
SMOKE RESULT (tooka_B): 1 FAILED out of 1 checks
  - [FAIL] 1. Environment & disk: free disk space is 15.00 GB (< 1000000000.00 GB required) -> free up disk space on the drive hosting the repository
------------------------------------------------------------------------
[PASS] Disk space guard halts cleanly at Check 1 when threshold is unmet (free disk space is 15.00 GB (< 1000000000.00 GB required))


[transformers] The tokenizer you are loading from '/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[PASS] mDeBERTa check 1 (special token IDs match HF) (SentencePiece and HF special IDs match {'first_id': 1, 'last_id': 2, 'pad_id': 0, 'unk_id': 3})
[PASS] mDeBERTa check 2 (100% parity with HF fast on non-UNK rows) (100% identical across all 485 non-[UNK] rows (out of 500 sampled rows))
[PASS] mDeBERTa check 3 ([UNK] row count reported as warning) (rows with [UNK] on 500 sampled texts: sentencepiece=0 (0.00%), hf_fast=15 (3.00%))
STARTING SMOKE TEST: run_name=smoke_unit_test, model=TookaBERT-Base, variant=B, batch_size=16, max_length=64
Smoke output directory: /tmp/dk_smoke_e2e_wdufi0u8/smoke
[PASS] 1. Environment & CUDA: os=Linux 7.0.0-42-generic, python=3.12.16, torch=2.15.0+cu132, gpu='NVIDIA GeForce RTX 4060 Laptop GPU' (7.62 GB), bf16=True, free_disk=15.0 GB
[PASS] 2. Training pack integrity: fingerprint=9faa2a97e4502701, rows(train=2,860,236, val=922,794, test=922,805)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12481.16it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] 4. Temporary token cache build & re-read: origin=(built -> cache), rows=64, tokens=928
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[timer] tokenize_TookaBERT-Base_test: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=dbe9821e8f5f3922, ep1=5325511e61ab9ce3), max_token_id=46517 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(16, 64), loss=2.6952, peak_vram=2356.8 MB (2.30 GB)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14695.13it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=smoke_unit_test device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=10 total_steps=20 warmup_steps=1
[train] run=smoke_unit_test epoch=1/2 step=5/10 global_step=5/20 progress=25.00% objective=2.345933 lr_encoder=1.579e-05 lr_heads=7.895e-05 samples_per_second=384.30 ETA=0.0 min
[train] run=smoke_unit_test epoch=1/2 step=10/10 global_step=10/20 progress=50.00% objective=2.190993 lr_encoder=1.053e-05 lr_heads=5.263e-05 samples_per_second=394.51 ETA=0.0 min
[timer] transformer_train_epoch_1: 0.4s
[timer] transformer_validation_epoch_1: 0.0s
[timer] transformer_train_subset_epoch_1: 0.0s
[epoch] epoch=1/2 train_objective=2.268463 val_mean_macro_f1=0.222325 best_epoch=1 is_best=True bad_epochs=0/2 epoch_seconds=0.5
[run] training_loop_status=paused stop_reason=paused_by_call_limit epochs_done=1 global_step=10 best_epoch=1 best_score=0.222325 train_s

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11196.48it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=smoke_unit_test device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[resume] Rebuilding best_model.pt and val_probs_best.parquet from the committed last.pt weights.
[resume] restored epochs_done=1 global_step=10 best_epoch=1 best_score=0.222325
[optimizer] fused=True steps_per_epoch=10 total_steps=20 warmup_steps=1
[train] run=smoke_unit_test epoch=2/2 step=5/10 global_step=15/20 progress=75.00% objective=2.175960 lr_encoder=5.263e-06 lr_heads=2.632e-05 samples_per_second=416.57 ETA=0.0 min
[train] run=smoke_unit_test epoch=2/2 step=10/10 global_step=20/20 progress=100.00% objective=2.128339 lr_encoder=0.000e+00 lr_heads=0.000e+00 samples_per_second=412.76 ETA=0.0 min
[timer] transformer_train_epoch_2: 0.4s
[timer] transformer_validation_epoch_2: 0.0s
[timer] transformer_train_subset_epoch_2: 0.0s
[epoch] epoch=2/2 train_objective=2.152149 val_mean_macro_f1=0.248496 best_epoch=2 is_

# Cell Title: Inspect: Failed Check Detail in Smoke Report (Read-Only)

In [2]:
for c in report.checks:
    if not c.passed:
        print(smoke.format_check_line(c))

[FAIL] 8. Predictions & run finalization: val_ids_match=False, test_ids_match=False, status=complete, last_removed=True -> inspect predict.finalize_run and sorted_eval_batches restoration


# Cell Title: Inspect: Split IDs Dtype and Verification Logic (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import predict

print("verify_probability_frame source:\n", inspect.getsource(predict.verify_probability_frame))

verify_probability_frame source:
 def verify_probability_frame(
    frame: pd.DataFrame,
    split: data.SplitData,
    *,
    source_name: str = "probability frame",
) -> dict[str, np.ndarray]:
    """Validate id alignment, float32 dtypes, and row sums; return probabilities."""
    if not isinstance(frame, pd.DataFrame):
        raise PredictionError(f"{source_name} is not a pandas DataFrame.")

    expected_columns = expected_probability_columns()
    actual_columns = list(frame.columns)
    if actual_columns != expected_columns:
        raise PredictionError(
            f"{source_name} has columns {actual_columns}; "
            f"expected {expected_columns}."
        )

    expected_ids = np.asarray(split.ids)
    actual_ids = frame["id"].to_numpy()
    if len(actual_ids) != len(expected_ids):
        raise PredictionError(
            f"{source_name} has {len(actual_ids)} rows; "
            f"expected {len(expected_ids)} rows for split {split.split!r}."
        )
    if not np.a

# Cell Title: Inspect: data.load_split and run_config for CLI Script (Read-Only)

In [3]:
import inspect

from digikala_review_transformer.training import data, run_config, tokenization

print("data.py source:\n", inspect.getsource(data))
print("=" * 60)
print("run_config.py source:\n", inspect.getsource(run_config))
print("=" * 60)
print("default_cache_dir source:\n", inspect.getsource(tokenization.default_cache_dir))

data.py source:
 """Data layer: pack loading, token arrays + labels + weights, batching, loaders.

Batches are built inside Dataset.__getitem__ (it receives an array of row indices), so the
DataLoader runs with batch_size=None and a sampler that yields index arrays.
Light imports only: this module also runs on the training machine.
"""

# ---- Section: Imports ----
import hashlib
import math
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

from digikala_review_transformer import config, labels
from digikala_review_transformer.training import pack, tokenization
from digikala_review_transformer.training.run_config import VARIANTS

# ---- Section: Constants ----
HEADS = pack.HEADS
IGNORE_INDEX = -100
CHUNK_BATCHES = 50  # plan: chunks of batch_size * 50 rows are sorted by length
PREFETCH_FACTOR = 2
# Separate random streams so different subsets never coincide.
SUBSET_STREAMS = 

# Cell Title: Test: CLI Script train_transformer.py

In [4]:
import importlib.util
import tempfile
from pathlib import Path

import numpy as np
import torch
import yaml

from digikala_review_transformer import config
from digikala_review_transformer.training import (
    checkpoint,
    data,
    predict,
    run_config,
    tokenization,
)

script_path = config.ROOT / "scripts" / "train_transformer.py"
spec = importlib.util.spec_from_file_location("train_transformer_cli", script_path)
cli = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cli)

results: list[tuple[str, bool, str]] = []


def record(name: str, cond: bool, detail: str = "") -> None:
    results.append((name, bool(cond), detail))
    status = "[PASS]" if cond else "[FAIL]"
    suffix = f" ({detail})" if detail else ""
    print(f"{status} {name}{suffix}")


# ---- 1. Argument Parser & Config Override ----
parser = cli.build_arg_parser()
parsed = parser.parse_args(
    ["--config", "a.yaml", "b.yaml", "--smoke", "--batch-size", "32"]
)
record(
    "CLI parser accepts multiple --config paths, --smoke, and --batch-size",
    parsed.configs == ["a.yaml", "b.yaml"]
    and parsed.smoke is True
    and parsed.batch_size == 32,
    f"configs={parsed.configs}, smoke={parsed.smoke}, bs={parsed.batch_size}",
)

tooka_b_path = config.ROOT / "configs" / "transformer" / "tooka_B.yaml"
cfg_override = cli.load_and_prepare_config(tooka_b_path, batch_size_override=48)
record(
    "load_and_prepare_config applies --batch-size override",
    cfg_override["training"]["batch_size"] == 48,
    f"batch_size={cfg_override['training']['batch_size']}",
)

bad_bs_caught = False
try:
    cli.load_and_prepare_config(tooka_b_path, batch_size_override=0)
except run_config.ConfigError:
    bad_bs_caught = True
record("Invalid --batch-size <= 0 raises ConfigError", bad_bs_caught)

# ---- 2. Zero-Copy build_monitor_subset Verification ----
pack_dir = run_config.project_path(cfg_override["data"]["pack_dir"])
manifest = data.open_pack(pack_dir)
bundle = tokenization.load_tokenizer(
    run_config.project_path(cfg_override["model"]["pretrained_path"]),
    backend=cfg_override["model"]["tokenizer_backend"],
    max_length=64,
)
small_train = data.load_split(
    "train",
    bundle,
    pack_dir,
    manifest,
    variant="B",
    subset_rows=200,
    subset_seed=config.SPLIT_SEED,
    cache_dir=None,
    show_progress=False,
)
monitor_sub = cli.build_monitor_subset(small_train, subset_rows=50, seed=config.SPLIT_SEED)
expected_monitor_idx = data.subset_row_indices(200, 50, config.SPLIT_SEED, purpose="monitor")
sample_batch = monitor_sub.dataset()[np.array([0, 1, 2], dtype=np.int64)]

record(
    "build_monitor_subset creates exact zero-copy monitor split",
    monitor_sub.n_rows == 50
    and np.array_equal(monitor_sub.source_rows, expected_monitor_idx)
    and np.array_equal(monitor_sub.ids, small_train.ids[expected_monitor_idx])
    and monitor_sub.arrays.ids is small_train.arrays.ids
    and sample_batch["input_ids"].shape[0] == 3,
    f"n_rows={monitor_sub.n_rows}, batch_shape={tuple(sample_batch['input_ids'].shape)}",
)

# ---- 3. Skip Completed Runs & Stop Sequence on Error ----
base_yaml_path = config.ROOT / "configs" / "transformer" / "base.yaml"

with tempfile.TemporaryDirectory(prefix="dk_cli_seq_") as tmp_dir_str:
    tmp_dir = Path(tmp_dir_str)
    shutil_base = tmp_dir / "base.yaml"
    shutil_base.write_text(base_yaml_path.read_text(encoding="utf-8"), encoding="utf-8")

    # Config 1: already complete run -> should skip and return 0
    cfg1_path = tmp_dir / "run_complete.yaml"
    cfg1_data = {
        "extends": "base.yaml",
        "run_name": "cli_done_run",
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 64,
            "tokenizer_backend": "hf_fast",
        },
        "training": {"variant": "B"},
        "output": {"root": str(tmp_dir / "runs")},
    }
    cfg1_path.write_text(yaml.safe_dump(cfg1_data, sort_keys=False), encoding="utf-8")

    done_run_dir = tmp_dir / "runs" / "cli_done_run"
    done_run_dir.mkdir(parents=True, exist_ok=True)
    predict.atomic_write_json({"status": "complete"}, done_run_dir / predict.RUN_SUMMARY_NAME)
    (done_run_dir / predict.BEST_MODEL_NAME).write_bytes(b"placeholder")
    (done_run_dir / predict.VAL_PROBS_NAME).write_bytes(b"placeholder")
    (done_run_dir / predict.TEST_PROBS_NAME).write_bytes(b"placeholder")

    code_skip = cli.main(["--config", str(cfg1_path)])
    record(
        "Completed run is skipped cleanly with exit code 0",
        code_skip == 0 and checkpoint.run_status(done_run_dir) == "complete",
        f"exit_code={code_skip}",
    )

    # Config 2: resumable run with mismatched config hash -> must fail with code != 0
    # and MUST NOT execute Config 3 after it.
    cfg2_path = tmp_dir / "run_bad_resume.yaml"
    cfg2_data = {
        "extends": "base.yaml",
        "run_name": "cli_bad_resume",
        "model": {
            "pretrained_path": "models/pretrained/TookaBERT-Base",
            "max_length": 64,
            "tokenizer_backend": "hf_fast",
        },
        "training": {"variant": "B"},
        "output": {"root": str(tmp_dir / "runs")},
    }
    cfg2_path.write_text(yaml.safe_dump(cfg2_data, sort_keys=False), encoding="utf-8")
    bad_resume_dir = tmp_dir / "runs" / "cli_bad_resume"
    bad_resume_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "format_version": checkpoint.CHECKPOINT_FORMAT_VERSION,
            "config_hash": "wrong_hash_0000",
            "state": {},
        },
        bad_resume_dir / predict.LAST_CHECKPOINT_NAME,
    )

    cfg3_path = tmp_dir / "run_after_fail.yaml"
    cfg3_data = dict(cfg2_data, run_name="cli_should_not_run")
    cfg3_path.write_text(yaml.safe_dump(cfg3_data, sort_keys=False), encoding="utf-8")

    code_fail_seq = cli.main(["--config", str(cfg2_path), str(cfg3_path)])
    record(
        "Sequential runner halts with non-zero exit code on hash mismatch and does not start next config",
        code_fail_seq != 0 and not (tmp_dir / "runs" / "cli_should_not_run").exists(),
        f"exit_code={code_fail_seq}",
    )

    # ---- 4. CLI --smoke Execution with --batch-size Override ----
    smoke_out = tmp_dir / "cli_smoke_out"
    code_smoke = cli.main(
        ["--config", str(cfg1_path), "--smoke", "--batch-size", "16"],
        smoke_directory=smoke_out,
        require_cuda_in_smoke=torch.cuda.is_available(),
    )
    record(
        "CLI --smoke mode executes end-to-end and returns exit code 0",
        code_smoke == 0 and checkpoint.run_status(smoke_out) == "complete",
        f"exit_code={code_smoke}, status={checkpoint.run_status(smoke_out)}",
    )

failed = [name for name, ok, _ in results if not ok]
print("-" * 60)
if failed:
    raise AssertionError(f"Failed checks ({len(failed)}): {failed}")
print(f"ALL PASSED ({len(results)}/{len(results)} checks)")

[PASS] CLI parser accepts multiple --config paths, --smoke, and --batch-size (configs=['a.yaml', 'b.yaml'], smoke=True, bs=32)
[PASS] load_and_prepare_config applies --batch-size override (batch_size=48)
[PASS] Invalid --batch-size <= 0 raises ConfigError
[timer] tokenize_TookaBERT-Base_train: 0.0s
[PASS] build_monitor_subset creates exact zero-copy monitor split (n_rows=50, batch_shape=(3, 19))
RUN [1/1]: config=/tmp/dk_cli_seq_0e0ljo9o/run_complete.yaml (smoke=False, batch_size_override=None)
[PASS] cli_done_run: run is already complete at /tmp/dk_cli_seq_0e0ljo9o/runs/cli_done_run -> skipping
[PASS] All 1 config(s) finished successfully.
[PASS] Completed run is skipped cleanly with exit code 0 (exit_code=0)
RUN [1/2]: config=/tmp/dk_cli_seq_0e0ljo9o/run_bad_resume.yaml (smoke=False, batch_size_override=None)
[FAIL] run_bad_resume.yaml: CheckpointError: The last checkpoint is missing keys: ['model_state_dict', 'optimizer_state_dict', 'rng_state', 'scheduler_state_dict', 'train_state'

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18876.43it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[PASS] 3. Model & tokenizer load: backend=hf_fast, model=TookaBERT-Base, params(total=122,319,366, trainable=122,319,366), ignored_unexpected=[]
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[PASS] 4. Temporary token cache build & re-read: origin=(built -> cache), rows=64, tokens=928
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[timer] tokenize_TookaBERT-Base_validation: 0.0s
[timer] tokenize_TookaBERT-Base_test: 0.0s
[timer] tokenize_TookaBERT-Base_train: 0.0s
[PASS] 5. Variant B/C batch order equivalence: B and C batch fingerprints match (ep0=0879763f767a4629, ep1=e94867aeb86bc3ee), max_token_id=47308 fits embedding table
[PASS] 6. Worst-case batch (forward/backward/step): shape=(16, 64), loss=2.6989, peak_vram=2356.8 MB (2.30 GB)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 21957.37it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=cli_done_run device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[optimizer] fused=True steps_per_epoch=125 total_steps=250 warmup_steps=15
[train] run=cli_done_run epoch=1/2 step=5/125 global_step=5/250 progress=2.00% objective=2.346177 lr_encoder=6.667e-06 lr_heads=3.333e-05 samples_per_second=13.44 ETA=4.9 min
[train] run=cli_done_run epoch=1/2 step=10/125 global_step=10/250 progress=4.00% objective=2.259209 lr_encoder=1.333e-05 lr_heads=6.667e-05 samples_per_second=26.08 ETA=2.5 min
[train] run=cli_done_run epoch=1/2 step=15/125 global_step=15/250 progress=6.00% objective=2.257822 lr_encoder=2.000e-05 lr_heads=1.000e-04 samples_per_second=37.88 ETA=1.7 min
[train] run=cli_done_run epoch=1/2 step=20/125 global_step=20/250 progress=8.00% objective=2.397271 lr_encoder=1.957e-05 lr_heads=9.787e-05 samples_per_second=49.17 ETA=1.2 min
[train] run=cli_done_run epoch=1/2 step=25/125 gl

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 23729.30it/s]
[transformers] BertModel LOAD REPORT from: /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/models/pretrained/TookaBERT-Base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

[run] start run=cli_done_run device=cuda:0 variant=B seed=2026
[model] parameter_counts={'total': 122319366, 'trainable': 122319366, 'encoder': 122314752, 'heads': 4614}
[resume] Rebuilding best_model.pt and val_probs_best.parquet from the committed last.pt weights.
[resume] restored epochs_done=1 global_step=125 best_epoch=1 best_score=0.569977
[optimizer] fused=True steps_per_epoch=125 total_steps=250 warmup_steps=15
[train] run=cli_done_run epoch=2/2 step=5/125 global_step=130/250 progress=52.00% objective=1.713451 lr_encoder=1.021e-05 lr_heads=5.106e-05 samples_per_second=126.69 ETA=0.3 min
[train] run=cli_done_run epoch=2/2 step=10/125 global_step=135/250 progress=54.00% objective=1.837259 lr_encoder=9.787e-06 lr_heads=4.894e-05 samples_per_second=130.17 ETA=0.2 min
[train] run=cli_done_run epoch=2/2 step=15/125 global_step=140/250 progress=56.00% objective=1.784875 lr_encoder=9.362e-06 lr_heads=4.681e-05 samples_per_second=133.60 ETA=0.2 min
[train] run=cli_done_run epoch=2/2 ste

# Cell Title: Test: All 6 YAML Configs, package_run_outputs.py, and download_pretrained_models.py

In [1]:
import ast
import copy
import importlib.util
import tempfile
import zipfile
from pathlib import Path

from digikala_review_transformer import config
from digikala_review_transformer.training import predict, run_config

results: list[tuple[str, bool, str]] = []


def record(name: str, cond: bool, detail: str = "") -> None:
    results.append((name, bool(cond), detail))
    status = "[PASS]" if cond else "[FAIL]"
    suffix = f" ({detail})" if detail else ""
    print(f"{status} {name}{suffix}")


# ---- 1. Validate All 6 YAML Run Configs ----
expected_specs = {
    "tooka_B": ("models/pretrained/TookaBERT-Base", 128, "hf_fast", "B"),
    "tooka_C": ("models/pretrained/TookaBERT-Base", 128, "hf_fast", "C"),
    "roberta_B": ("models/pretrained/roberta-fa-zwnj-base", 128, "hf_fast", "B"),
    "roberta_C": ("models/pretrained/roberta-fa-zwnj-base", 128, "hf_fast", "C"),
    "mdeberta_B": ("models/pretrained/mdeberta-v3-base", 160, "sentencepiece", "B"),
    "mdeberta_C": ("models/pretrained/mdeberta-v3-base", 160, "sentencepiece", "C"),
}

cfg_dir = config.ROOT / "configs" / "transformer"
loaded_cfgs: dict[str, dict] = {}
hashes: set[str] = set()

for run_name, (model_rel_path, max_len, backend, variant) in expected_specs.items():
    yaml_path = cfg_dir / f"{run_name}.yaml"
    cfg = run_config.load_run_config(yaml_path)
    loaded_cfgs[run_name] = cfg
    cfg_h = run_config.config_hash(cfg)
    hashes.add(cfg_h)

    model_disk_path = run_config.project_path(cfg["model"]["pretrained_path"])
    spec_ok = (
        cfg["run_name"] == run_name
        and cfg["seed"] == config.SPLIT_SEED
        and cfg["model"]["pretrained_path"] == model_rel_path
        and model_disk_path.is_dir()
        and cfg["model"]["max_length"] == max_len
        and cfg["model"]["tokenizer_backend"] == backend
        and cfg["training"]["variant"] == variant
    )
    record(
        f"Config '{run_name}.yaml' matches plan schema and local model folder",
        spec_ok,
        f"path={model_rel_path}, max_len={max_len}, backend={backend}, variant={variant}, hash={cfg_h[:10]}",
    )

record(
    "All 6 run configs produce pairwise distinct config hashes",
    len(hashes) == 6,
    f"unique_hashes={len(hashes)}",
)

for prefix in ("tooka", "roberta", "mdeberta"):
    cfg_b = copy.deepcopy(loaded_cfgs[f"{prefix}_B"])
    cfg_c = copy.deepcopy(loaded_cfgs[f"{prefix}_C"])
    cfg_b.pop("run_name")
    cfg_c.pop("run_name")
    cfg_b["training"].pop("variant")
    cfg_c["training"].pop("variant")
    record(
        f"'{prefix}_B' and '{prefix}_C' are identical except for run_name and training.variant",
        cfg_b == cfg_c,
    )

# ---- 2. Test scripts/package_run_outputs.py ----
pkg_script_path = config.ROOT / "scripts" / "package_run_outputs.py"
pkg_spec = importlib.util.spec_from_file_location("package_run_outputs_cli", pkg_script_path)
pkg_cli = importlib.util.module_from_spec(pkg_spec)
pkg_spec.loader.exec_module(pkg_cli)


def make_synthetic_completed_run(runs_root: Path, name: str) -> Path:
    rdir = runs_root / name
    rdir.mkdir(parents=True, exist_ok=True)
    for req_name in pkg_cli.REQUIRED_FILES:
        if req_name == predict.RUN_SUMMARY_NAME:
            predict.atomic_write_json({"run_name": name, "status": "complete"}, rdir / req_name)
        else:
            (rdir / req_name).write_text(f"sample content for {req_name}\n", encoding="utf-8")

    tok_dir = rdir / predict.TOKENIZER_DIR_NAME
    tok_dir.mkdir(parents=True, exist_ok=True)
    (tok_dir / "tokenizer_info.json").write_text('{"backend": "hf_fast"}\n', encoding="utf-8")
    (tok_dir / "tokenizer.json").write_text("{}\n", encoding="utf-8")

    # Heavy / excluded files that must exist on disk for run_status=='complete' or cleanup testing,
    # but MUST NOT be included inside the generated zip archive.
    (rdir / predict.BEST_MODEL_NAME).write_bytes(b"heavy_weights_bytes" * 100)
    (rdir / "extra_weights.ckpt").write_bytes(b"checkpoint_bytes")
    pycache_dir = rdir / "__pycache__"
    pycache_dir.mkdir(exist_ok=True)
    (pycache_dir / "temp.pyc").write_bytes(b"bytecode")
    return rdir


with tempfile.TemporaryDirectory(prefix="dk_pkg_test_") as tmp_str:
    tmp_root = Path(tmp_str)
    runs_root = tmp_root / "transformer_runs"
    out_dir = tmp_root / "outputs_to_send"

    make_synthetic_completed_run(runs_root, "tooka_B")
    make_synthetic_completed_run(runs_root, "tooka_C")

    exit_code = pkg_cli.main(
        [
            "--runs",
            "tooka_B",
            "tooka_C",
            "--runs-root",
            str(runs_root),
            "--output-dir",
            str(out_dir),
        ]
    )
    zip_b = out_dir / "tooka_B.zip"
    zip_c = out_dir / "tooka_C.zip"
    record(
        "package_run_outputs CLI packages multiple runs with exit code 0",
        exit_code == 0 and zip_b.is_file() and zip_c.is_file(),
        f"exit_code={exit_code}",
    )

    with zipfile.ZipFile(zip_b, "r") as zf:
        archived_names = set(zf.namelist())

    expected_in_zip = {f"tooka_B/{fname}" for fname in pkg_cli.REQUIRED_FILES} | {
        "tooka_B/tokenizer/tokenizer_info.json",
        "tooka_B/tokenizer/tokenizer.json",
    }
    forbidden_in_zip = {
        f"tooka_B/{predict.BEST_MODEL_NAME}",
        f"tooka_B/{predict.LAST_CHECKPOINT_NAME}",
        "tooka_B/extra_weights.ckpt",
        "tooka_B/__pycache__/temp.pyc",
    }
    has_any_pt = any(item.endswith((".pt", ".pth", ".ckpt")) for item in archived_names)

    record(
        "Archive contains all required light artifacts and tokenizer/ files under <run_name>/",
        expected_in_zip.issubset(archived_names),
        f"archived_count={len(archived_names)}",
    )
    record(
        "Archive excludes best_model.pt, last.pt, .ckpt files, and __pycache__",
        archived_names.isdisjoint(forbidden_in_zip) and (not has_any_pt),
        f"archived={sorted(archived_names)}",
    )

    # Guard 1: Incomplete run (has last.pt -> status == 'resumable') must fail
    incomplete_dir = make_synthetic_completed_run(runs_root, "incomplete_run")
    (incomplete_dir / predict.LAST_CHECKPOINT_NAME).write_bytes(b"resumable_state")
    code_incomplete = pkg_cli.main(
        [
            "--runs",
            "incomplete_run",
            "--runs-root",
            str(runs_root),
            "--output-dir",
            str(out_dir),
        ]
    )
    record(
        "package_run_outputs refuses to package an incomplete/resumable run",
        code_incomplete != 0 and not (out_dir / "incomplete_run.zip").exists(),
        f"exit_code={code_incomplete}",
    )

    # Guard 2: Completed status but missing encoder_config.json must fail
    missing_file_dir = make_synthetic_completed_run(runs_root, "missing_artifact_run")
    (missing_file_dir / predict.ENCODER_CONFIG_NAME).unlink()
    code_missing = pkg_cli.main(
        [
            "--runs",
            "missing_artifact_run",
            "--runs-root",
            str(runs_root),
            "--output-dir",
            str(out_dir),
        ]
    )
    record(
        "package_run_outputs refuses to package a run missing encoder_config.json",
        code_missing != 0 and not (out_dir / "missing_artifact_run.zip").exists(),
        f"exit_code={code_missing}",
    )

# ---- 3. Verify scripts/download_pretrained_models.py AST Structure ----
dl_script_path = config.ROOT / "scripts" / "download_pretrained_models.py"
dl_source = dl_script_path.read_text(encoding="utf-8")
dl_tree = ast.parse(dl_source, filename=str(dl_script_path))

models_dict_literal = None
for node in dl_tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == "MODELS":
                models_dict_literal = ast.literal_eval(node.value)

expected_models_map = {
    "TookaBERT-Base": "PartAI/TookaBERT-Base",
    "roberta-fa-zwnj-base": "HooshvareLab/roberta-fa-zwnj-base",
    "mdeberta-v3-base": "microsoft/mdeberta-v3-base",
}
record(
    "download_pretrained_models.py parses cleanly and defines exact 3 pretrained repos",
    models_dict_literal == expected_models_map,
    f"MODELS={models_dict_literal}",
)

failed = [name for name, ok, _ in results if not ok]
print("-" * 60)
if failed:
    raise AssertionError(f"Failed checks ({len(failed)}): {failed}")
print(f"ALL PASSED ({len(results)}/{len(results)} checks)")

/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] Config 'tooka_B.yaml' matches plan schema and local model folder (path=models/pretrained/TookaBERT-Base, max_len=128, backend=hf_fast, variant=B, hash=c5a1914e63)
[PASS] Config 'tooka_C.yaml' matches plan schema and local model folder (path=models/pretrained/TookaBERT-Base, max_len=128, backend=hf_fast, variant=C, hash=f04a343633)
[PASS] Config 'roberta_B.yaml' matches plan schema and local model folder (path=models/pretrained/roberta-fa-zwnj-base, max_len=128, backend=hf_fast, variant=B, hash=1cbd6fbb94)
[PASS] Config 'roberta_C.yaml' matches plan schema and local model folder (path=models/pretrained/roberta-fa-zwnj-base, max_len=128, backend=hf_fast, variant=C, hash=701f31bb68)
[PASS] Config 'mdeberta_B.yaml' matches plan schema and local model folder (path=models/pretrained/mdeberta-v3-base, max_len=160, backend=sentencepiece, variant=B, hash=1ce46e0d17)
[PASS] Config 'mdeberta_C.yaml' matches plan schema and local model folder (path=models/pretrained/mdeberta-v3-base, max_le

# Cell Title: Inspect: Git Status, .gitignore, pyproject.toml, and uv Lock Check (Read-Only)

In [2]:
import subprocess

from digikala_review_transformer import config


def run_cmd(cmd: list[str]) -> str:
    proc = subprocess.run(cmd, check=False, cwd=config.ROOT, capture_output=True, text=True)
    return f"$ {' '.join(cmd)} (exit={proc.returncode})\n{proc.stdout}{proc.stderr}".strip()

print(run_cmd(["git", "status", "--short", "--untracked-files=all"]))
print("=" * 72)
print(run_cmd(["uv", "lock", "--check"]))
print("=" * 72)
gitignore_path = config.ROOT / ".gitignore"
print("--- .gitignore ---")
print(gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else "<missing>")
print("=" * 72)
print("--- pyproject.toml ---")
print((config.ROOT / "pyproject.toml").read_text(encoding="utf-8"))

$ git status --short --untracked-files=all (exit=0)
 M notebooks/10_1st_transformer_finetune.ipynb
 M src/digikala_review_transformer/training/smoke.py
?? configs/transformer/mdeberta_B.yaml
?? configs/transformer/mdeberta_C.yaml
?? configs/transformer/roberta_B.yaml
?? configs/transformer/roberta_C.yaml
?? configs/transformer/tooka_C.yaml
?? scripts/download_pretrained_models.py
?? scripts/package_run_outputs.py
?? scripts/train_transformer.py
$ uv lock --check (exit=0)
Resolved 123 packages in 2ms
--- .gitignore ---
# Python-generated files
__pycache__/
*.py[oc]
build/
dist/
wheels/
*.egg-info

# Virtual environments
.venv

# data and models (large; not for Git)
.cache/
*.rar
*.parquet
*.pt
*.pth
*.safetensors
*.bin
*.joblib

# models: weights and tokenizers are downloaded by script, never committed
models/*
!models/.gitkeep

# zip files produced by package_run_outputs.py
outputs_to_send/

# notebooks and misc
.ipynb_checkpoints/
.env
notebooks/00_data_ingest_quality_gate_copy_for_re